---

# Tabular End-to-End Notebook (EDA + Pipeline)

*Exploratory analysis and the full modelling pipeline in one run, for any tabular regression or classification task. Part 1 explores the data and writes `eda-output/eda_decisions.json`; Part 2 reads it, so every `'auto'` setting in the pipeline follows the EDA findings. Outputs go to `eda-output/` and `pipeline-output/`.*

---

## Team Name

- Name 1 (Role)
- Name 2
- Name 3

---

## Table of Contents

**Part 1: Exploratory Data Analysis**

1. [**Introduction**](#1)
2. [**Initialization**](#2)
3. [**Plotting Toolkit**](#3)
4. [**Data Overview**](#4)
5. [**Data Quality**](#5)
6. [**Target Analysis**](#6)
7. [**Numerical Features**](#7)
8. [**Categorical Features**](#8)
9. [**Feature vs Target**](#9)
10. [**Multivariate Structure**](#10)
11. [**Temporal and Segment Analysis**](#11)
12. [**Text Columns Snapshot**](#12)
13. [**Train vs Test Drift**](#13)
14. [**Baseline Signal Check**](#14)
15. [**EDA Summary**](#15)

**Part 2: Modelling Pipeline** (reads `eda-output/eda_decisions.json` written by Part 1)

16. [**Pipeline Initialization**](#16)
17. [**Toolkit**](#17)
18. [**Data Preparation**](#18)
19. [**Feature Engineering**](#19)
20. [**Validation Strategy**](#20)
21. [**Baseline: Linear / Logistic Regression**](#21)
22. [**Gradient Boosting Models**](#22)
23. [**Hyperparameter Tuning**](#23)
24. [**Ensemble and Model Selection**](#24)
25. [**Final Evaluation on Holdout**](#25)
26. [**Model Interpretation**](#26)
27. [**Error Analysis**](#27)
28. [**Predictions and Export**](#28)

---

# Introduction <a name="1"></a>

---

## Overview

*Data analysis competitions reward teams that understand their data before they model it. This notebook automates the full exploratory workflow for a single tabular file so that the first hour of a competition goes into reading results, not writing plotting code.*

## Aim

*The notebook profiles structure, quality, distributions, relationships with the target, multivariate structure, temporal behaviour, train/test drift, and baseline predictive signal. Every figure is saved as a numbered PNG in `eda-output/` and every key table is saved as CSV, ready to be dropped into a report or slide deck.*

## How to Point the Notebook at a Dataset

*All changes happen in the `Settings` class (section 2). Nothing else needs editing.*

| Where you run | What to edit | Example |
|---|---|---|
| Kaggle | `KAGGLE_PATH` | `'/kaggle/input/my-comp'` |
| Google Colab | `COLAB_PATH` | `'/content/drive/MyDrive/my-comp'` |
| Local machine | `LOCAL_PATH` (relative to this notebook) | `'data'` (folder) or `'data/train.csv'` |
| Optional test file | `TEST_KAGGLE_PATH`, `TEST_COLAB_PATH`, `TEST_LOCAL_PATH` | `'.../test.csv'` or `None` |

*The environment is detected automatically. On Kaggle, when `KAGGLE_PATH` does not exist, the notebook lists every file under `/kaggle/input` and falls back to the largest tabular file it finds. Supported formats are CSV, TSV, TXT, Parquet, Feather, Excel, and JSON, including zipped CSV.*

*After the path, set `TARGET` (`'auto'`, a column, or `None` for unsupervised EDA), `ID_COLS` (`'auto'` = recognised by name), and optionally `TIME_COL`, `GROUP_COL`, and `CV_GROUP`. Column roles (numeric, categorical, text, id, datetime) are inferred automatically and can be overridden with explicit lists.*

## Dataset

*Any tabular dataset works: one row per observation, a target column, and any mix of numeric, categorical, date, and free-text columns. Put the files in `data/` next to this notebook, or point `KAGGLE_PATH` / `COLAB_PATH` / `LOCAL_PATH` at the dataset folder. The notebook prints which files it picked.*

```
data/
├── train.csv              (features + target)
├── test.csv               (features only, optional)
└── sample_submission.csv  (id + target columns, optional)
```

*With the default `'auto'` settings the target is taken from `sample_submission.csv` (or as the only train column missing from the test file), identifier columns are recognised by name, and column roles are inferred. Set them explicitly in `Settings` whenever the guess is wrong.*

## Approach: Automated Layered EDA

*The analysis moves from the cheapest checks to the most expensive ones, so a failure or a surprise is caught early.*

```
Raw table
    |
    +-- Overview      : shape, dtypes, inferred roles, profile table
    +-- Quality       : missing, duplicates, constants, cardinality, outliers
    +-- Univariate    : target, numerical, categorical distributions
    +-- Bivariate     : feature vs target plots + statistical tests + mutual information
    +-- Multivariate  : correlation, VIF, Cramer's V, clustering, pairplot, PCA
    +-- Temporal      : volume and target over time, segment stability (mean vs CV)
    +-- Text / Drift  : text length signal, train vs test shift, adversarial AUC
    +-- Baseline      : quick gradient boosting CV + permutation importance
         |
    EDA summary (markdown) + eda-output/ (PNG + CSV)
```

*Plots run on a random sample (`SAMPLE_N`) for speed, while statistics and tests use the full table unless stated otherwise.*

---

# Initialization <a name="2"></a>

---

## Environment Setup

The notebook runs on CPU only and needs no GPU. The `nvidia-smi` call is kept for consistency with the other templates and fails harmlessly on CPU machines.

In [ ]:
!nvidia-smi

The following cell installs all libraries used in this notebook.

In [ ]:
%pip install -q numpy pandas matplotlib seaborn scipy scikit-learn pyarrow openpyxl

## Import Libraries

In [ ]:
import os
import re
import json
import random
import warnings
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.colors import LinearSegmentedColormap
import seaborn as sns
from scipy import stats
from scipy.cluster import hierarchy
from scipy.spatial.distance import squareform
from sklearn.decomposition import PCA
from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor
from sklearn.feature_selection import mutual_info_classif, mutual_info_regression
from sklearn.inspection import permutation_importance
from sklearn.metrics import f1_score, mean_absolute_error, r2_score, roc_auc_score
from sklearn.model_selection import GroupKFold, KFold, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from IPython.display import Markdown, display

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)
pd.set_option('display.float_format', lambda v: f'{v:,.4f}')

## Seed Everything

In [ ]:
def seed_everything(seed: int = 42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)

seed_everything(42)

## Settings

All paths, column roles, and EDA knobs are centralised here. Switching between Kaggle, Colab, and local execution only requires editing the path lines in block 1. Any option set to `None`, `[]`, or `'auto'` is either skipped or inferred from the data.

In [ ]:
class Settings:
    SEED       = 42
    _ON_KAGGLE = Path('/kaggle/input').exists()
    _ON_COLAB  = Path('/content').exists() and not _ON_KAGGLE

    # 1) Data location: a file or a dataset folder (train / test / sample_submission are found by name)
    KAGGLE_PATH      = '/kaggle/input/<dataset-slug>'
    COLAB_PATH       = '/content/drive/MyDrive/<folder>'
    LOCAL_PATH       = 'data'
    TEST_KAGGLE_PATH = None
    TEST_COLAB_PATH  = None
    TEST_LOCAL_PATH  = None
    READ_KWARGS      = {}

    # 2) Column roles: 'auto' guesses from sample_submission / test; TARGET=None runs unsupervised EDA
    TARGET    = 'auto'
    TASK      = 'auto'
    ID_COLS   = 'auto'
    DROP_COLS = []
    DATE_COLS = []
    TIME_COL  = None
    GROUP_COL = None
    CV_GROUP  = None
    NUM_COLS  = 'auto'
    CAT_COLS  = 'auto'
    TEXT_COLS = 'auto'

    # 3) EDA knobs
    SAMPLE_N       = 20_000
    DISCRETE_MAX   = 15
    CLASS_MAX      = 20
    TOP_N_CATS     = 15
    MAX_PLOT_COLS  = 30
    HIGH_CORR      = 0.80
    PAIRPLOT_K     = 5
    MIN_PERIOD_N   = 30
    TIME_RANGE     = (1950, 2025)
    RUN_BASELINE   = True
    N_FOLDS        = 5

    # 4) Outputs
    DATA_PATH  = KAGGLE_PATH if _ON_KAGGLE else COLAB_PATH if _ON_COLAB else LOCAL_PATH
    TEST_PATH  = TEST_KAGGLE_PATH if _ON_KAGGLE else TEST_COLAB_PATH if _ON_COLAB else TEST_LOCAL_PATH
    OUTPUT_DIR = Path('/kaggle/working/eda-output') if _ON_KAGGLE else Path('eda-output')
    FIG_DIR    = OUTPUT_DIR / 'figures'
    TABLE_DIR  = OUTPUT_DIR / 'tables'

CFG = Settings()
for d in (CFG.OUTPUT_DIR, CFG.FIG_DIR, CFG.TABLE_DIR):
    d.mkdir(parents=True, exist_ok=True)
env = 'Kaggle' if CFG._ON_KAGGLE else 'Colab' if CFG._ON_COLAB else 'Local'
print(f'Environment : {env}')
print(f'Data path   : {CFG.DATA_PATH}')
print(f'Test path   : {CFG.TEST_PATH}')
print(f'Output dir  : {CFG.OUTPUT_DIR.resolve()}')

## Load Dataset

The path is resolved first. On Kaggle a missing path triggers a listing of `/kaggle/input` and a fallback to the largest tabular file. The loader picks the reader from the file extension. Configured drop columns are removed and declared date columns are parsed. No visualisation happens here.

In [ ]:
TABULAR_EXT = ('.csv', '.tsv', '.txt', '.parquet', '.pq', '.feather', '.xlsx', '.xls', '.json', '.zip', '.gz')

def load_table(path, **kwargs):
    suffixes = [s.lower() for s in path.suffixes]
    if '.parquet' in suffixes or '.pq' in suffixes:
        return pd.read_parquet(path, **kwargs)
    if '.feather' in suffixes:
        return pd.read_feather(path, **kwargs)
    if '.xlsx' in suffixes or '.xls' in suffixes:
        return pd.read_excel(path, **kwargs)
    if '.json' in suffixes:
        return pd.read_json(path, **kwargs)
    if '.tsv' in suffixes:
        kwargs.setdefault('sep', '\t')
    return pd.read_csv(path, low_memory=False, **kwargs)

DATE_PATTERN = r'^(\d{4}[-/.]\d{1,2}[-/.]\d{1,2}|\d{1,2}[-/.]\d{1,2}[-/.]\d{2,4})([ T]\d{1,2}:\d{2}(:\d{2})?.*)?$'

def looks_like_date(s):
    """True when at least 90% of a sampled string column matches a date or datetime pattern."""
    if pd.api.types.is_numeric_dtype(s) or pd.api.types.is_datetime64_any_dtype(s) or pd.api.types.is_bool_dtype(s):
        return False
    st = s.dropna().astype(str).str.strip()
    if st.empty:
        return False
    st = st.sample(min(len(st), 500), random_state=CFG.SEED)
    return st.str.match(DATE_PATTERN).mean() > 0.9

def parse_dates(s):
    year_first = s.dropna().astype(str).str.strip().str.match(r'^\d{4}').mean() > 0.5
    return pd.to_datetime(s, errors='coerce', format='mixed', dayfirst=not year_first)

def prepare(frame):
    frame = frame.drop(columns=[c for c in CFG.DROP_COLS if c in frame.columns]).copy()
    for c in frame.columns:
        if c != CFG.TARGET and (c in CFG.DATE_COLS or looks_like_date(frame[c])):
            frame[c] = parse_dates(frame[c])
    return frame

def file_role(f):
    """Classify a data file by its name: 'train', 'test', 'sample' (submission template), or None."""
    stem = f.name.lower().split('.')[0]
    if 'sample' in stem or 'submission' in stem:
        return 'sample'
    if stem.startswith('test'):
        return 'test'
    if stem.startswith('train'):
        return 'train'
    return None

SKIP_DIRS = {'eda-output', 'pipeline-output', 'catboost_info', '.ipynb_checkpoints', '.git', '.venv', 'site-packages'}
LOCAL_DATA_DIRS = ('data', 'input', 'dataset')

def list_tables(root, max_depth=6):
    # os.walk + extension check by name: no per-file disk call, which is slow on the /kaggle/input network mount
    root, out = Path(root), []
    for d, dirs, names in os.walk(root):
        depth = len(Path(d).relative_to(root).parts)
        dirs[:] = [x for x in dirs if x not in SKIP_DIRS and not x.startswith('.')] if depth < max_depth else []
        out += [Path(d) / n for n in names if Path(n).suffix.lower() in TABULAR_EXT]
    return out

def discover_files(data_path, test_path):
    """Resolve the train, test, and sample-submission files from a file, a dataset folder, or the input root."""
    p = Path(data_path) if data_path else None
    explicit_test = Path(test_path) if test_path and Path(test_path).exists() else None
    if test_path and explicit_test is None:
        print(f'[warn] test path {test_path} not found; searching the dataset folder instead')
    if p is not None and p.is_file():
        train, files = p, [f for f in p.parent.iterdir() if f.is_file() and f.suffix.lower() in TABULAR_EXT]
    else:
        if p is not None and p.is_dir():
            roots = [p]
        elif CFG._ON_KAGGLE:
            roots = [Path('/kaggle/input')]
        elif CFG._ON_COLAB:
            roots = [Path('/content')]
        else:
            # local: data/, input/, dataset/ next to the notebook first, then the notebook folder itself
            roots = [Path(d) for d in LOCAL_DATA_DIRS] + [Path('.')]
        if p is not None and not p.is_dir():
            print(f'[warn] {p} not found; searching {[str(r) for r in roots if r.exists()]}')
        train, files = None, []
        for r in (r for r in roots if r.exists()):
            files = list_tables(r)
            named = [f for f in files if file_role(f) == 'train']
            others = [f for f in files if file_role(f) is None]
            train = max(named or others, key=lambda f: f.stat().st_size, default=None)
            if train is not None:
                break
    if train is None:
        return None, explicit_test, None

    def pick(role):
        near = [f for f in files if file_role(f) == role and f.parent == train.parent]
        cands = near or [f for f in files if file_role(f) == role]
        exact = [f for f in cands if f.name.lower().split('.')[0] == role]
        return min(exact or cands, key=lambda f: len(f.name), default=None)

    test = explicit_test or pick('test')
    sample = pick('sample')
    for role, f in (('train', train), ('test', test), ('sample submission', sample)):
        print(f'{role:<18}: {f if f else "(none found)"}')
    return train, test, sample

DATA_PATH, TEST_PATH, SAMPLE_PATH = discover_files(CFG.DATA_PATH, CFG.TEST_PATH)
assert DATA_PATH is not None, f'{CFG.DATA_PATH} not found and no train file was discovered. Edit the path lines in Settings.'
if TEST_PATH is None:
    print('[warn] No test file configured or found in the dataset folder: the train vs test drift section is skipped.')

ID_NAME = re.compile(r'(^id$|_id$|^id_|^key$|_key$)', re.I)

def resolve_columns(train, test):
    """Fill 'auto' column settings: target from sample_submission or the train-only column, ids by name."""
    if CFG.TARGET == 'auto':
        sample = load_table(SAMPLE_PATH) if SAMPLE_PATH else None
        only_train = [c for c in train.columns if test is not None and c not in test.columns]
        if sample is not None and sample.shape[1] >= 2 and sample.columns[1] in train.columns:
            CFG.TARGET = sample.columns[1]
        elif len(only_train) == 1:
            CFG.TARGET = only_train[0]
        else:
            CFG.TARGET = None
        print(f'[info] TARGET auto-selected: {CFG.TARGET}' if CFG.TARGET else
              '[info] no target found (no sample_submission or test file): unsupervised EDA. Set TARGET in Settings.')
    if CFG.ID_COLS == 'auto':
        CFG.ID_COLS = [c for c in train.columns if c != CFG.TARGET and ID_NAME.search(str(c))]
        print(f'[info] ID_COLS auto-selected: {CFG.ID_COLS}')

raw_train = load_table(DATA_PATH, **CFG.READ_KWARGS)
raw_test = load_table(TEST_PATH, **CFG.READ_KWARGS) if TEST_PATH else None
resolve_columns(raw_train, raw_test)
df = prepare(raw_train)
test_df = prepare(raw_test) if raw_test is not None else None

HAS_TARGET = CFG.TARGET is not None and CFG.TARGET in df.columns
if CFG.TARGET is not None and not HAS_TARGET:
    print(f'[warn] TARGET={CFG.TARGET!r} not found, running unsupervised EDA')

print(f'Train : {df.shape[0]:,} rows x {df.shape[1]} cols  <- {DATA_PATH}')
if test_df is not None:
    print(f'Test  : {test_df.shape[0]:,} rows x {test_df.shape[1]} cols  <- {TEST_PATH}')

---

# Plotting Toolkit <a name="3"></a>

---

A single visual identity is applied to every chart so that exported figures look consistent in a report or slide deck. The palette pairs a deep navy primary with a coral accent, followed by eight distinguishable categorical colours. Sequential and diverging colour maps are derived from the same hues.

In [ ]:
PRIMARY   = '#3D5A80'
ACCENT    = '#EE6C4D'
SOFT      = '#98C1D9'
DARK      = '#1B263B'
MUTED     = '#8D99AE'
PALETTE   = ['#3D5A80', '#EE6C4D', '#2A9D8F', '#E9C46A', '#9B5DE5',
             '#F15BB5', '#00BBF9', '#8AB17D', '#E76F51', '#264653']
CMAP_SEQ  = LinearSegmentedColormap.from_list('seq', ['#F7F9FC', '#98C1D9', '#3D5A80', '#1B263B'])
CMAP_DIV  = LinearSegmentedColormap.from_list('div', ['#EE6C4D', '#FBE3DC', '#F7F7F7', '#D6E4F0', '#3D5A80'])
CMAP_HEAT = LinearSegmentedColormap.from_list('heat', ['#FFF8F0', '#F4A261', '#E76F51', '#9D0208'])

sns.set_theme(style='whitegrid', palette=PALETTE)
mpl.rcParams.update({
    'figure.dpi': 100, 'savefig.dpi': 200, 'savefig.bbox': 'tight',
    'figure.facecolor': 'white', 'axes.facecolor': 'white',
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.edgecolor': '#C9D1DB', 'axes.labelcolor': DARK, 'axes.titleweight': 'bold',
    'axes.titlesize': 12, 'axes.labelsize': 10, 'axes.titlecolor': DARK,
    'xtick.color': '#4A5568', 'ytick.color': '#4A5568',
    'grid.color': '#E2E8F0', 'grid.linewidth': 0.8, 'legend.frameon': False,
    'font.size': 10,
})

sns.palplot(PALETTE)
plt.title('Notebook palette', loc='left')
plt.show()

The helpers below handle figure numbering, subplot grids, titles, and sampling. `save_fig` writes every figure to `eda-output/figures/` with a running number prefix, so the export order matches the notebook order. `save_table` does the same for CSV tables.

In [ ]:
FIG_COUNTER = [0]
SAVED_FIGS = []

def slug(text):
    return re.sub(r'[^a-z0-9]+', '_', str(text).lower()).strip('_')[:60]

def save_fig(fig, name, show=True):
    FIG_COUNTER[0] += 1
    path = CFG.FIG_DIR / f'{FIG_COUNTER[0]:02d}_{slug(name)}.png'
    fig.savefig(path, facecolor='white')
    SAVED_FIGS.append(path.name)
    if show:
        plt.show()
    plt.close(fig)
    return path

def save_table(table, name, index=True):
    path = CFG.TABLE_DIR / f'{slug(name)}.csv'
    table.to_csv(path, index=index)
    return path

def make_grid(n, ncols=4, w=4.2, h=3.2):
    ncols = max(1, min(ncols, n))
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(w * ncols, h * nrows), squeeze=False)
    axes = axes.ravel()
    for ax in axes[n:]:
        ax.set_visible(False)
    return fig, axes[:n]

def suptitle(fig, title, subtitle=None, layout=True):
    h = fig.get_figheight()
    if layout:
        fig.tight_layout(rect=(0, 0, 1, 1 - (0.85 if subtitle else 0.55) / h))
    fig.text(0.01, 1 - 0.1 / h, title, ha='left', va='top', fontsize=15, fontweight='bold', color=DARK)
    if subtitle:
        fig.text(0.01, 1 - 0.42 / h, subtitle, ha='left', va='top', fontsize=10, color=MUTED)

def sample(frame, n=None):
    n = n or CFG.SAMPLE_N
    return frame.sample(n, random_state=CFG.SEED) if len(frame) > n else frame

def short(value, k=28):
    value = str(value)
    return value if len(value) <= k else value[:k - 3] + '...'

def bar_labels(ax, fmt='{:,.0f}', pad=3, size=8):
    for container in ax.containers:
        labels = [fmt.format(v) for v in container.datavalues]
        ax.bar_label(container, labels=labels, padding=pad, fontsize=size, color='#4A5568')

def no_data(msg):
    display(Markdown(f'> **Skipped:** {msg}'))

def top_levels(s, k=None):
    k = k or CFG.TOP_N_CATS
    s = s.astype(object).where(s.notna(), '(missing)').astype(str)
    keep = s.value_counts().index[:k]
    return s.where(s.isin(keep), '(other)')

def cramers_v(a, b):
    ct = pd.crosstab(a, b)
    if min(ct.shape) < 2:
        return np.nan
    chi2 = stats.chi2_contingency(ct, correction=False)[0]
    n = ct.values.sum()
    r, k = ct.shape
    # bias-corrected Cramer's V (Bergsma, 2013)
    phi2 = max(0.0, chi2 / n - (k - 1) * (r - 1) / (n - 1))
    rc, kc = r - (r - 1) ** 2 / (n - 1), k - (k - 1) ** 2 / (n - 1)
    return float(np.sqrt(phi2 / max(min(kc - 1, rc - 1), 1e-12)))

def encode_frame(frame, num_cols, cat_cols):
    X = pd.DataFrame(index=frame.index)
    for c in num_cols:
        X[c] = frame[c].astype(float).fillna(frame[c].median())
    for c in cat_cols:
        X[c] = top_levels(frame[c], 200).astype('category').cat.codes
    return X

---

# Data Overview <a name="4"></a>

---

The first look covers the raw rows, the table size, and the memory footprint. This confirms the file was parsed correctly (separator, header, encoding) before any analysis is trusted.

In [ ]:
display(df.head(5))
mem_mb = df.memory_usage(deep=True).sum() / 1e6
print(f'Rows {len(df):,} | Columns {df.shape[1]} | Memory {mem_mb:,.1f} MB')

## Column Roles

Each column is assigned a role. Numeric columns with few distinct values (for example a key or a rating) are treated as categorical. Object columns with long multi-word values are treated as text, and near-unique object columns are treated as identifiers. The task type is inferred from the target unless `TASK` is set explicitly.

In [ ]:
def infer_roles(frame):
    roles = {}
    for c in frame.columns:
        if c == CFG.TARGET:
            continue
        s = frame[c]
        nun = s.nunique(dropna=True)
        if c in CFG.ID_COLS:
            roles[c] = 'id'
        elif c in CFG.DATE_COLS or pd.api.types.is_datetime64_any_dtype(s):
            roles[c] = 'datetime'
        elif nun <= 1:
            roles[c] = 'constant'
        elif pd.api.types.is_bool_dtype(s) or nun == 2:
            roles[c] = 'binary'
        elif pd.api.types.is_numeric_dtype(s):
            roles[c] = 'categorical' if nun <= CFG.DISCRETE_MAX else 'numeric'
        else:
            st = s.dropna().astype(str)
            st = st.sample(min(len(st), 3000), random_state=CFG.SEED)
            words = st.str.split().str.len().mean()
            if words >= 6:
                roles[c] = 'text'
            elif nun / max(s.notna().sum(), 1) > 0.95:
                roles[c] = 'id'
            else:
                roles[c] = 'categorical'
    overrides = (('numeric', CFG.NUM_COLS), ('categorical', CFG.CAT_COLS), ('text', CFG.TEXT_COLS))
    for role, cols in overrides:
        if isinstance(cols, (list, tuple)):
            for c in cols:
                if c in roles:
                    roles[c] = role
    return roles

ROLES      = infer_roles(df)
NUM_COLS   = [c for c, r in ROLES.items() if r == 'numeric']
CAT_COLS   = [c for c, r in ROLES.items() if r in ('categorical', 'binary')]
TEXT_COLS  = [c for c, r in ROLES.items() if r == 'text']
ID_COLS    = [c for c, r in ROLES.items() if r == 'id']
DT_COLS    = [c for c, r in ROLES.items() if r == 'datetime']
CONST_COLS = [c for c, r in ROLES.items() if r == 'constant']
FEATURES   = NUM_COLS + CAT_COLS
T = CFG.TARGET if HAS_TARGET else None

if HAS_TARGET:
    y_all = df[T]
    if CFG.TASK == 'auto':
        is_num = pd.api.types.is_numeric_dtype(y_all)
        TASK = 'classification' if (not is_num or y_all.nunique() <= CFG.CLASS_MAX) else 'regression'
    else:
        TASK = CFG.TASK
else:
    TASK = None
IS_REG = TASK == 'regression'
IS_CLF = TASK == 'classification'
if IS_CLF:
    df['__target_label__'] = df[T].astype(str)
    CLASSES = df['__target_label__'].value_counts().index.tolist()

print(f'Target: {T}  |  Task: {TASK}')
for role in ['numeric', 'categorical', 'binary', 'text', 'id', 'datetime', 'constant']:
    cols = [c for c, r in ROLES.items() if r == role]
    if cols:
        print(f'{role:<12} ({len(cols):>2}): {cols}')

## Column Profile

The profile table is the single most useful reference during a competition. It lists each column with its role, dtype, missing share, cardinality, dominant value, and an example. It is exported to `eda-output/tables/column_profile.csv`.

In [ ]:
def top_value(s):
    vc = s.value_counts(dropna=True)
    if vc.empty:
        return None, np.nan
    return short(vc.index[0], 30), vc.iloc[0] / s.notna().sum() * 100

rows = []
for c in [c for c in df.columns if c != '__target_label__']:
    s = df[c]
    tv, ts = top_value(s)
    rows.append({
        'column': c,
        'role': 'target' if c == T else ROLES.get(c),
        'dtype': str(s.dtype),
        'missing': int(s.isna().sum()),
        'missing_%': s.isna().mean() * 100,
        'unique': int(s.nunique()),
        'unique_%': s.nunique() / len(s) * 100,
        'top_value': tv,
        'top_share_%': ts,
        'example': short(s.dropna().iloc[0], 30) if s.notna().any() else None,
    })
profile = pd.DataFrame(rows).set_index('column')
save_table(profile, 'column_profile')
display(profile)

The dashboard below summarises the inferred roles, the storage dtypes, and the columns that dominate memory. Heavy text columns usually dominate memory and are worth dropping from tabular models once their signal is extracted.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.4))

rc = profile['role'].value_counts().sort_values()
axes[0].barh(rc.index, rc.values, color=PALETTE[:len(rc)])
bar_labels(axes[0])
axes[0].set_title('Inferred column roles', loc='left')
axes[0].set_xlabel('columns')

dc = profile['dtype'].value_counts()
axes[1].pie(dc.values, labels=dc.index, colors=PALETTE, startangle=90, autopct='%1.0f%%',
            pctdistance=0.78, wedgeprops=dict(width=0.42, edgecolor='white'), textprops=dict(color=DARK))
axes[1].set_title('Storage dtypes', loc='left')

mem = (df.drop(columns=['__target_label__'], errors='ignore').memory_usage(deep=True, index=False) / 1e6)
mem = mem.sort_values().tail(10)
axes[2].barh([short(i) for i in mem.index], mem.values, color=SOFT, edgecolor=PRIMARY)
bar_labels(axes[2], '{:,.1f}')
axes[2].set_title('Memory by column (MB)', loc='left')

suptitle(fig, 'Dataset at a glance', f'{len(df):,} rows x {profile.shape[0]} columns, {mem_mb:,.1f} MB in memory')
save_fig(fig, 'dataset_at_a_glance')

#### Insights

> *Fill in after running.* Are the inferred roles correct? If a column is misclassified (for example a numeric code that is really categorical), add it to `NUM_COLS` or `CAT_COLS` in `Settings` and re-run.

> Note the task type and which columns are identifiers, text, or constants, since these are excluded from the numeric and categorical analyses below.

---

# Data Quality <a name="5"></a>

---

Data quality problems silently corrupt every downstream result, so they are checked before any distribution is interpreted. This section covers explicit and disguised missing values, duplicates, constant columns, cardinality, and outliers.

## Missing Values

The bar chart ranks columns by missing share and the matrix shows whether missingness co-occurs across rows. Co-occurring gaps usually point to a shared data-collection cause and should be handled together.

In [ ]:
miss = df.drop(columns=['__target_label__'], errors='ignore').isna().mean().mul(100)
miss = miss[miss > 0].sort_values(ascending=False)
save_table(miss.rename('missing_%').to_frame(), 'missing_values')

if miss.empty:
    no_data('no explicit missing values (NaN) in any column.')
else:
    top = miss.head(30)
    fig, axes = plt.subplots(1, 2, figsize=(16, max(4, 0.3 * len(top) + 1.5)),
                             gridspec_kw={'width_ratios': [1, 1.4]})
    colors = [ACCENT if v > 30 else PRIMARY if v > 5 else SOFT for v in top.values[::-1]]
    axes[0].barh([short(i) for i in top.index[::-1]], top.values[::-1], color=colors)
    bar_labels(axes[0], '{:.1f}%')
    axes[0].set_xlabel('% missing')
    axes[0].set_title('Missing share per column', loc='left')
    matrix = sample(df[top.index], 500).isna().T
    sns.heatmap(matrix, cmap=['#EDF2F7', ACCENT], cbar=False, ax=axes[1], xticklabels=False,
                yticklabels=[short(i) for i in top.index])
    axes[1].set_title('Missingness pattern (500 sampled rows, orange = missing)', loc='left')
    suptitle(fig, 'Missing values', f'{len(miss)} of {df.shape[1]} columns contain NaN')
    save_fig(fig, 'missing_values')

    if len(miss) >= 2:
        null_corr = df[miss.index[:20]].isna().astype(int).corr()
        fig, ax = plt.subplots(figsize=(1 + 0.55 * len(null_corr), 0.5 + 0.5 * len(null_corr)))
        sns.heatmap(null_corr, cmap=CMAP_DIV, vmin=-1, vmax=1, annot=len(null_corr) <= 15, fmt='.2f',
                    square=True, linewidths=0.5, ax=ax, cbar_kws={'shrink': 0.7})
        ax.set_title('Nullity correlation (1 = always missing together)', loc='left')
        save_fig(fig, 'nullity_correlation')

## Disguised Missing Values

Many datasets encode missing values as placeholder strings (`'?'`, `'NA'`, `'unknown'`) or numeric sentinels (`-1`, `-999`, `9999`). These are invisible to `isna()` and bias every statistic, so they are counted explicitly.

In [ ]:
SENTINEL_STR = {'', 'na', 'n/a', 'nan', 'null', 'none', '?', '-', '--', 'unknown', 'missing', 'undefined'}
SENTINEL_NUM = [-1, -99, -999, -9999, 999, 9999, 99999]

rows = []
for c in [c for c in df.columns if c != '__target_label__']:
    s = df[c].dropna()
    if pd.api.types.is_numeric_dtype(s) and not pd.api.types.is_bool_dtype(s):
        for v in SENTINEL_NUM:
            n = int((s == v).sum())
            if n:
                rows.append({'column': c, 'value': v, 'count': n, 'share_%': n / len(df) * 100})
    elif not pd.api.types.is_datetime64_any_dtype(s):
        vals = s.astype(str).str.strip().str.lower()
        for v, n in vals[vals.isin(SENTINEL_STR)].value_counts().items():
            rows.append({'column': c, 'value': repr(v), 'count': int(n), 'share_%': n / len(df) * 100})

disguised = pd.DataFrame(rows, columns=['column', 'value', 'count', 'share_%'])
save_table(disguised, 'disguised_missing', index=False)
if disguised.empty:
    no_data('no placeholder strings or numeric sentinels found.')
else:
    display(disguised.sort_values('count', ascending=False))
    print('Check the data dictionary: a sentinel may be a legitimate value (e.g. key = -1 means no key detected).')

## Duplicates

Four kinds of duplication are measured: identical rows, rows identical once identifier columns are ignored, repeated identifier values, and feature-identical rows whose target disagrees. The last two matter most for modelling because they leak between train and validation folds and put a ceiling on achievable accuracy.

In [ ]:
def row_hash(frame, cols):
    return pd.util.hash_pandas_object(frame[cols].astype(str), index=False)

base_cols = [c for c in df.columns if c != '__target_label__']
no_id_cols = [c for c in base_cols if c not in ID_COLS]
feat_only = [c for c in no_id_cols if c != T]

dup_summary = {
    'identical rows': int(row_hash(df, base_cols).duplicated().sum()),
    'identical ignoring IDs': int(row_hash(df, no_id_cols).duplicated().sum()),
}
for c in ID_COLS:
    dup_summary[f'repeated {c}'] = int(df[c].duplicated().sum())
if HAS_TARGET and feat_only:
    h = row_hash(df, feat_only)
    conflict = df.groupby(h)[T].nunique()
    dup_summary['same features, different target'] = int((conflict > 1).sum())

dup_table = pd.Series(dup_summary, name='count').to_frame()
dup_table['share_%'] = dup_table['count'] / len(df) * 100
save_table(dup_table, 'duplicates')
display(dup_table)

fig, ax = plt.subplots(figsize=(10, 0.55 * len(dup_table) + 1.4))
colors = [ACCENT if v > 0 else '#C9D1DB' for v in dup_table['count'][::-1]]
ax.barh(dup_table.index[::-1], dup_table['count'][::-1], color=colors)
bar_labels(ax)
ax.set_xlabel('rows')
suptitle(fig, 'Duplicate audit', 'orange bars need a decision before modelling (drop, aggregate, or group-split)')
save_fig(fig, 'duplicate_audit')

## Constant, Quasi-Constant, and High-Cardinality Columns

Columns where one value covers 99% of rows carry almost no information. At the other extreme, categorical columns with hundreds of levels need special encoding such as target, frequency, or hashing encoders.

In [ ]:
quasi = profile[(profile['top_share_%'] >= 99) & (profile['role'] != 'target')]
print(f'Constant columns       : {CONST_COLS}')
print(f'Quasi-constant (>=99%) : {quasi.index.tolist()}')

card_cols = CAT_COLS + [c for c in ID_COLS + TEXT_COLS if c in df.columns]
if not card_cols:
    no_data('no categorical, id, or text columns.')
else:
    card = df[card_cols].nunique().sort_values()
    fig, ax = plt.subplots(figsize=(11, max(3.5, 0.32 * len(card) + 1.5)))
    colors = [ACCENT if v > 50 else PRIMARY for v in card.values]
    ax.barh([f'{short(c)} [{ROLES.get(c)}]' for c in card.index], card.values, color=colors)
    ax.set_xscale('log')
    ax.axvline(50, color=MUTED, ls='--', lw=1)
    bar_labels(ax)
    ax.set_xlabel('distinct values (log scale)')
    suptitle(fig, 'Cardinality', 'orange = more than 50 levels, needs target / frequency encoding or grouping')
    save_fig(fig, 'cardinality')

## Outliers

Outliers are counted with the Tukey rule (beyond 1.5 IQR is mild, beyond 3 IQR is extreme) and with the z-score rule. A high extreme share suggests a heavy tail that benefits from a log or rank transform rather than deletion.

In [ ]:
rows = []
for c in NUM_COLS:
    s = df[c].dropna()
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    mild = ((s < q1 - 1.5 * iqr) | (s > q3 + 1.5 * iqr)).mean() * 100
    extreme = ((s < q1 - 3 * iqr) | (s > q3 + 3 * iqr)).mean() * 100
    z = ((s - s.mean()).abs() / (s.std() + 1e-12) > 3).mean() * 100
    rows.append({'column': c, 'iqr_1.5_%': mild, 'iqr_3_%': extreme, 'z>3_%': z,
                 'min': s.min(), 'max': s.max()})
outliers = pd.DataFrame(rows).set_index('column').sort_values('iqr_1.5_%', ascending=False) if rows else pd.DataFrame()

if outliers.empty:
    no_data('no numeric columns.')
else:
    save_table(outliers, 'outliers')
    display(outliers)
    o = outliers.head(CFG.MAX_PLOT_COLS)[::-1]
    fig, ax = plt.subplots(figsize=(11, max(3.5, 0.32 * len(o) + 1.5)))
    ax.barh(o.index, o['iqr_1.5_%'] - o['iqr_3_%'], color=SOFT, label='mild (1.5-3 IQR)')
    ax.barh(o.index, o['iqr_3_%'], left=o['iqr_1.5_%'] - o['iqr_3_%'], color=ACCENT, label='extreme (>3 IQR)')
    for i, v in enumerate(o['iqr_1.5_%']):
        ax.text(v + 0.2, i, f'{v:.1f}%', va='center', fontsize=8, color='#4A5568')
    ax.set_xlabel('% of rows')
    ax.legend(loc='lower right')
    suptitle(fig, 'Outlier share per numeric column', 'Tukey fences on the full data')
    save_fig(fig, 'outlier_share')

#### Insights

> *Fill in after running.* Which columns need imputation, and is missingness itself informative (consider a `was_missing` flag)? Which sentinels are real missing values?

> How many duplicates exist and how will they be handled? Repeated IDs or feature-identical rows require a group-aware split (`CV_GROUP`) to avoid leakage. Which columns need special encoding or transforms because of cardinality or heavy tails?

---

# Target Analysis <a name="6"></a>

---

The target distribution determines the loss function, the evaluation metric, and the split strategy. A regression target is checked for skew, zero inflation, and normality, while a classification target is checked for imbalance.

In [ ]:
if not HAS_TARGET:
    no_data('TARGET is None, so target analysis is skipped.')
elif IS_REG:
    yv = df[T].dropna().astype(float)
    ys = sample(yv.to_frame(), 5000)[T]
    k2_p = stats.normaltest(ys).pvalue
    ks_p = stats.kstest((ys - ys.mean()) / ys.std(), 'norm').pvalue
    target_stats = pd.Series({
        'count': len(yv), 'mean': yv.mean(), 'std': yv.std(), 'min': yv.min(),
        'p01': yv.quantile(0.01), 'p25': yv.quantile(0.25), 'median': yv.median(),
        'p75': yv.quantile(0.75), 'p99': yv.quantile(0.99), 'max': yv.max(),
        'skew': yv.skew(), 'kurtosis': yv.kurt(), 'zero_share_%': (yv == 0).mean() * 100,
        'mode': yv.mode().iloc[0], 'mode_share_%': (yv == yv.mode().iloc[0]).mean() * 100,
        'dagostino_p': k2_p, 'ks_p': ks_p,
    }, name=T)
    save_table(target_stats.to_frame(), 'target_stats')
    display(target_stats.to_frame().T)

    fig, axes = plt.subplots(2, 2, figsize=(14, 9))
    ax = axes[0, 0]
    sns.histplot(yv, bins=50, kde=True, color=PRIMARY, edgecolor='white', alpha=0.85, ax=ax)
    ax.lines[-1].set_color(ACCENT)
    ax.axvline(yv.mean(), color=ACCENT, ls='--', lw=1.5, label=f'mean {yv.mean():,.2f}')
    ax.axvline(yv.median(), color=DARK, ls=':', lw=1.5, label=f'median {yv.median():,.2f}')
    ax.legend()
    ax.set_title('Distribution', loc='left')

    ax = axes[0, 1]
    sns.boxplot(x=yv, color=SOFT, width=0.35, ax=ax, flierprops=dict(marker='o', markersize=2, markerfacecolor=ACCENT, alpha=0.4))
    sns.stripplot(x=sample(yv.to_frame(), 1500)[T], color=PRIMARY, alpha=0.15, size=2, ax=ax)
    ax.set_title('Spread and outliers', loc='left')

    ax = axes[1, 0]
    (osm, osr), (slope, intercept, _) = stats.probplot(ys, dist='norm')
    ax.scatter(osm, osr, s=6, color=PRIMARY, alpha=0.5)
    ax.plot(osm, slope * np.asarray(osm) + intercept, color=ACCENT, lw=2)
    ax.set_xlabel('theoretical quantiles')
    ax.set_ylabel('observed quantiles')
    ax.set_title(f'Normal Q-Q (D\'Agostino p = {k2_p:.2g})', loc='left')

    ax = axes[1, 1]
    if yv.min() >= 0:
        yt, label = np.log1p(yv), 'log1p(target)'
    else:
        yt, label = pd.Series(stats.yeojohnson(yv)[0]), 'Yeo-Johnson(target)'
    sns.histplot(yt, bins=50, kde=True, color=PALETTE[2], edgecolor='white', alpha=0.85, ax=ax)
    ax.set_title(f'{label}, skew {pd.Series(yt).skew():.2f}', loc='left')

    suptitle(fig, f'Target: {T}', f'skew {yv.skew():.2f} | zeros {(yv == 0).mean():.1%} | most frequent value covers {target_stats["mode_share_%"]:.1f}%')
    save_fig(fig, 'target_distribution')
else:
    vc = df['__target_label__'].value_counts()
    pct = vc / vc.sum() * 100
    imbalance = vc.max() / vc.min()
    target_stats = pd.DataFrame({'count': vc, 'share_%': pct})
    save_table(target_stats, 'target_stats')
    display(target_stats)

    fig, axes = plt.subplots(1, 2, figsize=(15, max(4.5, 0.35 * len(vc) + 2)), gridspec_kw={'width_ratios': [1.5, 1]})
    colors = (PALETTE * 5)[:len(vc)]
    axes[0].barh([short(i) for i in vc.index[::-1]], vc.values[::-1], color=colors[::-1])
    for i, (n, p) in enumerate(zip(vc.values[::-1], pct.values[::-1])):
        axes[0].text(n, i, f'  {n:,} ({p:.1f}%)', va='center', fontsize=9, color='#4A5568')
    axes[0].set_xlabel('rows')
    axes[0].set_title('Class counts', loc='left')
    axes[1].pie(vc.values, labels=[short(i, 15) for i in vc.index], colors=colors, startangle=90,
                autopct=lambda p: f'{p:.0f}%' if p > 3 else '', pctdistance=0.78,
                wedgeprops=dict(width=0.42, edgecolor='white'))
    axes[1].set_title('Class share', loc='left')
    suptitle(fig, f'Target: {T}', f'{len(vc)} classes | imbalance ratio (max/min) = {imbalance:.1f}x')
    save_fig(fig, 'target_distribution')

#### Insights

> *Fill in after running.* For regression: is the target skewed or zero-inflated? A large spike at one value (often 0) suggests a two-stage (hurdle) model or a separate analysis of that group. Is a log transform worth trying?

> For classification: how imbalanced are the classes? Imbalance above roughly 5x calls for stratified folds, class weights, and a macro-averaged metric.

---

# Numerical Features <a name="7"></a>

---

Each continuous feature is described by location, spread, shape, and a normality test. Shape matters because linear models, distance-based models, and Pearson correlation all assume roughly symmetric distributions, whereas tree models do not.

## Descriptive Statistics

The table extends `describe()` with skewness, excess kurtosis, the D'Agostino-Pearson normality p-value (on a 5,000-row sample, since every large sample rejects normality), and a suggested transform based on skew and sign.

In [ ]:
def suggest_transform(s):
    sk = s.skew()
    if abs(sk) < 0.5:
        return 'none'
    if sk > 0 and s.min() >= 0:
        return 'log1p' if sk > 1 else 'sqrt'
    return 'yeo-johnson'

if not NUM_COLS:
    no_data('no numeric columns.')
    num_stats = pd.DataFrame()
else:
    num_stats = df[NUM_COLS].describe(percentiles=[0.01, 0.25, 0.5, 0.75, 0.99]).T
    num_stats['skew'] = df[NUM_COLS].skew()
    num_stats['kurtosis'] = df[NUM_COLS].kurt()
    num_stats['zeros_%'] = (df[NUM_COLS] == 0).mean() * 100
    num_stats['normal_p'] = [stats.normaltest(sample(df[[c]].dropna(), 5000)[c]).pvalue for c in NUM_COLS]
    num_stats['transform'] = [suggest_transform(df[c].dropna()) for c in NUM_COLS]
    save_table(num_stats, 'numeric_stats')
    display(num_stats)

## Distributions

Histograms with a KDE overlay show modality, skew, spikes, and truncation. The mean and median lines make asymmetry visible at a glance, and the panel title carries the skew.

In [ ]:
if NUM_COLS:
    cols = NUM_COLS[:CFG.MAX_PLOT_COLS]
    ds = sample(df[cols])
    fig, axes = make_grid(len(cols), ncols=4, w=4.3, h=3.1)
    for i, (ax, c) in enumerate(zip(axes, cols)):
        x = ds[c].dropna()
        color = PALETTE[i % len(PALETTE)]
        sns.histplot(x, bins=40, kde=True, color=color, edgecolor='white', alpha=0.8, ax=ax)
        if ax.lines:
            ax.lines[-1].set_color(DARK)
            ax.lines[-1].set_linewidth(1.4)
        ax.axvline(x.mean(), color=ACCENT, ls='--', lw=1.2)
        ax.axvline(x.median(), color=DARK, ls=':', lw=1.2)
        ax.set_title(f'{short(c, 24)}  (skew {x.skew():+.2f})', loc='left', fontsize=10)
        ax.set_xlabel('')
        ax.set_ylabel('')
    suptitle(fig, 'Numeric distributions', 'dashed = mean, dotted = median')
    save_fig(fig, 'numeric_distributions')

## Spread Comparison and Skewness

Robust-scaled box plots put every feature on the same axis (median 0, IQR 1), so spread and tail length can be compared directly. The skewness bar chart then ranks features by how much a transform would help.

In [ ]:
if NUM_COLS:
    cols = NUM_COLS[:CFG.MAX_PLOT_COLS]
    ds = sample(df[cols]).astype(float)
    iqr = (ds.quantile(0.75) - ds.quantile(0.25)).replace(0, 1)
    robust = ((ds - ds.median()) / iqr).clip(-15, 15)
    long = robust.melt(var_name='feature', value_name='robust z').dropna()

    fig, axes = plt.subplots(1, 2, figsize=(16, max(4.5, 0.36 * len(cols) + 1.8)), gridspec_kw={'width_ratios': [1.6, 1]})
    sns.boxplot(data=long, y='feature', x='robust z', ax=axes[0], color=SOFT, width=0.55, linewidth=1,
                flierprops=dict(marker='o', markersize=1.5, markerfacecolor=ACCENT, markeredgecolor='none', alpha=0.4))
    axes[0].axvline(0, color=MUTED, lw=1)
    axes[0].set_title('Robust-scaled spread (clipped at +/-15 IQR)', loc='left')
    axes[0].set_ylabel('')

    sk = num_stats.loc[cols, 'skew'].sort_values()
    colors = [ACCENT if abs(v) > 1 else PALETTE[3] if abs(v) > 0.5 else PALETTE[2] for v in sk.values]
    axes[1].barh([short(c) for c in sk.index], sk.values, color=colors)
    for v in (-1, -0.5, 0.5, 1):
        axes[1].axvline(v, color=MUTED, ls=':', lw=1)
    axes[1].set_title('Skewness (green < 0.5 < yellow < 1 < orange)', loc='left')
    suptitle(fig, 'Spread and shape of numeric features')
    save_fig(fig, 'numeric_spread_skew')

#### Insights

> *Fill in after running.* Which features are heavily skewed or zero-inflated, and which transform is planned? Are there spikes at round numbers, hard floors or caps, or bimodal shapes that hint at hidden sub-populations?

---

# Categorical Features <a name="8"></a>

---

Categorical features are summarised by number of levels, dominance of the top level, rare-level mass, and normalised entropy (1 means perfectly balanced levels). Rare levels are a common source of overfitting and are usually merged into an 'other' bucket.

In [ ]:
rows = []
for c in CAT_COLS:
    vc = df[c].value_counts(normalize=True, dropna=False)
    rare = vc[vc < 0.01]
    ent = stats.entropy(vc.values) / np.log(len(vc)) if len(vc) > 1 else 0.0
    rows.append({'column': c, 'levels': len(vc), 'top_level': short(vc.index[0], 25),
                 'top_share_%': vc.iloc[0] * 100, 'rare_levels(<1%)': len(rare),
                 'rare_mass_%': rare.sum() * 100, 'norm_entropy': ent})
cat_stats = pd.DataFrame(rows).set_index('column') if rows else pd.DataFrame()
if cat_stats.empty:
    no_data('no categorical columns.')
else:
    save_table(cat_stats, 'categorical_stats')
    display(cat_stats)

## Level Frequencies

Each panel shows the most frequent levels with their share of rows. Remaining levels are pooled into a grey `(other)` bar so the tail mass stays visible.

In [ ]:
if CAT_COLS:
    cols = CAT_COLS[:CFG.MAX_PLOT_COLS]
    n_lv = min(CFG.TOP_N_CATS, max(df[c].nunique() for c in cols)) + 1
    fig, axes = make_grid(len(cols), ncols=3, w=5.4, h=0.3 * n_lv + 1.3)
    for i, (ax, c) in enumerate(zip(axes, cols)):
        vc = top_levels(df[c]).value_counts(normalize=True) * 100
        other = vc.pop('(other)') if '(other)' in vc.index else None
        vc = vc.sort_values()
        if other is not None:
            vc = pd.concat([pd.Series({'(other)': other}), vc])
        colors = [MUTED if k == '(other)' else PALETTE[i % len(PALETTE)] for k in vc.index]
        ax.barh([short(k, 22) for k in vc.index], vc.values, color=colors)
        bar_labels(ax, '{:.1f}%', size=7)
        ax.set_title(f'{short(c, 26)}  ({df[c].nunique():,} levels)', loc='left', fontsize=10)
        ax.set_xlabel('% of rows')
        ax.tick_params(axis='y', labelsize=8)
    suptitle(fig, 'Categorical level frequencies', f'top {CFG.TOP_N_CATS} levels per column, grey = pooled tail')
    save_fig(fig, 'categorical_frequencies')

#### Insights

> *Fill in after running.* Which categorical columns are dominated by one level, and which have a long tail of rare levels that should be grouped? Which high-cardinality columns (for example names) need target or frequency encoding instead of one-hot?

---

# Feature vs Target <a name="9"></a>

---

This section quantifies and visualises how each feature relates to the target. Statistical tests are chosen by feature and target type, so every association is backed by a defensible test and an effect size, not only a p-value.

## Association Ranking

The test matrix is as follows. A numeric feature against a regression target uses Spearman rho, which is robust to skew and non-linearity. A numeric feature against a class target uses Kruskal-Wallis with epsilon-squared as effect size. A categorical feature against a regression target uses Kruskal-Wallis across levels. A categorical feature against a class target uses chi-square with bias-corrected Cramer's V. Mutual information is added as a model-free score that also captures non-monotonic relationships, and the features are ranked by it.

In [ ]:
def safe_kruskal(groups):
    groups = [g for g in groups if len(g) > 1]
    if len(groups) < 2:
        return np.nan, np.nan
    try:
        return stats.kruskal(*groups)
    except ValueError:
        return np.nan, np.nan

ASSOC = pd.DataFrame()
if not HAS_TARGET or not FEATURES:
    no_data('needs a TARGET and at least one numeric or categorical feature.')
else:
    yser = df[T]
    ylab = df['__target_label__'] if IS_CLF else None
    rows = []
    for c in NUM_COLS:
        s = df[c]
        m = s.notna() & yser.notna()
        if IS_REG:
            rho, p = stats.spearmanr(s[m], yser[m])
            rows.append({'feature': c, 'type': 'numeric', 'test': 'Spearman rho', 'statistic': rho, 'p_value': p, 'effect': abs(rho)})
        else:
            H, p = safe_kruskal([s[m & (ylab == k)] for k in CLASSES])
            rows.append({'feature': c, 'type': 'numeric', 'test': 'Kruskal-Wallis', 'statistic': H, 'p_value': p, 'effect': H / (m.sum() - 1)})
    for c in CAT_COLS:
        lv = top_levels(df[c], 50)
        if IS_REG:
            H, p = safe_kruskal([yser[lv == k].dropna() for k in lv.unique()])
            rows.append({'feature': c, 'type': 'categorical', 'test': 'Kruskal-Wallis', 'statistic': H, 'p_value': p, 'effect': H / (yser.notna().sum() - 1)})
        else:
            p = stats.chi2_contingency(pd.crosstab(lv, ylab))[1]
            v = cramers_v(lv, ylab)
            rows.append({'feature': c, 'type': 'categorical', 'test': "Cramer's V", 'statistic': v, 'p_value': p, 'effect': v})
    ASSOC = pd.DataFrame(rows).set_index('feature')

    ds = df.dropna(subset=[T]).drop_duplicates(subset=FEATURES + [T])
    ds = sample(ds)
    X = encode_frame(ds, NUM_COLS, CAT_COLS)
    disc = np.array([c in CAT_COLS for c in X.columns])
    if IS_REG:
        mi = mutual_info_regression(X, ds[T].astype(float), discrete_features=disc, random_state=CFG.SEED)
    else:
        mi = mutual_info_classif(X, ds['__target_label__'], discrete_features=disc, random_state=CFG.SEED)
    ASSOC['mutual_info'] = pd.Series(mi, index=X.columns)
    ASSOC['significant'] = ASSOC['p_value'] < 0.05
    ASSOC = ASSOC.sort_values('mutual_info', ascending=False)
    save_table(ASSOC, 'feature_target_association')
    display(ASSOC)

    a = ASSOC.head(CFG.MAX_PLOT_COLS).iloc[::-1]
    type_color = {'numeric': PRIMARY, 'categorical': ACCENT}
    fig, axes = plt.subplots(1, 2, figsize=(16, max(4.5, 0.34 * len(a) + 1.8)), sharey=True)
    axes[0].barh([short(i) for i in a.index], a['mutual_info'], color=[type_color[t] for t in a['type']])
    bar_labels(axes[0], '{:.3f}', size=7)
    axes[0].set_title('Mutual information with target', loc='left')
    eff_colors = [type_color[t] if s else '#D5DBE3' for t, s in zip(a['type'], a['significant'])]
    axes[1].barh([short(i) for i in a.index], a['effect'], color=eff_colors)
    for i, (v, t) in enumerate(zip(a['effect'], a['test'])):
        axes[1].text(v, i, f'  {v:.3f} ({t})', va='center', fontsize=7, color='#4A5568')
    axes[1].set_title('Effect size of the matching test (grey = p >= 0.05)', loc='left')
    handles = [plt.Rectangle((0, 0), 1, 1, color=v) for v in type_color.values()]
    axes[0].legend(handles, type_color.keys(), loc='lower right')
    suptitle(fig, f'Which features carry signal about {T}?', 'ranked by mutual information on a sample')
    save_fig(fig, 'feature_target_association')

ORDERED_NUM = [c for c in ASSOC.index if c in NUM_COLS] if not ASSOC.empty else NUM_COLS
ORDERED_CAT = [c for c in ASSOC.index if c in CAT_COLS] if not ASSOC.empty else CAT_COLS

## Numeric Features vs Target

For regression, each panel is a hexbin density with the mean target per quantile bin overlaid, which reveals non-linear shapes that a single correlation hides. For classification, each panel compares class-conditional densities. Panels are ordered by mutual information.

In [ ]:
if not HAS_TARGET or not NUM_COLS:
    no_data('needs a TARGET and numeric features.')
elif IS_REG:
    cols = ORDERED_NUM[:CFG.MAX_PLOT_COLS]
    ds = sample(df[cols + [T]].dropna(subset=[T]))
    fig, axes = make_grid(len(cols), ncols=4, w=4.4, h=3.5)
    for ax, c in zip(axes, cols):
        d = ds[[c, T]].dropna()
        lo, hi = d[c].quantile([0.005, 0.995])
        d = d[d[c].between(lo, hi)]
        ax.hexbin(d[c], d[T], gridsize=32, cmap=CMAP_SEQ, mincnt=1, linewidths=0, bins='log')
        bins = pd.qcut(d[c], q=20, duplicates='drop')
        binned = d.groupby(bins, observed=True).agg(xm=(c, 'mean'), ym=(T, 'mean'))
        ax.plot(binned['xm'], binned['ym'], color=ACCENT, marker='o', ms=3.5, lw=2)
        rho = ASSOC.loc[c, 'statistic']
        ax.set_title(f'{short(c, 22)}  rho={rho:+.2f}', loc='left', fontsize=10)
        ax.set_xlabel('')
        ax.set_ylabel(T, fontsize=8)
    suptitle(fig, f'Numeric features vs {T}', 'hexbin = log density (0.5-99.5 pct of x), orange = mean target per quantile bin')
    save_fig(fig, 'numeric_vs_target')
else:
    cols = ORDERED_NUM[:CFG.MAX_PLOT_COLS]
    top_classes = CLASSES[:6]
    ds = sample(df[df['__target_label__'].isin(top_classes)])
    fig, axes = make_grid(len(cols), ncols=4, w=4.4, h=3.3)
    for i, (ax, c) in enumerate(zip(axes, cols)):
        lo, hi = ds[c].quantile([0.01, 0.99])
        d = ds[ds[c].between(lo, hi)]
        sns.kdeplot(data=d, x=c, hue='__target_label__', hue_order=top_classes, common_norm=False,
                    fill=True, alpha=0.25, linewidth=1.4, palette=PALETTE[:len(top_classes)],
                    ax=ax, warn_singular=False, legend=(i == 0))
        ax.set_title(f'{short(c, 22)}  eps2={ASSOC.loc[c, "effect"]:.3f}', loc='left', fontsize=10)
        ax.set_xlabel('')
        ax.set_ylabel('')
    suptitle(fig, f'Numeric features by {T} class', 'class-conditional densities, 1st-99th percentile range')
    save_fig(fig, 'numeric_vs_target')

## Categorical Features vs Target

For regression, each panel shows the mean target per level with a 95% confidence interval against the global mean (dashed line). For classification, each panel shows the class mix per level as a 100% stacked bar, so levels that skew toward one class stand out.

In [ ]:
if not HAS_TARGET or not CAT_COLS:
    no_data('needs a TARGET and categorical features.')
else:
    cols = ORDERED_CAT[:CFG.MAX_PLOT_COLS]
    n_lv = min(CFG.TOP_N_CATS, max(df[c].nunique() for c in cols)) + 1
    fig, axes = make_grid(len(cols), ncols=3, w=5.6, h=0.3 * n_lv + 1.4)
    for i, (ax, c) in enumerate(zip(axes, cols)):
        lv = top_levels(df[c])
        if IS_REG:
            gm = df[T].mean()
            g = df.groupby(lv)[T].agg(['mean', 'count', 'sem'])
            g = g[g['count'] >= 10].sort_values('mean')
            colors = [PRIMARY if v >= gm else ACCENT for v in g['mean']]
            ax.barh([short(k, 22) for k in g.index], g['mean'], xerr=1.96 * g['sem'].fillna(0), color=colors,
                    error_kw=dict(ecolor=DARK, lw=0.8, capsize=2, alpha=0.6))
            ax.axvline(gm, color=DARK, ls='--', lw=1)
            ax.set_xlabel(f'mean {T}')
        else:
            ct = pd.crosstab(lv, df['__target_label__'], normalize='index')[CLASSES[:10]]
            ct = ct.loc[lv.value_counts().index[::-1]]
            left = np.zeros(len(ct))
            for k, col in zip(ct.columns, PALETTE):
                ax.barh([short(x, 22) for x in ct.index], ct[k], left=left, color=col, label=k, edgecolor='white', linewidth=0.5)
                left += ct[k].values
            ax.set_xlim(0, 1)
            ax.xaxis.set_major_formatter(mticker.PercentFormatter(1.0))
            if i == 0:
                ax.legend(title=T, bbox_to_anchor=(0, 1.08), loc='lower left', ncol=min(5, len(ct.columns)), fontsize=8)
        ax.set_title(f'{short(c, 26)}  effect={ASSOC.loc[c, "effect"]:.3f}', loc='left', fontsize=10)
        ax.tick_params(axis='y', labelsize=8)
    sub = 'blue = above global mean, orange = below, whiskers = 95% CI, levels with n < 10 hidden' if IS_REG else 'class mix per level, levels sorted by frequency'
    suptitle(fig, f'Categorical features vs {T}', sub)
    save_fig(fig, 'categorical_vs_target')

#### Insights

> *Fill in after running.* Which features carry the strongest signal, and is it monotonic (trust Spearman) or curved (trust MI and the binned line)? With a large n almost every test is significant, so the narrative should rely on effect sizes.

> Which categorical levels sit far above or below the global mean? These are the business-facing findings (for example 'genre X averages 20 points higher').

---

# Multivariate Structure <a name="10"></a>

---

Features rarely act alone. This section looks at redundancy (correlation, VIF, Cramer's V), natural feature groups (hierarchical clustering), joint structure with the target (pairplot), and the dimensionality of the numeric space (PCA).

## Correlation Matrices

Pearson captures linear association and Spearman captures monotonic association. A large gap between the two for a pair signals a non-linear or outlier-driven relationship.

In [ ]:
corr_cols = ORDERED_NUM[:CFG.MAX_PLOT_COLS] + ([T] if IS_REG else [])
if len(corr_cols) < 2:
    no_data('fewer than two numeric columns.')
    corr_s = pd.DataFrame()
else:
    corr_p = df[corr_cols].corr('pearson')
    corr_s = df[corr_cols].corr('spearman')
    save_table(corr_p, 'corr_pearson')
    save_table(corr_s, 'corr_spearman')
    n = len(corr_cols)
    mask = np.triu(np.ones((n, n), dtype=bool), k=1)
    size = max(6, 0.5 * n + 2)
    fig, axes = plt.subplots(1, 2, figsize=(2 * size + 1, size))
    for ax, cm, name in zip(axes, (corr_p, corr_s), ('Pearson', 'Spearman')):
        sns.heatmap(cm, mask=mask, cmap=CMAP_DIV, vmin=-1, vmax=1, center=0, square=True,
                    annot=n <= 18, fmt='.2f', annot_kws={'size': 7}, linewidths=0.5, linecolor='white',
                    cbar_kws={'shrink': 0.6}, ax=ax,
                    xticklabels=[short(c, 18) for c in corr_cols], yticklabels=[short(c, 18) for c in corr_cols])
        ax.set_title(f'{name} correlation', loc='left')
    suptitle(fig, 'Correlation structure', 'blue = positive, orange = negative')
    save_fig(fig, 'correlation_matrices')

## Highly Correlated Pairs and VIF

Pairs above `HIGH_CORR` (absolute Spearman) are listed explicitly. The variance inflation factor, computed as the diagonal of the inverse correlation matrix, measures how well each feature is explained by all others together. VIF above 5 is a warning and above 10 is severe multicollinearity, which matters for linear models and coefficient interpretation.

In [ ]:
if corr_s.empty:
    no_data('fewer than two numeric columns.')
    high_pairs, vif = pd.DataFrame(), pd.Series(dtype=float)
else:
    cs = df[NUM_COLS].corr('spearman') if len(NUM_COLS) > 1 else corr_s
    iu = np.triu_indices_from(cs, k=1)
    pairs = pd.DataFrame({'feature_a': cs.index[iu[0]], 'feature_b': cs.columns[iu[1]], 'spearman': cs.values[iu]})
    pairs['abs'] = pairs['spearman'].abs()
    pairs = pairs.sort_values('abs', ascending=False)
    high_pairs = pairs[pairs['abs'] >= CFG.HIGH_CORR]
    save_table(pairs, 'correlated_pairs', index=False)
    print(f'Pairs with |rho| >= {CFG.HIGH_CORR}: {len(high_pairs)}  (top 10 pairs shown)')
    display(pairs.head(10))

    Xv = df[NUM_COLS].astype(float)
    Xv = Xv.fillna(Xv.median())
    Xv = Xv.loc[:, Xv.std() > 0]
    if Xv.shape[1] >= 2:
        R = np.corrcoef(Xv.values, rowvar=False)
        vif = pd.Series(np.diag(np.linalg.pinv(R)), index=Xv.columns).sort_values()
        save_table(vif.rename('vif').to_frame(), 'vif')
        v = vif.tail(CFG.MAX_PLOT_COLS)
        fig, ax = plt.subplots(figsize=(11, max(3.5, 0.32 * len(v) + 1.5)))
        colors = [ACCENT if x > 10 else PALETTE[3] if x > 5 else PALETTE[2] for x in v.values]
        ax.barh([short(c) for c in v.index], v.values, color=colors)
        for x in (5, 10):
            ax.axvline(x, color=MUTED, ls='--', lw=1)
        bar_labels(ax, '{:.1f}', size=8)
        ax.set_xlabel('VIF')
        suptitle(fig, 'Variance inflation factor', 'green < 5 < yellow < 10 < orange')
        save_fig(fig, 'vif')

## Feature Clustering

Hierarchical clustering on the distance `1 - |Spearman rho|` groups features that carry overlapping information. Branches joined below the `1 - HIGH_CORR` cut line are candidates for keeping one representative or combining into an index.

In [ ]:
if len(NUM_COLS) < 3:
    no_data('needs at least three numeric columns.')
else:
    cs = df[NUM_COLS].corr('spearman').fillna(0)
    dist = (1 - cs.abs()).clip(lower=0).to_numpy(copy=True)
    np.fill_diagonal(dist, 0)
    Z = hierarchy.linkage(squareform(dist, checks=False), method='average')
    hierarchy.set_link_color_palette(PALETTE[1:])
    fig, ax = plt.subplots(figsize=(11, max(4, 0.3 * len(NUM_COLS) + 1.5)))
    hierarchy.dendrogram(Z, labels=[short(c) for c in cs.index], orientation='left', ax=ax,
                         color_threshold=1 - CFG.HIGH_CORR + 0.15, above_threshold_color=MUTED)
    ax.axvline(1 - CFG.HIGH_CORR, color=ACCENT, ls='--', lw=1)
    ax.set_xlabel('1 - |Spearman rho|')
    ax.grid(False)
    suptitle(fig, 'Feature clustering dendrogram', 'orange line = HIGH_CORR cut')
    save_fig(fig, 'feature_dendrogram')

## Categorical Association Matrix

Cramer's V (bias-corrected) measures association between pairs of categorical columns on a 0 to 1 scale. Columns with more than 100 levels are excluded to keep the statistic stable.

In [ ]:
cc = [c for c in CAT_COLS if df[c].nunique() <= 100][:15] + (['__target_label__'] if IS_CLF else [])
if len(cc) < 2:
    no_data('fewer than two low-cardinality categorical columns.')
else:
    ds = sample(df[cc], 50_000)
    M = pd.DataFrame(np.eye(len(cc)), index=cc, columns=cc)
    for i, a in enumerate(cc):
        for b in cc[i + 1:]:
            M.loc[a, b] = M.loc[b, a] = cramers_v(top_levels(ds[a], 100), top_levels(ds[b], 100))
    labels = [T if c == '__target_label__' else short(c, 18) for c in cc]
    save_table(M.rename(index=dict(zip(cc, labels)), columns=dict(zip(cc, labels))), 'cramers_v')
    size = max(5, 0.55 * len(cc) + 2)
    fig, ax = plt.subplots(figsize=(size + 1, size))
    sns.heatmap(M, cmap=CMAP_SEQ, vmin=0, vmax=1, square=True, annot=len(cc) <= 15, fmt='.2f',
                annot_kws={'size': 8}, linewidths=0.5, linecolor='white', cbar_kws={'shrink': 0.6},
                xticklabels=labels, yticklabels=labels, ax=ax)
    suptitle(fig, "Cramer's V between categorical columns")
    save_fig(fig, 'cramers_v')

## Pairplot of Top Features

The `PAIRPLOT_K` numeric features with the highest mutual information are plotted pairwise on a 2,000-row sample, coloured by class or by target quartile. This exposes interactions and clusters that univariate plots cannot show.

In [ ]:
k_cols = ORDERED_NUM[:CFG.PAIRPLOT_K]
if len(k_cols) < 2:
    no_data('needs at least two numeric columns.')
else:
    ds = sample(df, 2000).copy()
    hue, hue_order, pal = None, None, None
    if IS_CLF:
        hue_order = CLASSES[:6]
        ds = ds[ds['__target_label__'].isin(hue_order)]
        hue, pal = '__target_label__', PALETTE[:len(hue_order)]
    elif IS_REG:
        q = pd.qcut(ds[T].rank(method='first'), 4, labels=['Q1 (low)', 'Q2', 'Q3', 'Q4 (high)'])
        ds[f'{T} quartile'] = q.astype(str)
        hue, hue_order = f'{T} quartile', ['Q1 (low)', 'Q2', 'Q3', 'Q4 (high)']
        pal = ['#3D5A80', '#98C1D9', '#F4A261', '#9D0208']
    plot_cols = k_cols + ([hue] if hue else [])
    g = sns.pairplot(ds[plot_cols], hue=hue, hue_order=hue_order, palette=pal, corner=True, height=2.4,
                     diag_kind='kde', plot_kws=dict(s=9, alpha=0.55, edgecolor='none'),
                     diag_kws=dict(common_norm=False, fill=True, alpha=0.3, warn_singular=False))
    g.fig.subplots_adjust(top=0.9)
    suptitle(g.fig, f'Top {len(k_cols)} features by mutual information', 'pairwise view on a 2,000-row sample', layout=False)
    save_fig(g.fig, 'pairplot_top_features')

## Principal Component Analysis

PCA on standardised numeric features shows how many independent directions the data really has (scree plot), whether the target separates in the first two components (projection), and which features define those components (loadings).

In [ ]:
if len(NUM_COLS) < 2:
    no_data('needs at least two numeric columns.')
else:
    Xp = df[NUM_COLS].astype(float)
    Xp = Xp.fillna(Xp.median())
    Xp = Xp.loc[:, Xp.std() > 0]
    Zs = StandardScaler().fit_transform(Xp)
    pca = PCA(random_state=CFG.SEED).fit(Zs)
    ev = pca.explained_variance_ratio_
    n80 = int(np.searchsorted(np.cumsum(ev), 0.8) + 1)

    fig, axes = plt.subplots(1, 3, figsize=(19, 5.2))
    ax = axes[0]
    ax.bar(range(1, len(ev) + 1), ev * 100, color=SOFT, edgecolor=PRIMARY)
    ax2 = ax.twinx()
    ax2.plot(range(1, len(ev) + 1), np.cumsum(ev) * 100, color=ACCENT, marker='o', ms=4)
    ax2.axhline(80, color=MUTED, ls='--', lw=1)
    ax2.set_ylim(0, 105)
    ax2.grid(False)
    ax.set_xlabel('component')
    ax.set_ylabel('% variance')
    ax.set_title(f'Scree plot ({n80} PCs reach 80%)', loc='left')

    pos = np.random.RandomState(CFG.SEED).choice(len(Zs), min(5000, len(Zs)), replace=False)
    P = pca.transform(Zs[pos])[:, :2]
    ax = axes[1]
    if IS_REG:
        sc = ax.scatter(P[:, 0], P[:, 1], c=df[T].iloc[pos], cmap=CMAP_HEAT, s=6, alpha=0.6)
        fig.colorbar(sc, ax=ax, shrink=0.8, label=T)
    elif IS_CLF:
        lab = df['__target_label__'].iloc[pos].values
        for k, col in zip(CLASSES[:8], PALETTE):
            m = lab == k
            ax.scatter(P[m, 0], P[m, 1], s=6, alpha=0.5, color=col, label=short(k, 15))
        ax.legend(markerscale=3, fontsize=8)
    else:
        ax.scatter(P[:, 0], P[:, 1], s=6, alpha=0.4, color=PRIMARY)
    ax.set_xlabel(f'PC1 ({ev[0]:.1%})')
    ax.set_ylabel(f'PC2 ({ev[1]:.1%})')
    ax.set_title('Projection on PC1-PC2', loc='left')

    load = pd.DataFrame(pca.components_[:2].T, index=Xp.columns, columns=['PC1', 'PC2'])
    top = load.abs().max(axis=1).sort_values().tail(12).index
    load.loc[top].rename(index=short).plot.barh(ax=axes[2], color=[PRIMARY, ACCENT], width=0.8)
    axes[2].axvline(0, color=MUTED, lw=1)
    axes[2].set_title('Loadings of PC1 and PC2', loc='left')
    save_table(load, 'pca_loadings')
    suptitle(fig, 'Principal component analysis', f'{Xp.shape[1]} standardised numeric features')
    save_fig(fig, 'pca')

#### Insights

> *Fill in after running.* Which features are redundant (high rho, high VIF, same dendrogram branch), and which representative will be kept? How many components carry 80% of the variance, which indicates whether the feature space is genuinely multi-dimensional?

> Does the target separate in the pairplot or the PCA projection? If not, the signal probably lives in categorical, text, or interaction features rather than raw numerics.

---

# Temporal and Segment Analysis <a name="11"></a>

---

When the data has a time axis (`TIME_COL`, either a datetime or a year-like number), this section tracks data volume and target behaviour over time, then crosses time with a segment column (`GROUP_COL`) to separate stable segments from volatile ones. This is the standard tool for questions such as 'which genre, product, or region is evergreen?'

## Time Axis Preparation

Datetime columns are bucketed into months when the span is five years or less and into years otherwise. Numeric year columns are restricted to `TIME_RANGE`, and the rows outside it are reported, because placeholder years (0, 1, 1900) are common data errors. For a classification target, the tracked metric is the rate of the rarest class.

In [ ]:
HAS_TIME = bool(CFG.TIME_COL) and CFG.TIME_COL in df.columns
if not HAS_TIME:
    no_data('TIME_COL is not set or not found.')
else:
    tc = df[CFG.TIME_COL]
    if not pd.api.types.is_numeric_dtype(tc) and not pd.api.types.is_datetime64_any_dtype(tc):
        tc = pd.to_datetime(tc, errors='coerce')
    if pd.api.types.is_datetime64_any_dtype(tc):
        span_years = (tc.max() - tc.min()).days / 365.25
        freq = 'M' if span_years <= 5 else 'Y'
        df['__period__'] = tc.dt.to_period(freq).dt.to_timestamp()
        bad = int(tc.isna().sum())
    else:
        valid = tc.between(*CFG.TIME_RANGE)
        df['__period__'] = tc.where(valid).round()
        bad = int((~valid).sum())
        print('Out-of-range values:', tc[~valid].value_counts().head(10).to_dict())
    print(f'Rows without a valid period: {bad:,}')

METRIC_NAME = None
if HAS_TARGET:
    if IS_REG:
        df['__metric__'] = df[T].astype(float)
        METRIC_NAME = f'mean {T}'
    else:
        POS = CLASSES[-1]
        df['__metric__'] = (df['__target_label__'] == POS).astype(float)
        METRIC_NAME = f'rate of class {POS!r}'

## Volume and Target over Time

The upper panel shows how many rows each period contributes, which flags thin periods whose averages are unreliable. The lower panel shows the target metric with an interquartile band. Periods below `MIN_PERIOD_N` rows are drawn as hollow markers.

In [ ]:
if HAS_TIME:
    per = df.dropna(subset=['__period__']).groupby('__period__')
    counts = per.size()
    fig, axes = plt.subplots(2, 1, figsize=(15, 8), sharex=True, gridspec_kw={'height_ratios': [1, 1.6]})
    is_dt = pd.api.types.is_datetime64_any_dtype(counts.index)
    width = 20 if is_dt else 0.8
    colors = [PRIMARY if n >= CFG.MIN_PERIOD_N else '#D5DBE3' for n in counts.values]
    axes[0].bar(counts.index, counts.values, color=colors, width=width)
    axes[0].set_ylabel('rows')
    axes[0].set_title(f'Rows per period (grey = fewer than {CFG.MIN_PERIOD_N})', loc='left')
    if METRIC_NAME:
        agg = per['__metric__'].agg(['mean', 'median', lambda s: s.quantile(0.25), lambda s: s.quantile(0.75)])
        agg.columns = ['mean', 'median', 'q25', 'q75']
        ok = counts >= CFG.MIN_PERIOD_N
        ax = axes[1]
        if IS_REG:
            ax.fill_between(agg.index, agg['q25'], agg['q75'], color=SOFT, alpha=0.35, label='IQR')
            ax.plot(agg.index, agg['median'], color=PRIMARY, lw=1.2, ls=':', label='median')
        ax.plot(agg.index, agg['mean'], color=ACCENT, lw=1.6, label='mean')
        ax.scatter(agg.index[ok], agg['mean'][ok], color=ACCENT, s=18, zorder=3)
        ax.scatter(agg.index[~ok], agg['mean'][~ok], facecolor='white', edgecolor=ACCENT, s=18, zorder=3)
        ax.axhline(df['__metric__'].mean(), color=MUTED, ls='--', lw=1)
        ax.set_ylabel(METRIC_NAME)
        ax.legend(loc='upper left', ncol=3)
        ax.set_title(f'{METRIC_NAME} per period (hollow = thin period)', loc='left')
        save_table(agg.assign(rows=counts), 'target_over_time')
    else:
        axes[1].set_visible(False)
    suptitle(fig, f'Temporal profile by {CFG.TIME_COL}', f'{len(counts)} periods, {counts.sum():,} rows with a valid period')
    save_fig(fig, 'temporal_profile')

## Segment by Period Heatmap

The heatmap crosses the most frequent `GROUP_COL` segments with time. Cells with fewer than five rows are left blank so that noise is not mistaken for a trend. Yearly axes with more than 40 periods are pooled into five-year bins for readability.

In [ ]:
HAS_SEG = HAS_TIME and METRIC_NAME is not None and bool(CFG.GROUP_COL) and CFG.GROUP_COL in df.columns
if not HAS_SEG:
    no_data('needs TIME_COL, TARGET, and GROUP_COL.')
else:
    d = df.dropna(subset=['__period__']).copy()
    is_dt = pd.api.types.is_datetime64_any_dtype(d['__period__'])
    if not is_dt and d['__period__'].nunique() > 40:
        d['__bin__'] = (d['__period__'] // 5 * 5).astype(int)
        bin_label = '5-year bin'
    else:
        d['__bin__'] = d['__period__'].dt.year if is_dt and d['__period__'].nunique() > 40 else d['__period__']
        bin_label = 'period'
    top_groups = d[CFG.GROUP_COL].value_counts().index[:20]
    d = d[d[CFG.GROUP_COL].isin(top_groups)]
    piv_mean = d.pivot_table(index=CFG.GROUP_COL, columns='__bin__', values='__metric__', aggfunc='mean')
    piv_n = d.pivot_table(index=CFG.GROUP_COL, columns='__bin__', values='__metric__', aggfunc='size')
    piv = piv_mean.where(piv_n >= 5)
    piv = piv.loc[piv.mean(axis=1).sort_values(ascending=False).index]
    if is_dt and bin_label == 'period':
        piv.columns = [str(c)[:7] for c in piv.columns]
    fig, ax = plt.subplots(figsize=(max(10, 0.42 * piv.shape[1] + 4), 0.38 * len(piv) + 2))
    sns.heatmap(piv, cmap=CMAP_HEAT, linewidths=0.4, linecolor='white', cbar_kws={'shrink': 0.7, 'label': METRIC_NAME},
                annot=piv.shape[1] <= 16, fmt='.0f' if IS_REG else '.2f', annot_kws={'size': 7}, ax=ax)
    ax.set_xlabel(bin_label)
    ax.set_ylabel('')
    ax.set_yticklabels([short(t.get_text(), 24) for t in ax.get_yticklabels()])
    suptitle(fig, f'{METRIC_NAME} by {CFG.GROUP_COL} over time', 'top 20 segments by size, blank = fewer than 5 rows')
    save_fig(fig, 'segment_time_heatmap')

## Segment Stability Quadrant

For every segment with at least three reliable periods, the per-period metric is summarised by its level (mean), its volatility (coefficient of variation across periods), and its trend (Theil-Sen slope with a Mann-Kendall-style Kendall tau test). Plotting level against volatility gives four quadrants: evergreen (high and stable), hit-driven (high and volatile), stable niche (low and stable), and speculative (low and volatile).

In [ ]:
seg_table = pd.DataFrame()
if HAS_SEG:
    d = df.dropna(subset=['__period__'])
    if pd.api.types.is_datetime64_any_dtype(d['__period__']):
        d = d.assign(__t__=d['__period__'].dt.year + d['__period__'].dt.month / 12)
    else:
        d = d.assign(__t__=d['__period__'])
    rows = []
    for gname, gdf in d.groupby(CFG.GROUP_COL):
        pm = gdf.groupby('__t__')['__metric__'].agg(['mean', 'size'])
        pm = pm[pm['size'] >= 5]
        if len(pm) < 3:
            continue
        tau, p = stats.kendalltau(pm.index, pm['mean'])
        slope = stats.theilslopes(pm['mean'], pm.index)[0]
        level = gdf['__metric__'].mean()
        rows.append({'segment': gname, 'rows': len(gdf), 'periods': len(pm), 'level': level,
                     'period_std': pm['mean'].std(), 'cv': pm['mean'].std() / (abs(pm['mean'].mean()) + 1e-9),
                     'trend_slope': slope, 'kendall_tau': tau, 'trend_p': p})
    seg_table = pd.DataFrame(rows).set_index('segment') if rows else pd.DataFrame()

if HAS_SEG and seg_table.empty:
    no_data('no segment has three or more periods with at least 5 rows.')
elif HAS_SEG:
    lv_med, cv_med = seg_table['level'].median(), seg_table['cv'].median()
    def quadrant(r):
        hi, stable = r['level'] >= lv_med, r['cv'] <= cv_med
        return {(True, True): 'evergreen', (True, False): 'hit-driven', (False, True): 'stable niche', (False, False): 'speculative'}[(hi, stable)]
    seg_table['quadrant'] = seg_table.apply(quadrant, axis=1)
    seg_table['trend'] = np.where(seg_table['trend_p'] < 0.05, np.where(seg_table['kendall_tau'] > 0, 'rising', 'falling'), 'flat')
    seg_table = seg_table.sort_values('level', ascending=False)
    save_table(seg_table, 'segment_stability')
    display(seg_table)

    trend_color = {'rising': PALETTE[2], 'falling': ACCENT, 'flat': PRIMARY}
    fig, ax = plt.subplots(figsize=(13, 8.5))
    sizes = 40 + 600 * seg_table['rows'] / seg_table['rows'].max()
    ax.scatter(seg_table['cv'], seg_table['level'], s=sizes, c=[trend_color[t] for t in seg_table['trend']],
               alpha=0.7, edgecolor='white', linewidth=1.2)
    for name, r in seg_table.iterrows():
        ax.annotate(short(name, 20), (r['cv'], r['level']), fontsize=8, color=DARK,
                    xytext=(5, 4), textcoords='offset points')
    ax.margins(x=0.12, y=0.1)
    ax.axvline(cv_med, color=MUTED, ls='--', lw=1)
    ax.axhline(lv_med, color=MUTED, ls='--', lw=1)
    box = dict(boxstyle='round,pad=0.3', fc='white', ec='#E2E8F0')
    for (x, y, ha, va), lab in zip([(0.01, 0.99, 'left', 'top'), (0.99, 0.99, 'right', 'top'),
                                    (0.01, 0.01, 'left', 'bottom'), (0.99, 0.01, 'right', 'bottom')],
                                   ['EVERGREEN', 'HIT-DRIVEN', 'STABLE NICHE', 'SPECULATIVE']):
        ax.text(x, y, lab, transform=ax.transAxes, ha=ha, va=va, fontsize=10, fontweight='bold', color=MUTED, bbox=box)
    handles = [plt.Line2D([], [], marker='o', ls='', color=v, markersize=9, label=k) for k, v in trend_color.items()]
    ax.legend(handles=handles, title='trend (p < 0.05)', loc='center right')
    ax.set_xlabel('volatility: coefficient of variation across periods')
    ax.set_ylabel(METRIC_NAME)
    suptitle(fig, f'{CFG.GROUP_COL} stability quadrant', 'bubble size = rows, dashed lines = medians')
    save_fig(fig, 'segment_stability_quadrant')

#### Insights

> *Fill in after running.* Is the time axis evenly populated, or do a few periods dominate? Thin periods must not drive conclusions. Is there a global trend in the target?

> Which segments are evergreen, hit-driven, stable niche, or speculative, and which are significantly rising or falling? Note that a snapshot metric (for example current a target against the year it was recorded) can measure survival in the data rather than the historical trend, and should be framed that way in the report.

---

# Text Columns Snapshot <a name="12"></a>

---

Free-text columns are profiled lightly here: length distributions, the relationship between length and the target, and the most frequent tokens. Full NLP (topics, sentiment, embeddings) belongs in the NLP template, but length alone is often a useful tabular feature and a quick sanity check of text quality.

In [ ]:
STOPWORDS = set('''a an the and or but if of to in on at by for with from as is are was were be been being
i me my we our you your he him his she her it its they them their this that these those there here
not no so do does did have has had will would can could should just than then too very all any some
what when where who which how up down out over into about oh yeah im dont its ill ive youre cant
s t m re ll ve d na la da'''.split())

def tokens(text):
    text = re.sub(r'\[[^\]]*\]', ' ', str(text).lower())
    return [w for w in re.findall(r"[a-z][a-z']+", text.replace("'", '')) if w not in STOPWORDS]

text_stats = pd.DataFrame()
if not TEXT_COLS:
    no_data('no text columns detected (set TEXT_COLS to force one).')
else:
    rows = []
    for c in TEXT_COLS[:3]:
        s = df[c].fillna('').astype(str)
        words = s.str.split().str.len()
        df[f'__len_{c}__'] = words
        rows.append({'column': c, 'empty_%': (s.str.strip() == '').mean() * 100,
                     'median_words': words.median(), 'p95_words': words.quantile(0.95),
                     'duplicate_text_%': s.duplicated().mean() * 100,
                     'non_ascii_heavy_%': (s.str.count(r'[^\x00-\x7f]') / s.str.len().clip(lower=1) > 0.2).mean() * 100})

        cnt = Counter()
        for t in sample(s.to_frame(), 5000)[c]:
            cnt.update(tokens(t))
        top_tok = pd.Series(dict(cnt.most_common(20))).sort_values()

        fig, axes = plt.subplots(1, 3, figsize=(18, 5))
        sns.histplot(words.clip(upper=words.quantile(0.99)), bins=50, color=PRIMARY, edgecolor='white', ax=axes[0])
        axes[0].axvline(words.median(), color=ACCENT, ls='--', lw=1.5)
        axes[0].set_title('Word count (clipped at p99)', loc='left')
        ax = axes[1]
        if IS_REG:
            d = pd.DataFrame({'w': words, 'y': df[T]}).dropna()
            b = d.groupby(pd.qcut(d['w'].rank(method='first'), 20), observed=True).agg(w=('w', 'median'), y=('y', 'mean'))
            ax.plot(b['w'], b['y'], color=ACCENT, marker='o', lw=2)
            ax.set_xlabel('word count (median of bin)')
            ax.set_ylabel(f'mean {T}')
            rho = stats.spearmanr(d['w'], d['y'])[0]
            ax.set_title(f'Target vs length (rho={rho:+.2f})', loc='left')
        elif IS_CLF:
            d = pd.DataFrame({'w': words.clip(upper=words.quantile(0.99)), 'k': df['__target_label__']})
            sns.boxplot(data=d[d['k'].isin(CLASSES[:8])], x='w', y='k', order=CLASSES[:8], palette=PALETTE, ax=ax, fliersize=1)
            ax.set_title('Length by class', loc='left')
        else:
            ax.set_visible(False)
        axes[2].barh(top_tok.index, top_tok.values, color=PALETTE[2])
        axes[2].set_title('Top tokens (stopwords and [headers] removed)', loc='left')
        suptitle(fig, f'Text column: {c}', f'median {words.median():.0f} words, {rows[-1]["empty_%"]:.1f}% empty')
        save_fig(fig, f'text_{c}')
    text_stats = pd.DataFrame(rows).set_index('column')
    save_table(text_stats, 'text_stats')
    display(text_stats)

#### Insights

> *Fill in after running.* Is the text clean (few empties, few duplicates, consistent language)? Does length relate to the target? If so, length and simple lexical counts are cheap features for the tabular model before moving to embeddings.

---

# Train vs Test Drift <a name="13"></a>

---

When a separate test file is provided, its distribution is compared with the training data. Per-feature drift is measured with the Kolmogorov-Smirnov statistic and the Population Stability Index (PSI below 0.1 is stable, 0.1 to 0.25 is moderate, above 0.25 is major). Adversarial validation then trains a classifier to tell train from test, where an AUC near 0.5 means the two sets are indistinguishable.

In [ ]:
def psi(expected, actual, bins=10):
    edges = np.unique(np.quantile(expected.dropna(), np.linspace(0, 1, bins + 1)))
    if len(edges) < 3:
        return 0.0
    e = np.histogram(np.clip(expected.dropna(), edges[0], edges[-1]), edges)[0] / expected.notna().sum()
    a = np.histogram(np.clip(actual.dropna(), edges[0], edges[-1]), edges)[0] / max(actual.notna().sum(), 1)
    e, a = np.clip(e, 1e-4, None), np.clip(a, 1e-4, None)
    # PSI = sum((a - e) * ln(a / e))
    return float(np.sum((a - e) * np.log(a / e)))

drift, ADV_AUC = pd.DataFrame(), None
if test_df is None:
    no_data('no test file configured (set TEST_*_PATH in Settings).')
else:
    only_train = sorted(set(df.columns) - set(test_df.columns) - {T, '__target_label__', '__period__', '__metric__'})
    only_test = sorted(set(test_df.columns) - set(df.columns))
    print(f'Columns only in train: {[c for c in only_train if not c.startswith("__")]}')
    print(f'Columns only in test : {only_test}')
    num_c = [c for c in NUM_COLS if c in test_df.columns]
    cat_c = [c for c in CAT_COLS if c in test_df.columns]
    rows = []
    for c in num_c:
        ks = stats.ks_2samp(df[c].dropna(), test_df[c].dropna())
        rows.append({'feature': c, 'type': 'numeric', 'ks_stat': ks.statistic, 'ks_p': ks.pvalue, 'psi': psi(df[c], test_df[c]),
                     'unseen_levels_%': np.nan, 'missing_train_%': df[c].isna().mean() * 100, 'missing_test_%': test_df[c].isna().mean() * 100})
    for c in cat_c:
        seen = set(df[c].dropna().unique())
        unseen = (~test_df[c].dropna().isin(seen)).mean() * 100
        p_tr = df[c].value_counts(normalize=True)
        p_te = test_df[c].value_counts(normalize=True).reindex(p_tr.index).fillna(0)
        tvd = 0.5 * np.abs(p_tr - p_te).sum()
        rows.append({'feature': c, 'type': 'categorical', 'ks_stat': tvd, 'ks_p': np.nan, 'psi': np.nan,
                     'unseen_levels_%': unseen, 'missing_train_%': df[c].isna().mean() * 100, 'missing_test_%': test_df[c].isna().mean() * 100})
    drift = pd.DataFrame(rows).set_index('feature').sort_values('ks_stat', ascending=False)
    save_table(drift, 'train_test_drift')
    display(drift)

    feats = num_c + cat_c
    both = pd.concat([df[feats].assign(__is_test__=0), test_df[feats].assign(__is_test__=1)], ignore_index=True)
    both = sample(both, 60_000)
    Xa = encode_frame(both, num_c, cat_c)
    ya = both['__is_test__'].values
    oof = np.zeros(len(ya))
    for tr, va in StratifiedKFold(5, shuffle=True, random_state=CFG.SEED).split(Xa, ya):
        clf = HistGradientBoostingClassifier(max_iter=200, learning_rate=0.08, random_state=CFG.SEED,
                                             categorical_features=np.array([c in cat_c for c in Xa.columns]))
        clf.fit(Xa.iloc[tr], ya[tr])
        oof[va] = clf.predict_proba(Xa.iloc[va])[:, 1]
    ADV_AUC = roc_auc_score(ya, oof)

    top = [c for c in drift.index if c in num_c][:6]
    fig = plt.figure(figsize=(18, 9))
    gs = fig.add_gridspec(2, 4)
    ax = fig.add_subplot(gs[:, :2])
    p = drift['psi'].dropna().sort_values().tail(CFG.MAX_PLOT_COLS)
    ax.barh([short(i) for i in p.index], p.values, color=[ACCENT if v > 0.25 else PALETTE[3] if v > 0.1 else PALETTE[2] for v in p.values])
    bar_labels(ax, '{:.3f}', size=8)
    for v in (0.1, 0.25):
        ax.axvline(v, color=MUTED, ls='--', lw=1)
    ax.set_title(f'PSI per numeric feature | adversarial AUC = {ADV_AUC:.3f}', loc='left')
    for i, c in enumerate(top[:4]):
        a2 = fig.add_subplot(gs[i // 2, 2 + i % 2])
        lo, hi = df[c].quantile([0.01, 0.99])
        sns.kdeplot(df[c].clip(lo, hi), ax=a2, color=PRIMARY, fill=True, alpha=0.3, label='train', warn_singular=False)
        sns.kdeplot(test_df[c].clip(lo, hi), ax=a2, color=ACCENT, fill=True, alpha=0.3, label='test', warn_singular=False)
        a2.set_title(f'{short(c, 20)}  KS={drift.loc[c, "ks_stat"]:.3f}', loc='left', fontsize=10)
        a2.set_xlabel('')
        a2.legend(fontsize=8)
    suptitle(fig, 'Train vs test drift', 'green < 0.1 < yellow < 0.25 < orange (PSI), right = most shifted features')
    save_fig(fig, 'train_test_drift')
    print(f'Adversarial validation AUC: {ADV_AUC:.3f}  (0.5 = no drift, > 0.7 = strong drift)')

#### Insights

> *Fill in after running.* Is the test set drawn from the same distribution? Features with major PSI or unseen levels need robust encoding, and if the adversarial AUC is high the validation scheme should mimic the shift (for example a time-based split).

---

# Baseline Signal Check <a name="14"></a>

---

A quick gradient boosting model with no tuning answers the question every EDA should end with: how much predictable signal do the features hold? The model is evaluated against a naive baseline (mean or majority class). When `CV_GROUP` is set, the same model is scored with random K-fold and with group K-fold. A large gap between the two is direct evidence of leakage through duplicated entities, and the group score is the honest one.

In [ ]:
BASE = {}
if not (CFG.RUN_BASELINE and HAS_TARGET and FEATURES):
    no_data('RUN_BASELINE is False, or no TARGET or features.')
else:
    d = df.dropna(subset=[T]).reset_index(drop=True)
    if IS_CLF:
        vc = d['__target_label__'].value_counts()
        rare = vc[vc < 2 * CFG.N_FOLDS].index.tolist()
        if rare:
            print(f'[warn] classes with fewer than {2 * CFG.N_FOLDS} rows excluded from the baseline: {rare}')
            d = d[~d['__target_label__'].isin(rare)].reset_index(drop=True)
    X = encode_frame(d, NUM_COLS, CAT_COLS)
    cat_mask = np.array([c in CAT_COLS for c in X.columns])
    if IS_REG:
        y = d[T].astype(float).values
    else:
        y = d['__target_label__'].astype('category').cat.codes.values
        class_names = d['__target_label__'].astype('category').cat.categories.tolist()

    def make_model():
        kw = dict(max_iter=300, learning_rate=0.06, random_state=CFG.SEED, categorical_features=cat_mask)
        return HistGradientBoostingRegressor(**kw) if IS_REG else HistGradientBoostingClassifier(**kw)

    def scores(y_true, pred, proba=None):
        if IS_REG:
            return {'RMSE': float(np.sqrt(np.mean((y_true - pred) ** 2))), 'MAE': mean_absolute_error(y_true, pred), 'R2': r2_score(y_true, pred)}
        out = {'macro_F1': f1_score(y_true, pred, average='macro')}
        if proba is not None:
            out['ROC_AUC'] = roc_auc_score(y_true, proba[:, 1]) if proba.shape[1] == 2 else roc_auc_score(y_true, proba, multi_class='ovr')
        return out

    def run_cv(splitter, groups=None):
        oof = np.zeros(len(y))
        oof_p = np.zeros((len(y), len(np.unique(y)))) if IS_CLF else None
        model = None
        for tr, va in splitter.split(X, y, groups):
            model = make_model().fit(X.iloc[tr], y[tr])
            oof[va] = model.predict(X.iloc[va])
            if IS_CLF:
                oof_p[np.ix_(va, model.classes_)] = model.predict_proba(X.iloc[va])
        return oof, oof_p, model, va

    if IS_REG:
        BASE['naive (predict mean)'] = scores(y, np.full(len(y), y.mean()))
        random_split = KFold(CFG.N_FOLDS, shuffle=True, random_state=CFG.SEED)
    else:
        major = np.bincount(y).argmax()
        BASE['naive (majority class)'] = scores(y, np.full(len(y), major))
        random_split = StratifiedKFold(CFG.N_FOLDS, shuffle=True, random_state=CFG.SEED)

    oof, oof_p, model, va = run_cv(random_split)
    BASE['GBM random K-fold'] = scores(y, oof, oof_p)
    if CFG.CV_GROUP and CFG.CV_GROUP in d.columns:
        oof, oof_p, model, va = run_cv(GroupKFold(CFG.N_FOLDS), d[CFG.CV_GROUP].astype(str).values)
        BASE[f'GBM group K-fold ({CFG.CV_GROUP})'] = scores(y, oof, oof_p)
    base_table = pd.DataFrame(BASE).T
    save_table(base_table, 'baseline_scores')
    display(base_table)

    va_s = np.random.RandomState(CFG.SEED).choice(va, min(4000, len(va)), replace=False)
    perm = permutation_importance(model, X.iloc[va_s], y[va_s], n_repeats=5, random_state=CFG.SEED,
                                  scoring='r2' if IS_REG else 'f1_macro')
    imp = pd.DataFrame({'mean': perm.importances_mean, 'std': perm.importances_std}, index=X.columns).sort_values('mean')
    save_table(imp, 'permutation_importance')

    fig, axes = plt.subplots(1, 3, figsize=(20, max(5.5, 0.3 * len(imp) + 1.5)))
    key = 'R2' if IS_REG else 'macro_F1'
    vals = base_table[key]
    axes[0].bar(range(len(vals)), vals.values, color=[MUTED] + PALETTE[:len(vals) - 1])
    axes[0].set_xticks(range(len(vals)))
    axes[0].set_xticklabels([s.replace(' (', '\n(') for s in vals.index], fontsize=8)
    bar_labels(axes[0], '{:.3f}')
    axes[0].set_title(f'{key}: naive vs GBM' + (' (random vs group gap = leakage)' if len(vals) > 2 else ''), loc='left')

    ax = axes[1]
    if IS_REG:
        ax.hexbin(y, oof, gridsize=40, cmap=CMAP_SEQ, mincnt=1, bins='log', linewidths=0)
        lims = [min(y.min(), oof.min()), max(y.max(), oof.max())]
        ax.plot(lims, lims, color=ACCENT, ls='--', lw=1.5)
        ax.set_xlabel(f'actual {T}')
        ax.set_ylabel('out-of-fold prediction')
        ax.set_title('Predicted vs actual (last CV scheme)', loc='left')
    else:
        cm = pd.crosstab(pd.Series(y, name='actual'), pd.Series(oof.astype(int), name='predicted'), normalize='index')
        keep = pd.Series(y).value_counts().index[:15].sort_values()
        cm = cm.reindex(index=keep, columns=keep, fill_value=0)
        names = [short(class_names[i], 12) for i in keep]
        sns.heatmap(cm, cmap=CMAP_SEQ, annot=len(keep) <= 12, fmt='.2f', vmin=0, vmax=1, ax=ax,
                    xticklabels=names, yticklabels=names)
        extra = f', top {len(keep)} of {len(class_names)} classes' if len(class_names) > len(keep) else ''
        ax.set_title(f'Row-normalised confusion matrix{extra}', loc='left')

    top = imp.tail(CFG.MAX_PLOT_COLS)
    axes[2].barh([short(i) for i in top.index], top['mean'], xerr=top['std'],
                 color=[ACCENT if c in CAT_COLS else PRIMARY for c in top.index], error_kw=dict(ecolor=MUTED, lw=1))
    axes[2].set_title('Permutation importance (held-out fold)', loc='left')
    suptitle(fig, 'Baseline signal check', 'HistGradientBoosting, no tuning, out-of-fold scores')
    save_fig(fig, 'baseline_signal')

#### Insights

> *Fill in after running.* How far does the untuned model beat the naive baseline? That gap is the realistic room for modelling. If random K-fold scores much higher than group K-fold, duplicated entities are leaking and every final model must use the group split.

> Do the permutation importances agree with the association ranking in section 9? Disagreement usually means interactions or redundancy between features.

---

# EDA Summary <a name="15"></a>

---

The key facts from every section are collected automatically into one markdown summary, displayed below and saved as `eda-output/eda_summary.md`. It is a starting point for the report's data description and preprocessing chapters, and should be edited into prose with the insights written above.

In [ ]:
G = globals()
lines = [f'# EDA Summary: {DATA_PATH.name}', '']
lines += ['## Dataset',
          f'- {len(df):,} rows x {profile.shape[0]} columns, target `{T}`, task **{TASK}**',
          f'- Roles: {len(NUM_COLS)} numeric, {len(CAT_COLS)} categorical, {len(TEXT_COLS)} text, '
          f'{len(ID_COLS)} id, {len(DT_COLS)} datetime, {len(CONST_COLS)} constant', '']

lines += ['## Data quality']
lines.append(f'- Columns with NaN: {len(miss)}' + (f' (worst: `{miss.index[0]}` {miss.iloc[0]:.1f}%)' if len(miss) else ''))
if len(disguised):
    lines.append(f'- Disguised missing values: ' + ', '.join(f'`{r.column}`={r.value} ({r.count:,})' for r in disguised.head(5).itertuples()))
for k, v in dup_summary.items():
    if v:
        lines.append(f'- Duplicates, {k}: {v:,} ({v / len(df):.1%})')
if CONST_COLS or len(quasi):
    lines.append(f'- Constant / quasi-constant: {CONST_COLS + quasi.index.tolist()}')
high_card = [c for c in CAT_COLS if df[c].nunique() > 50]
if high_card:
    lines.append(f'- High-cardinality categoricals (>50 levels): {high_card}')
if not num_stats.empty:
    skewed = num_stats.index[num_stats['skew'].abs() > 1].tolist()
    lines.append(f'- Heavily skewed numerics (|skew| > 1): {skewed}')
lines.append('')

if HAS_TARGET:
    lines += ['## Target']
    if IS_REG:
        lines.append(f'- mean {target_stats["mean"]:.3f}, median {target_stats["median"]:.3f}, std {target_stats["std"]:.3f}, skew {target_stats["skew"]:.2f}')
        lines.append(f'- zeros {target_stats["zero_share_%"]:.1f}%, most frequent value {target_stats["mode"]} covers {target_stats["mode_share_%"]:.1f}%')
    else:
        lines.append(f'- {len(target_stats)} classes, imbalance ratio {target_stats["count"].max() / target_stats["count"].min():.1f}x')
    lines.append('')

if not ASSOC.empty:
    lines += ['## Strongest feature-target associations (by mutual information)']
    for f, r in ASSOC.head(8).iterrows():
        lines.append(f'- `{f}` ({r["type"]}): MI {r["mutual_info"]:.3f}, {r["test"]} stat {r["statistic"]:.3f}, effect {r["effect"]:.3f}, p = {r["p_value"]:.2g}')
    lines.append('')

lines += ['## Redundancy']
hp = G.get('high_pairs', pd.DataFrame())
lines.append(f'- Pairs with |Spearman| >= {CFG.HIGH_CORR}: {len(hp)}' + (': ' + ', '.join(f'{a}~{b} ({r:+.2f})' for a, b, r in hp[['feature_a', 'feature_b', 'spearman']].head(5).values) if len(hp) else ''))
vif_s = G.get('vif', pd.Series(dtype=float))
if len(vif_s):
    lines.append(f'- VIF > 10: {vif_s[vif_s > 10].index.tolist()}, max VIF {vif_s.max():.1f} (`{vif_s.idxmax()}`)')
lines.append('')

if not seg_table.empty:
    lines += [f'## {CFG.GROUP_COL} stability over {CFG.TIME_COL}']
    for q in ['evergreen', 'hit-driven', 'stable niche', 'speculative']:
        members = seg_table.index[seg_table['quadrant'] == q].tolist()
        lines.append(f'- {q}: {[short(m, 25) for m in members[:8]]}')
    for t in ['rising', 'falling']:
        lines.append(f'- significantly {t}: {[short(m, 25) for m in seg_table.index[seg_table["trend"] == t][:8]]}')
    lines.append('')

if ADV_AUC is not None:
    lines += ['## Train vs test', f'- Adversarial AUC {ADV_AUC:.3f}; PSI > 0.25: {drift.index[drift["psi"] > 0.25].tolist()}', '']

if BASE:
    lines += ['## Baseline (untuned GBM, out-of-fold)']
    for k, v in BASE.items():
        lines.append(f'- {k}: ' + ', '.join(f'{m} {x:.3f}' for m, x in v.items()))
    lines.append('')

lines += ['## Exported files', f'- Figures ({len(SAVED_FIGS)}): `{CFG.FIG_DIR}`', f'- Tables: `{CFG.TABLE_DIR}`']
summary_md = '\n'.join(lines)
(CFG.OUTPUT_DIR / 'eda_summary.md').write_text(summary_md, encoding='utf-8')
display(Markdown(summary_md))

## Decisions File for the Pipeline

The findings that should change how a model is built are written as machine-readable JSON to `eda-output/eda_decisions.json`. The tabular pipeline notebook reads it for every setting left on `'auto'`, so each modelling choice can be traced back to the EDA evidence below.

| Decision | EDA evidence |
|---|---|
| Columns to drop | constant and quasi-constant columns (section 5) |
| Sentinels to treat as missing | placeholder values outside the column's normal range (section 5) |
| Group split column | repeated identifiers and the random vs group K-fold gap (sections 5, 14) |
| Time split | a time column plus strong train/test drift (sections 11, 13) |
| Target transform, metric, class weights | target skew, zeros, and imbalance (section 6) |
| Weak features | no mutual information and no significant association (section 9) |

In [ ]:
def out_of_range_sentinels():
    found = {}
    for c in [c for c in df.columns if c not in (T, '__target_label__', '__period__', '__metric__') and not c.startswith('__len_')]:
        s = df[c]
        if pd.api.types.is_numeric_dtype(s) and not pd.api.types.is_bool_dtype(s):
            vals = []
            for v in SENTINEL_NUM:
                if (s == v).any():
                    others = s[(s != v) & s.notna()]
                    if len(others) and (v < others.min() or v > others.max()):
                        vals.append(v)
            if vals:
                found[c] = vals
        elif not pd.api.types.is_datetime64_any_dtype(s):
            st = s.dropna().astype(str).str.strip().str.lower()
            vals = sorted(set(st[st.isin(SENTINEL_STR)]))
            if vals:
                found[c] = vals
    return found

repeated = {c: dup_summary.get(f'repeated {c}', 0) for c in ID_COLS}
# an explicit CV_GROUP wins; otherwise the identifier repeated in the most rows
cv_group = CFG.CV_GROUP if CFG.CV_GROUP and CFG.CV_GROUP in df.columns else \
    next((c for c, n in sorted(repeated.items(), key=lambda kv: -kv[1]) if n / len(df) > 0.01), None)
group_gap = None
if BASE:
    key = 'R2' if IS_REG else 'macro_F1'
    gk = [k for k in BASE if k.startswith('GBM group')]
    if gk and 'GBM random K-fold' in BASE:
        r, g = BASE['GBM random K-fold'][key], BASE[gk[0]][key]
        group_gap = float((r - g) / abs(r)) if r else None
time_col = CFG.TIME_COL if CFG.TIME_COL and CFG.TIME_COL in df.columns else (DT_COLS[0] if DT_COLS else None)
drifted = ADV_AUC is not None and ADV_AUC > 0.7
scheme = 'time' if (time_col and drifted) else 'group' if cv_group else ('stratified' if IS_CLF else 'kfold')
weak = []
if not ASSOC.empty:
    weak = ASSOC.index[(ASSOC['mutual_info'] < 0.005) & (~ASSOC['significant'])].tolist()
if HAS_TARGET and IS_REG:
    yv = df[T].dropna()
    transform = 'log1p' if (yv.min() >= 0 and yv.skew() > 1) else 'none'
    metric = 'rmse'
    class_weights = False
elif HAS_TARGET:
    vc = df['__target_label__'].value_counts()
    transform = 'none'
    metric = 'roc_auc' if len(vc) == 2 else 'f1_macro'
    class_weights = bool(vc.max() / vc.min() > 3)
else:
    transform, metric, class_weights = 'none', None, False

hp = G.get('high_pairs')
pair_list = ([[a, b, round(float(r), 3)] for a, b, r in hp[['feature_a', 'feature_b', 'spearman']].values]
             if isinstance(hp, pd.DataFrame) and {'feature_a', 'feature_b', 'spearman'} <= set(hp.columns) else [])
decisions = {
    'source': str(DATA_PATH), 'target': T, 'task': TASK,
    'id_cols': ID_COLS, 'num_cols': NUM_COLS, 'cat_cols': CAT_COLS, 'text_cols': TEXT_COLS, 'datetime_cols': DT_COLS,
    'drop_cols': sorted({c for c in CONST_COLS + quasi.index.tolist() if c != T}),
    'sentinels': out_of_range_sentinels(),
    'high_cardinality': {c: int(df[c].nunique()) for c in high_card},
    'dedup': bool(dup_summary.get('identical rows', 0) > 0),
    'cv_group': cv_group, 'group_vs_random_gap': group_gap, 'conflicting_targets': int(dup_summary.get('same features, different target', 0)),
    'time_col': time_col, 'adversarial_auc': ADV_AUC, 'cv_scheme': scheme,
    'target_transform': transform, 'metric': metric, 'class_weights': class_weights,
    'weak_features': weak, 'top_features': ASSOC.index[:10].tolist() if not ASSOC.empty else [],
    'high_corr_pairs': pair_list,
}
(CFG.OUTPUT_DIR / 'eda_decisions.json').write_text(json.dumps(decisions, indent=2, default=str), encoding='utf-8')
print(json.dumps({k: v for k, v in decisions.items() if k not in ('num_cols', 'cat_cols')}, indent=2, default=str))

The final cell lists every exported artefact so they can be located quickly when writing the report or building slides.

In [ ]:
print(f'Figures in {CFG.FIG_DIR.resolve()}:')
for f in SAVED_FIGS:
    print('  ', f)
print(f'\nTables in {CFG.TABLE_DIR.resolve()}:')
for f in sorted(CFG.TABLE_DIR.glob('*.csv')):
    print('  ', f.name)

#### Insights

> *Fill in after running.* Summarise the three to five findings that will shape preprocessing, feature engineering, validation, and the business narrative, each linked to the figure that supports it.

---

# Part 2: Modelling Pipeline

---

*Everything above is the EDA; everything below is the pipeline. The pipeline reloads its own libraries and settings, inherits the data files resolved in Part 1, and reads `eda-output/eda_decisions.json` through `EDA_DIR`, so its target, task, columns to drop, validation scheme, metric, and transforms are the ones the EDA justified. Settings that are not on `'auto'` override the EDA, and the decision table records the source of every choice.*

*To use the pipeline settings, edit the `Settings` cell in the Pipeline Initialization section (models, tuning, ensembling). Data paths only need to be set once, in Part 1.*

## How to Configure

*Everything is controlled from the `Settings` class in section 2. Nothing else needs editing.*

| Setting | Purpose | Example |
|---|---|---|
| `KAGGLE_PATH` / `COLAB_PATH` / `LOCAL_PATH` | Training file per environment (auto-detected) | `'/kaggle/input/comp'` (folder) or a file path |
| `TEST_*_PATH` | Optional unlabeled test file, used to write `submission.csv` | `'.../test.csv'` or `None` |
| `SUBMISSION_FORMAT` | `'label'`, `'proba'` (positive-class probability, or one column per class), or `'auto'` (probabilities for ROC AUC, PR AUC, and log loss) | `'auto'` |
| `EDA_DIR` | Folder with `eda_decisions.json` from the tabular EDA; every `'auto'` setting below reads it | `'eda-output'` |
| `TARGET`, `TASK` | Target column (a list trains one model set per target), and `'auto'`, `'regression'`, `'classification'`, or `'ordinal'` | `'price'`, `['y1', 'y2']`, `'ordinal'` |
| `ORDINAL_ORDER`, `MULTILABEL_SEP` | Class order for an ordinal string target; separator when one column holds several labels | `['low', 'mid', 'high']`, `'|'` |
| `TUNE_DECISION` | Tune the label rule on out-of-fold predictions: binary threshold, multiclass class scales, ordinal cut-points | `True` |
| `DROP_COLS`, `SENTINELS`, `DROP_WEAK` | Columns to drop, placeholder values to blank, weak features (`'auto'` from EDA) | `['const_col']`, `{'key': [-1]}` |
| `ID_COLS`, `DROP_COLS` | Columns never used as features | `['id']` |
| `CV_SCHEME`, `CV_GROUP`, `TIME_COL` | Validation design: `'auto'`, `'kfold'`, `'stratified'`, `'group'`, `'time'` | `'group'`, `'user_id'` |
| `METRIC` | Selection metric, or `'auto'` (RMSE, ROC AUC for binary, macro F1 for multiclass, QWK for ordinal, F1 per label for multi-label) | `'mae'`, `'f1_macro'`, `'qwk'` |
| `MODELS` | Any subset of `['lightgbm', 'xgboost', 'catboost']`, or `[]` for baseline only | `['catboost']` |
| `TUNE`, `TUNE_MODEL`, `N_TRIALS` | Optional Optuna search for one model | `True`, `'lightgbm'`, `40` |

*A model whose library is not installed is skipped with a warning, so the notebook never breaks on a missing dependency. Competition-specific features go into the `custom_features` function in section 5.*

## Metric

***RMSE** is the default regression metric. It penalises large errors quadratically and is reported together with MAE and R-squared, which explains the share of target variance captured.*

$$\text{RMSE} = \sqrt{\frac{1}{n}\sum_{i=1}^{n}(y_i - \hat{y}_i)^2} \qquad R^2 = 1 - \frac{\sum_i (y_i - \hat{y}_i)^2}{\sum_i (y_i - \bar{y})^2}$$

***ROC AUC** is the default for binary classification because it is threshold-free and robust to imbalance. **Macro F1** is the default for multiclass because it weights every class equally.*

$$\text{Macro F1} = \frac{1}{K}\sum_{k=1}^{K} \frac{2 \cdot P_k \cdot R_k}{P_k + R_k}$$

---

# Pipeline Initialization <a name="16"></a>

---

## Environment Setup

The pipeline runs on CPU. A GPU is only used by the boosting libraries when `USE_GPU = True`. Runtime with all three boosting models and no tuning is a few minutes on a Kaggle CPU session.

In [ ]:
!nvidia-smi

The following cell installs all libraries used in this notebook.

In [ ]:
%pip install -q numpy pandas scipy scikit-learn matplotlib seaborn joblib pyarrow openpyxl lightgbm xgboost catboost optuna

## Import Libraries

The boosting libraries and Optuna are imported defensively. A failed import marks the library as unavailable instead of stopping the notebook, which keeps the linear baseline guaranteed to run.

In [ ]:
import os
import re
import json
import time
import random
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import seaborn as sns
from scipy.optimize import minimize
import sklearn
from sklearn.calibration import calibration_curve
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, LogisticRegression, Ridge
from sklearn.metrics import (accuracy_score, average_precision_score, balanced_accuracy_score, cohen_kappa_score,
                             confusion_matrix, f1_score, log_loss, mean_absolute_error,
                             mean_squared_error, median_absolute_error, precision_recall_curve,
                             precision_recall_fscore_support, r2_score, roc_auc_score, roc_curve)
from sklearn.model_selection import (GroupKFold, GroupShuffleSplit, KFold, StratifiedGroupKFold,
                                     StratifiedKFold, TimeSeriesSplit, train_test_split)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, LabelEncoder, OneHotEncoder, PowerTransformer, StandardScaler
from IPython.display import Markdown, display

AVAILABLE = {}
try:
    from sklearn.preprocessing import TargetEncoder
    AVAILABLE['target_encoder'] = True
except ImportError:
    AVAILABLE['target_encoder'] = False
try:
    import lightgbm as lgb
    AVAILABLE['lightgbm'] = lgb.__version__
except Exception as e:
    AVAILABLE['lightgbm'] = False
    print(f'[warn] lightgbm unavailable: {str(e).strip().splitlines()[0][:120]}')
try:
    import xgboost as xgb
    AVAILABLE['xgboost'] = xgb.__version__
except Exception as e:
    AVAILABLE['xgboost'] = False
    print(f'[warn] xgboost unavailable: {str(e).strip().splitlines()[0][:120]}')
try:
    import catboost
    from catboost import CatBoostClassifier, CatBoostRegressor, Pool
    AVAILABLE['catboost'] = catboost.__version__
except Exception as e:
    AVAILABLE['catboost'] = False
    print(f'[warn] catboost unavailable: {str(e).strip().splitlines()[0][:120]}')
try:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    AVAILABLE['optuna'] = optuna.__version__
except Exception:
    AVAILABLE['optuna'] = False

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)
print(f'scikit-learn {sklearn.__version__} | available: {AVAILABLE}')

## Seed Everything

In [ ]:
def seed_everything(seed: int = 42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)

seed_everything(42)

## Settings

All paths, column roles, validation choices, model choices, and hyperparameters are centralised here. Switching between Kaggle, Colab, and local execution only requires editing the path lines in block 1. Any option set to `None`, `[]`, or `'auto'` is either skipped or inferred from the data.

In [ ]:
class Settings:
    SEED       = 42
    _ON_KAGGLE = Path('/kaggle/input').exists()
    _ON_COLAB  = Path('/content').exists() and not _ON_KAGGLE

    # 1) Data location: a file or a dataset folder (train / test / sample_submission are found by name)
    KAGGLE_PATH      = '/kaggle/input/<dataset-slug>'
    COLAB_PATH       = '/content/drive/MyDrive/<folder>'
    LOCAL_PATH       = 'data'
    TEST_KAGGLE_PATH = None
    TEST_COLAB_PATH  = None
    TEST_LOCAL_PATH  = None
    READ_KWARGS      = {}

    # 2) Columns
    EDA_DIR       = 'eda-output'
    TARGET        = 'auto'
    TASK          = 'auto'
    ORDINAL_ORDER = None
    MULTILABEL_SEP = None
    ID_COLS       = 'auto'
    DROP_COLS     = 'auto'
    SENTINELS     = 'auto'
    DATE_COLS     = []
    NUM_COLS      = 'auto'
    CAT_COLS      = 'auto'
    TEXT_COLS     = 'auto'
    SUBMISSION_ID = None
    SUBMISSION_FORMAT = 'auto'

    # 3) Validation
    CV_SCHEME    = 'auto'
    CV_GROUP     = 'auto'
    TIME_COL     = 'auto'
    N_FOLDS      = 5
    HOLDOUT_SIZE = 0.2
    DEDUP        = 'auto'
    METRIC       = 'auto'

    # 4) Preprocessing and features
    TARGET_TRANSFORM   = 'auto'
    CLASS_WEIGHT       = 'auto'
    TUNE_DECISION      = True
    DROP_WEAK          = 'auto'
    DISCRETE_MAX       = 15
    MAX_OHE_LEVELS     = 30
    GBM_MAX_CAT_LEVELS = 200
    ADD_FREQ_ENCODING  = True
    ADD_MISSING_FLAGS  = True
    TEXT_SVD_DIM       = 16

    # 5) Models
    BASELINE       = 'linear'
    MODELS         = ['lightgbm', 'xgboost', 'catboost']
    N_ESTIMATORS   = 3000
    LEARNING_RATE  = 0.05
    EARLY_STOPPING = 100
    USE_GPU        = False
    MODEL_PARAMS   = {'lightgbm': {}, 'xgboost': {}, 'catboost': {}}
    TUNE           = False
    TUNE_MODEL     = 'lightgbm'
    N_TRIALS       = 30
    TUNE_TIMEOUT   = 900
    TUNE_FOLDS     = 3
    ENSEMBLE       = True

    # 6) Outputs
    DATA_PATH    = KAGGLE_PATH if _ON_KAGGLE else COLAB_PATH if _ON_COLAB else LOCAL_PATH
    TEST_PATH    = TEST_KAGGLE_PATH if _ON_KAGGLE else TEST_COLAB_PATH if _ON_COLAB else TEST_LOCAL_PATH
    OUTPUT_DIR   = Path('/kaggle/working/pipeline-output') if _ON_KAGGLE else Path('pipeline-output')
    BASELINE_DIR = OUTPUT_DIR / 'baseline'
    FINAL_DIR    = OUTPUT_DIR / 'final'

CFG = Settings()
for d in (CFG.BASELINE_DIR / 'figures', CFG.FINAL_DIR / 'figures', CFG.FINAL_DIR / 'models', CFG.FINAL_DIR / 'oof'):
    d.mkdir(parents=True, exist_ok=True)
env = 'Kaggle' if CFG._ON_KAGGLE else 'Colab' if CFG._ON_COLAB else 'Local'
print(f'Environment : {env}')
print(f'Data path   : {CFG.DATA_PATH}')
print(f'Test path   : {CFG.TEST_PATH}')
print(f'Output dir  : {CFG.OUTPUT_DIR.resolve()}')
# reuse the data files resolved in Part 1, so the paths are edited in one place only
CFG.DATA_PATH = str(DATA_PATH)
CFG.TEST_PATH = str(TEST_PATH) if globals().get('TEST_PATH') else None
print(f'Pipeline data inherited from Part 1: {CFG.DATA_PATH} | test: {CFG.TEST_PATH}')

## Load Dataset

The path is resolved first. On Kaggle a missing path triggers a listing of `/kaggle/input` and a fallback to the largest tabular file. The reader is chosen from the file extension. Columns are cleaned in section 4, once the EDA decisions are known.

In [ ]:
TABULAR_EXT = ('.csv', '.tsv', '.txt', '.parquet', '.pq', '.feather', '.xlsx', '.xls', '.json', '.zip', '.gz')

def load_table(path, **kwargs):
    suffixes = [s.lower() for s in path.suffixes]
    if '.parquet' in suffixes or '.pq' in suffixes:
        return pd.read_parquet(path, **kwargs)
    if '.feather' in suffixes:
        return pd.read_feather(path, **kwargs)
    if '.xlsx' in suffixes or '.xls' in suffixes:
        return pd.read_excel(path, **kwargs)
    if '.json' in suffixes:
        return pd.read_json(path, **kwargs)
    if '.tsv' in suffixes:
        kwargs.setdefault('sep', '\t')
    return pd.read_csv(path, low_memory=False, **kwargs)

def file_role(f):
    """Classify a data file by its name: 'train', 'test', 'sample' (submission template), or None."""
    stem = f.name.lower().split('.')[0]
    if 'sample' in stem or 'submission' in stem:
        return 'sample'
    if stem.startswith('test'):
        return 'test'
    if stem.startswith('train'):
        return 'train'
    return None

SKIP_DIRS = {'eda-output', 'pipeline-output', 'catboost_info', '.ipynb_checkpoints', '.git', '.venv', 'site-packages'}
LOCAL_DATA_DIRS = ('data', 'input', 'dataset')

def list_tables(root, max_depth=6):
    # os.walk + extension check by name: no per-file disk call, which is slow on the /kaggle/input network mount
    root, out = Path(root), []
    for d, dirs, names in os.walk(root):
        depth = len(Path(d).relative_to(root).parts)
        dirs[:] = [x for x in dirs if x not in SKIP_DIRS and not x.startswith('.')] if depth < max_depth else []
        out += [Path(d) / n for n in names if Path(n).suffix.lower() in TABULAR_EXT]
    return out

def discover_files(data_path, test_path):
    """Resolve the train, test, and sample-submission files from a file, a dataset folder, or the input root."""
    p = Path(data_path) if data_path else None
    explicit_test = Path(test_path) if test_path and Path(test_path).exists() else None
    if test_path and explicit_test is None:
        print(f'[warn] test path {test_path} not found; searching the dataset folder instead')
    if p is not None and p.is_file():
        train, files = p, [f for f in p.parent.iterdir() if f.is_file() and f.suffix.lower() in TABULAR_EXT]
    else:
        if p is not None and p.is_dir():
            roots = [p]
        elif CFG._ON_KAGGLE:
            roots = [Path('/kaggle/input')]
        elif CFG._ON_COLAB:
            roots = [Path('/content')]
        else:
            # local: data/, input/, dataset/ next to the notebook first, then the notebook folder itself
            roots = [Path(d) for d in LOCAL_DATA_DIRS] + [Path('.')]
        if p is not None and not p.is_dir():
            print(f'[warn] {p} not found; searching {[str(r) for r in roots if r.exists()]}')
        train, files = None, []
        for r in (r for r in roots if r.exists()):
            files = list_tables(r)
            named = [f for f in files if file_role(f) == 'train']
            others = [f for f in files if file_role(f) is None]
            train = max(named or others, key=lambda f: f.stat().st_size, default=None)
            if train is not None:
                break
    if train is None:
        return None, explicit_test, None

    def pick(role):
        near = [f for f in files if file_role(f) == role and f.parent == train.parent]
        cands = near or [f for f in files if file_role(f) == role]
        exact = [f for f in cands if f.name.lower().split('.')[0] == role]
        return min(exact or cands, key=lambda f: len(f.name), default=None)

    test = explicit_test or pick('test')
    sample = pick('sample')
    for role, f in (('train', train), ('test', test), ('sample submission', sample)):
        print(f'{role:<18}: {f if f else "(none found)"}')
    return train, test, sample

DATA_PATH, TEST_PATH, SAMPLE_PATH = discover_files(CFG.DATA_PATH, CFG.TEST_PATH)
assert DATA_PATH is not None, f'{CFG.DATA_PATH} not found and no train file was discovered. Edit the path lines in Settings.'
if TEST_PATH is None:
    print('[warn] No test file configured or found in the dataset folder: submission.csv will not be written.')
df_raw = load_table(DATA_PATH, **CFG.READ_KWARGS)
test_raw = load_table(TEST_PATH, **CFG.READ_KWARGS) if TEST_PATH else None
SAMPLE = load_table(SAMPLE_PATH) if SAMPLE_PATH else None
print(f'Train : {df_raw.shape[0]:,} rows x {df_raw.shape[1]} cols  <- {DATA_PATH}')
if test_raw is not None:
    print(f'Test  : {test_raw.shape[0]:,} rows x {test_raw.shape[1]} cols  <- {TEST_PATH}')

---

# Toolkit <a name="17"></a>

---

The same visual identity as the EDA template is applied to every chart: a deep navy primary, a coral accent, and eight categorical colours. `save_fig` writes each figure into the `figures/` folder of either the baseline or the final output directory with a running number.

In [ ]:
PRIMARY   = '#3D5A80'
ACCENT    = '#EE6C4D'
SOFT      = '#98C1D9'
DARK      = '#1B263B'
MUTED     = '#8D99AE'
PALETTE   = ['#3D5A80', '#EE6C4D', '#2A9D8F', '#E9C46A', '#9B5DE5',
             '#F15BB5', '#00BBF9', '#8AB17D', '#E76F51', '#264653']
CMAP_SEQ  = LinearSegmentedColormap.from_list('seq', ['#F7F9FC', '#98C1D9', '#3D5A80', '#1B263B'])
CMAP_DIV  = LinearSegmentedColormap.from_list('div', ['#EE6C4D', '#FBE3DC', '#F7F7F7', '#D6E4F0', '#3D5A80'])

sns.set_theme(style='whitegrid', palette=PALETTE)
mpl.rcParams.update({
    'figure.dpi': 100, 'savefig.dpi': 200, 'savefig.bbox': 'tight',
    'figure.facecolor': 'white', 'axes.facecolor': 'white',
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.edgecolor': '#C9D1DB', 'axes.labelcolor': DARK, 'axes.titleweight': 'bold',
    'axes.titlesize': 12, 'axes.labelsize': 10, 'axes.titlecolor': DARK,
    'xtick.color': '#4A5568', 'ytick.color': '#4A5568',
    'grid.color': '#E2E8F0', 'grid.linewidth': 0.8, 'legend.frameon': False, 'font.size': 10,
})

FIG_COUNTER = {'baseline': 0, 'final': 0}

def slug(text):
    return re.sub(r'[^a-z0-9]+', '_', str(text).lower()).strip('_')[:60]

def save_fig(fig, name, stage='final'):
    FIG_COUNTER[stage] += 1
    root = CFG.BASELINE_DIR if stage == 'baseline' else CFG.FINAL_DIR
    path = root / 'figures' / f'{FIG_COUNTER[stage]:02d}_{slug(name)}.png'
    fig.savefig(path, facecolor='white')
    plt.show()
    plt.close(fig)

def suptitle(fig, title, subtitle=None, layout=True):
    h = fig.get_figheight()
    if layout:
        fig.tight_layout(rect=(0, 0, 1, 1 - (0.85 if subtitle else 0.55) / h))
    fig.text(0.01, 1 - 0.1 / h, title, ha='left', va='top', fontsize=15, fontweight='bold', color=DARK)
    if subtitle:
        fig.text(0.01, 1 - 0.42 / h, subtitle, ha='left', va='top', fontsize=10, color=MUTED)

def bar_labels(ax, fmt='{:,.0f}', pad=3, size=8):
    for container in ax.containers:
        if hasattr(container, 'datavalues'):
            ax.bar_label(container, labels=[fmt.format(v) for v in container.datavalues],
                         padding=pad, fontsize=size, color='#4A5568')

def short(value, k=28):
    value = str(value)
    return value if len(value) <= k else value[:k - 3] + '...'

def note(msg):
    display(Markdown(f'> **Note:** {msg}'))

def to_json(obj, path):
    path.write_text(json.dumps(obj, indent=2, default=lambda o: o.tolist() if hasattr(o, 'tolist') else str(o)), encoding='utf-8')

## Metrics

Every model is scored with the same functions. Regression predictions are values on the original target scale. Classification predictions are probability matrices with one column per class, and labels are derived from them with `to_label`, which applies a tuned threshold for binary tasks when one has been chosen.

In [ ]:
GREATER_IS_BETTER = {'rmse': False, 'mae': False, 'medae': False, 'rmsle': False, 'r2': True,
                     'roc_auc': True, 'pr_auc': True, 'f1_macro': True, 'f1_weighted': True,
                     'f1': True, 'accuracy': True, 'balanced_accuracy': True, 'logloss': False,
                     'qwk': True, 'mae_class': False}
THRESHOLD, CLASS_SCALE, CUTS = [0.5], [None], [None]
IS_ORD = False

def to_label(proba):
    """Labels from probabilities: tuned threshold (binary) or tuned class scales (multiclass)."""
    if proba.shape[1] == 2:
        return (proba[:, 1] >= THRESHOLD[0]).astype(int)
    if CLASS_SCALE[0] is not None:
        return (proba * CLASS_SCALE[0]).argmax(axis=1)
    return proba.argmax(axis=1)

def to_ordinal(pred):
    """Ordinal class index from a continuous score, using the tuned cut-points."""
    cuts = CUTS[0] if CUTS[0] is not None else np.arange(N_CLASSES - 1) + 0.5
    return np.digitize(pred, cuts)
def score_all(y_true, pred):
    if IS_ORD:
        yt, lab = np.asarray(y_true).astype(int), to_ordinal(pred)
        return {'qwk': float(cohen_kappa_score(yt, lab, weights='quadratic')), 'accuracy': float(accuracy_score(yt, lab)),
                'f1_macro': float(f1_score(yt, lab, average='macro')), 'mae_class': float(np.mean(np.abs(yt - lab))),
                'rmse': float(np.sqrt(mean_squared_error(yt, pred)))}
    if IS_REG:
        out = {'rmse': float(np.sqrt(mean_squared_error(y_true, pred))), 'mae': mean_absolute_error(y_true, pred),
               'medae': median_absolute_error(y_true, pred), 'r2': r2_score(y_true, pred)}
        if y_true.min() >= 0:
            out['rmsle'] = float(np.sqrt(mean_squared_error(np.log1p(y_true), np.log1p(np.clip(pred, 0, None)))))
        return out
    lab = to_label(pred)
    out = {'accuracy': accuracy_score(y_true, lab), 'balanced_accuracy': balanced_accuracy_score(y_true, lab),
           'f1_macro': f1_score(y_true, lab, average='macro'), 'f1_weighted': f1_score(y_true, lab, average='weighted')}
    p = np.clip(pred, 1e-7, 1)
    out['logloss'] = log_loss(y_true, p / p.sum(axis=1, keepdims=True), labels=list(range(N_CLASSES)))
    try:
        if IS_BINARY:
            out['roc_auc'] = roc_auc_score(y_true, pred[:, 1])
            out['pr_auc'] = average_precision_score(y_true, pred[:, 1])
            out['f1'] = f1_score(y_true, lab)
        else:
            out['roc_auc'] = roc_auc_score(y_true, p / p.sum(axis=1, keepdims=True), multi_class='ovr', labels=list(range(N_CLASSES)))
    except ValueError:
        out['roc_auc'] = np.nan
    return {k: float(v) for k, v in out.items()}

def primary(y_true, pred):
    return score_all(y_true, pred)[METRIC]

def better(a, b):
    return a > b if GREATER_IS_BETTER[METRIC] else a < b

---

# Data Preparation <a name="18"></a>

---

Preparation fixes everything that must be decided before features are built: column roles, the task type, the target encoding or transform, duplicate removal, and the holdout split. Every decision is printed so it can be quoted in the report's analytical steps chapter.

## EDA-Driven Decisions

`eda-output/eda_decisions.json`, written by the tabular EDA notebook, is loaded when it exists. Every setting on `'auto'` takes its value from it, in priority order: an explicit value in `Settings`, then the EDA file, then a value computed from the data. Each decision is recorded with its source and the EDA evidence behind it, and the table is printed at the end of this section and saved to `pipeline-output/final/decisions.csv`.

In [ ]:
eda_path = Path(CFG.EDA_DIR) / 'eda_decisions.json'
EDA = json.loads(eda_path.read_text()) if eda_path.exists() else {}
print(f'EDA decisions: {eda_path.resolve()} ({"found" if EDA else "not found, computing from data"})')
if EDA and Path(EDA.get('source', '')).name != DATA_PATH.name:
    print(f'[warn] EDA was run on {EDA.get("source")}, not {DATA_PATH.name}')
DECISIONS = []

def decide(name, user_value, eda_key, compute, why=''):
    """Resolve a setting: explicit user value > EDA decision > computed fallback. `why` may be a function of the value."""
    if not (isinstance(user_value, str) and user_value == 'auto'):
        value, source = user_value, 'user'
    elif eda_key and EDA.get(eda_key) is not None:
        value, source = EDA[eda_key], 'eda'
    else:
        value, source = compute(), 'computed'
    why = why(value) if callable(why) else why
    DECISIONS.append({'setting': name, 'value': str(value)[:80], 'source': source, 'why': why})
    return value

def guess_target(train, test):
    """Infer the target: the sample-submission target column, else the one train column missing from the test file."""
    if SAMPLE is not None and SAMPLE.shape[1] == 2 and SAMPLE.columns[1] in train.columns:
        return SAMPLE.columns[1]
    if test is not None:
        only_train = [c for c in train.columns if c not in test.columns]
        if len(only_train) == 1:
            return only_train[0]
    return None

T = decide('TARGET', CFG.TARGET, 'target', lambda: guess_target(df_raw, test_raw) or df_raw.columns[-1],
           'target column analysed in the EDA, else sample_submission or the column missing from test')
TARGETS = list(T) if isinstance(T, (list, tuple)) else [T]
if len(TARGETS) == 1 and CFG.TARGET == 'auto' and SAMPLE is not None and SAMPLE.shape[1] > 2 \
        and all(c in df_raw.columns for c in SAMPLE.columns[1:]):
    TARGETS = list(SAMPLE.columns[1:])
    DECISIONS.append({'setting': 'TARGETS', 'value': str(TARGETS)[:80], 'source': 'computed',
                      'why': 'sample_submission lists several target columns'})
T = TARGETS[0]
missing_targets = [t for t in TARGETS if t not in df_raw.columns]
assert not missing_targets, f'TARGET {missing_targets} not found. Columns: {list(df_raw.columns)}'
ID_NAME = re.compile(r'(^id$|_id$|^id_|^key$|_key$)', re.I)
ID_LIST = [c for c in decide('ID_COLS', CFG.ID_COLS, 'id_cols', lambda: [c for c in df_raw.columns if c not in TARGETS and ID_NAME.search(str(c))],
                             'identifier columns (EDA, else recognised by name)') if c in df_raw.columns]
CV_GROUP = decide('CV_GROUP', CFG.CV_GROUP, 'cv_group', lambda: None,
                  lambda v: 'identifier repeated across rows, so copies must share a fold' if v else 'no repeated identifier found')
CV_GROUP = CV_GROUP if CV_GROUP in df_raw.columns else None
TIME_COL = decide('TIME_COL', CFG.TIME_COL, 'time_col', lambda: None,
                  lambda v: 'time column used when a time split is chosen' if v else 'no time column')
TIME_COL = TIME_COL if TIME_COL in df_raw.columns else None
keep = {*TARGETS, CV_GROUP, TIME_COL, *ID_LIST}
DROP = [c for c in decide('DROP_COLS', CFG.DROP_COLS, 'drop_cols', lambda: [], 'constant or quasi-constant (one value >= 99%)') if c in df_raw.columns and c not in keep]
SENTINELS = decide('SENTINELS', CFG.SENTINELS, 'sentinels', lambda: {}, 'placeholder values outside the normal range -> missing')
SENTINELS = SENTINELS if isinstance(SENTINELS, dict) else {}

DATE_PATTERN = r'^(\d{4}[-/.]\d{1,2}[-/.]\d{1,2}|\d{1,2}[-/.]\d{1,2}[-/.]\d{2,4})([ T]\d{1,2}:\d{2}(:\d{2})?.*)?$'

def looks_like_date(s):
    """True when at least 90% of a sampled string column matches a date or datetime pattern."""
    if pd.api.types.is_numeric_dtype(s) or pd.api.types.is_datetime64_any_dtype(s) or pd.api.types.is_bool_dtype(s):
        return False
    st = s.dropna().astype(str).str.strip()
    if st.empty:
        return False
    st = st.sample(min(len(st), 500), random_state=CFG.SEED)
    return st.str.match(DATE_PATTERN).mean() > 0.9

def parse_dates(s):
    year_first = s.dropna().astype(str).str.strip().str.match(r'^\d{4}').mean() > 0.5
    return pd.to_datetime(s, errors='coerce', format='mixed', dayfirst=not year_first)

def prepare(frame):
    frame = frame.drop(columns=[c for c in DROP if c in frame.columns]).copy()
    for c, vals in SENTINELS.items():
        if c in frame.columns and c not in TARGETS:
            s = frame[c]
            hit = s.isin(vals) if pd.api.types.is_numeric_dtype(s) else s.astype(str).str.strip().str.lower().isin([str(v) for v in vals])
            frame[c] = s.where(~hit)
    for c in frame.columns:
        if c not in TARGETS and (c in CFG.DATE_COLS or c in EDA.get('datetime_cols', []) or looks_like_date(frame[c])):
            frame[c] = parse_dates(frame[c])
    return frame

df = prepare(df_raw)
test_df = prepare(test_raw) if test_raw is not None else None

# multi-label: one column holding several labels becomes one binary target per label
ML_SOURCE, ML_LABELS = None, []
if CFG.MULTILABEL_SEP:
    ML_SOURCE = T
    tags = df[T].fillna('').astype(str).map(lambda v: {x.strip() for x in v.split(CFG.MULTILABEL_SEP) if x.strip()})
    counts = pd.Series([x for ts in tags for x in ts]).value_counts()
    ML_LABELS = sorted(counts[counts >= 2 * CFG.N_FOLDS].index)
    rare_tags = sorted(set(counts.index) - set(ML_LABELS))
    if rare_tags:
        print(f'[warn] labels with fewer than {2 * CFG.N_FOLDS} positives are not modelled: {rare_tags[:20]}')
    for lab in ML_LABELS:
        df[f'{T}__{lab}'] = tags.map(lambda ts, lab=lab: int(lab in ts))
    TARGETS = [f'{T}__{lab}' for lab in ML_LABELS]
    T = TARGETS[0]
    print(f'Multi-label: {len(ML_LABELS)} labels from {ML_SOURCE!r}, mean labels per row {tags.map(len).mean():.2f}')
n_sent = sum(int(df_raw[c].isin(v).sum()) if pd.api.types.is_numeric_dtype(df_raw[c]) else 0 for c, v in SENTINELS.items() if c in df_raw.columns)
print(f'Target {T!r}' + (f' (+{len(TARGETS) - 1} more targets)' if len(TARGETS) > 1 else '') + f' | ids {ID_LIST} | dropped {DROP} | sentinel values set to NaN: {n_sent:,} in {list(SENTINELS)}')

## Column Roles

Roles are inferred with the same rules as the EDA template. Numeric columns with at most `DISCRETE_MAX` distinct values are treated as categorical, long multi-word strings as text, and near-unique strings as identifiers. Explicit lists in `Settings` override the inference.

In [ ]:
def infer_roles(frame):
    roles = {}
    for c in frame.columns:
        if c in TARGETS or c == ML_SOURCE:
            continue
        s = frame[c]
        nun = s.nunique(dropna=True)
        if c in ID_LIST:
            roles[c] = 'id'
        elif c in CFG.DATE_COLS or pd.api.types.is_datetime64_any_dtype(s):
            roles[c] = 'datetime'
        elif nun <= 1:
            roles[c] = 'constant'
        elif pd.api.types.is_numeric_dtype(s) and not pd.api.types.is_bool_dtype(s):
            roles[c] = 'categorical' if nun <= CFG.DISCRETE_MAX else 'numeric'
        elif pd.api.types.is_bool_dtype(s):
            roles[c] = 'categorical'
        else:
            st = s.dropna().astype(str)
            st = st.sample(min(len(st), 3000), random_state=CFG.SEED)
            words = st.str.split().str.len().mean()
            if words >= 6:
                roles[c] = 'text'
            elif nun / max(s.notna().sum(), 1) > 0.95:
                roles[c] = 'id'
            else:
                roles[c] = 'categorical'
    overrides = (('numeric', CFG.NUM_COLS, 'num_cols'), ('categorical', CFG.CAT_COLS, 'cat_cols'), ('text', CFG.TEXT_COLS, 'text_cols'))
    for role, user, key in overrides:
        cols = user if user != 'auto' else EDA.get(key, 'auto')
        if isinstance(cols, (list, tuple)):
            for c in cols:
                if c in roles:
                    roles[c] = role
    return roles

ROLES     = infer_roles(df)
DECISIONS.append({'setting': 'COLUMN_ROLES', 'value': f'{sum(r == "numeric" for r in ROLES.values())} num / {sum(r == "categorical" for r in ROLES.values())} cat / {sum(r == "text" for r in ROLES.values())} text',
                  'source': 'eda' if EDA.get('num_cols') is not None else 'computed', 'why': 'role inference shared with the EDA'})
NUM_COLS  = [c for c, r in ROLES.items() if r == 'numeric']
CAT_COLS  = [c for c, r in ROLES.items() if r == 'categorical']
TEXT_COLS = [c for c, r in ROLES.items() if r == 'text']
DT_COLS   = [c for c, r in ROLES.items() if r == 'datetime']
for c in NUM_COLS:
    df[c] = pd.to_numeric(df[c], errors='coerce')
for role in ['numeric', 'categorical', 'text', 'datetime', 'id', 'constant']:
    cols = [c for c, r in ROLES.items() if r == role]
    if cols:
        print(f'{role:<12} ({len(cols):>2}): {cols}')

## Target Preparation

Rows with a missing target are dropped. For classification the labels are encoded to `0..K-1`, classes too small to appear in every fold are removed, and class weighting is switched on automatically when the imbalance ratio exceeds 3. For regression a `log1p` transform is applied automatically when the target is non-negative and right-skewed (skew above 1). Models train on the transformed scale and predictions are always mapped back.

In [ ]:
n0 = len(df)
df = df.dropna(subset=TARGETS).reset_index(drop=True)
print(f'Dropped {n0 - len(df):,} rows with missing target')

TASK = decide('TASK', CFG.TASK, 'task', lambda: 'classification' if (not pd.api.types.is_numeric_dtype(df[T]) or df[T].nunique() <= 20) else 'regression',
              lambda v: 'string or few-valued target' if v == 'classification' else 'numeric target with many distinct values')
if CFG.TASK == 'auto' and CFG.METRIC == 'qwk':
    TASK = 'ordinal'
if ML_SOURCE:
    TASK = 'classification'
IS_ORD = TASK == 'ordinal'
IS_REG = TASK in ('regression', 'ordinal')
IS_CLF = TASK == 'classification'

def ordinal_order(values):
    """Class order for an ordinal target: ORDINAL_ORDER, else numeric order, else sorted strings."""
    if CFG.ORDINAL_ORDER:
        return list(CFG.ORDINAL_ORDER)
    uniq = pd.Series(values.dropna().unique())
    as_num = pd.to_numeric(uniq, errors='coerce')
    return list(uniq.iloc[np.argsort(as_num.values)]) if as_num.notna().all() else sorted(uniq.astype(str))

if IS_ORD:
    order = ordinal_order(df[T])
    rank = {str(v): i for i, v in enumerate(order)}
    known = df[T].astype(str).isin(rank)
    if (~known).any():
        print(f'[warn] dropping {int((~known).sum()):,} rows whose label is not in ORDINAL_ORDER')
        df = df[known].reset_index(drop=True)
    y_all = df[T].astype(str).map(rank).astype(float).values
    CLASS_NAMES, N_CLASSES, IS_BINARY, USE_CW, TRANSFORM, LE = list(order), len(order), False, False, None, None
    CUTS[0] = np.arange(N_CLASSES - 1) + 0.5
    print(f'Task: ordinal | {N_CLASSES} ordered classes {CLASS_NAMES} | trained as regression on the class rank, '
          'labels from tuned cut-points')
elif IS_CLF:
    labels = df[T].astype(str)
    vc = labels.value_counts()
    min_count = 2 * CFG.N_FOLDS
    rare = vc[vc < min_count].index.tolist()
    if rare:
        print(f'[warn] dropping classes with fewer than {min_count} rows: {rare}')
        df = df[~labels.isin(rare)].reset_index(drop=True)
        labels = df[T].astype(str)
    LE = LabelEncoder().fit(labels)
    y_all = LE.transform(labels)
    CLASS_NAMES = list(LE.classes_)
    N_CLASSES = len(CLASS_NAMES)
    IS_BINARY = N_CLASSES == 2
    counts = np.bincount(y_all)
    imbalance = counts.max() / counts.min()
    cw = decide('CLASS_WEIGHT', CFG.CLASS_WEIGHT, 'class_weights', lambda: bool(imbalance > 3), f'imbalance ratio {imbalance:.1f}x')
    USE_CW = cw in (True, 'balanced')
    TRANSFORM = None
    print(f'Task: classification | {N_CLASSES} classes | imbalance {imbalance:.1f}x | class weights: {USE_CW}')
else:
    y_all = df[T].astype(float).values
    N_CLASSES, IS_BINARY, USE_CW, CLASS_NAMES = 1, False, False, None
    skew = pd.Series(y_all).skew()
    TRANSFORM = decide('TARGET_TRANSFORM', CFG.TARGET_TRANSFORM, 'target_transform',
                       lambda: 'log1p' if (y_all.min() >= 0 and skew > 1) else 'none', f'target skew {skew:.2f}')
    TRANSFORM = None if TRANSFORM in (None, 'none') else TRANSFORM
    print(f'Task: regression | skew {skew:.2f} | target transform: {TRANSFORM}')

METRIC = decide('METRIC', CFG.METRIC, 'metric',
                lambda: 'qwk' if IS_ORD else 'f1' if ML_SOURCE else 'rmse' if IS_REG else 'roc_auc' if IS_BINARY else 'f1_macro',
                'QWK for ordinal, F1 per label for multi-label, RMSE for regression, ROC AUC for binary, macro F1 for multiclass')
if IS_ORD and METRIC not in ('qwk', 'accuracy', 'f1_macro', 'mae_class', 'rmse'):
    METRIC = 'qwk'
print(f'Selection metric: {METRIC} ({"higher" if GREATER_IS_BETTER[METRIC] else "lower"} is better)')

def fwd(y):
    return np.log1p(y) if TRANSFORM == 'log1p' else y

def inv(y):
    return np.expm1(y) if TRANSFORM == 'log1p' else y

## Duplicate Removal

Exact duplicate rows add no information and inflate cross-validation scores when copies land in different folds, so they are removed. Near-duplicates (same entity, different rows) are handled later by the group split rather than by deletion.

In [ ]:
DEDUP = bool(decide('DEDUP', CFG.DEDUP, 'dedup', lambda: True,
                   lambda v: 'identical rows found in the EDA duplicate audit' if v else 'no identical rows in the EDA duplicate audit'))
if DEDUP:
    key = pd.util.hash_pandas_object(df.astype(str), index=False)
    keep = ~key.duplicated().values
    print(f'Removed {int((~keep).sum()):,} exact duplicate rows ({(~keep).mean():.1%})')
    df = df[keep].reset_index(drop=True)
    y_all = y_all[keep]
print(f'Rows available for modelling: {len(df):,}')

## Holdout Split

The validation scheme follows the EDA: `group` when an identifier repeats across rows (the EDA also measures how much a random split overstates the score), `time` when a time column exists and train and test drift apart, otherwise `stratified` for classification and plain `kfold`. The holdout split follows the same logic. Time order keeps the latest rows, groups never straddle the boundary, and classes keep their proportions. The holdout is used once, in section 11.

In [ ]:
gap = EDA.get('group_vs_random_gap')
why = f'EDA: random vs group K-fold gap {gap:.1%}' if gap is not None else 'repeated ids -> group, drift -> time'
SCHEME = decide('CV_SCHEME', CFG.CV_SCHEME, 'cv_scheme',
                lambda: 'time' if TIME_COL else 'group' if CV_GROUP else ('stratified' if (IS_CLF or IS_ORD) else 'kfold'), why)
if SCHEME == 'time' and TIME_COL is None:
    SCHEME = 'group' if CV_GROUP else 'kfold'
if SCHEME == 'group' and CV_GROUP is None:
    SCHEME = 'stratified' if (IS_CLF or IS_ORD) else 'kfold'
if SCHEME == 'stratified' and IS_REG and not IS_ORD:
    SCHEME = 'kfold'
GROUPS_ALL = df[CV_GROUP].astype(str).values if SCHEME == 'group' else None

idx = np.arange(len(df))
if CFG.HOLDOUT_SIZE and CFG.HOLDOUT_SIZE > 0:
    if SCHEME == 'time':
        order = np.argsort(df[TIME_COL].values, kind='stable')
        cut = int(len(order) * (1 - CFG.HOLDOUT_SIZE))
        dev_idx, hold_idx = order[:cut], order[cut:]
    elif SCHEME == 'group':
        if IS_CLF or IS_ORD:
            sgk = StratifiedGroupKFold(n_splits=max(2, int(round(1 / CFG.HOLDOUT_SIZE))), shuffle=True, random_state=CFG.SEED)
            dev_idx, hold_idx = next(sgk.split(idx, y_all.astype(int), GROUPS_ALL))
        else:
            gss = GroupShuffleSplit(n_splits=1, test_size=CFG.HOLDOUT_SIZE, random_state=CFG.SEED)
            dev_idx, hold_idx = next(gss.split(idx, y_all, GROUPS_ALL))
    else:
        dev_idx, hold_idx = train_test_split(idx, test_size=CFG.HOLDOUT_SIZE, random_state=CFG.SEED,
                                             stratify=y_all.astype(int) if (IS_CLF or IS_ORD) else None)
else:
    dev_idx, hold_idx = (np.argsort(df[TIME_COL].values, kind='stable') if SCHEME == 'time' else idx), np.array([], dtype=int)

dev_raw, hold_raw = df.iloc[dev_idx].reset_index(drop=True), df.iloc[hold_idx].reset_index(drop=True)
y_dev, y_hold = y_all[dev_idx], y_all[hold_idx]
groups_dev = GROUPS_ALL[dev_idx] if GROUPS_ALL is not None else None
HAS_HOLD = len(hold_idx) > 0
if GROUPS_ALL is not None and HAS_HOLD:
    overlap = len(set(GROUPS_ALL[dev_idx]) & set(GROUPS_ALL[hold_idx]))
    print(f'Group overlap between dev and holdout: {overlap}')
print(f'Scheme: {SCHEME} | dev {len(dev_idx):,} rows | holdout {len(hold_idx):,} rows')
display(pd.DataFrame(DECISIONS).set_index('setting'))

#### Insights

> *Fill in after running.* Record the task, the metric and why it fits the business question, the target transform, how many duplicates and rare classes were removed, and the validation scheme. These decisions belong in the report's methodology chapter.

---

# Feature Engineering <a name="19"></a>

---

Feature engineering here is generic, so it is safe on any dataset. Date columns become calendar parts and elapsed days. Text columns become length statistics plus a compact TF-IDF to SVD semantic summary. Categorical columns gain a frequency encoding. Numeric columns with gaps gain missing-value flags. Every stateful step is fitted on the development set only and then applied unchanged to the holdout and test sets.

## Custom Features Hook

Domain features usually beat generic ones. Add competition-specific features to `custom_features`, which receives the raw rows and the engineered feature frame. It must only use information available in a single row (or fitted on dev), so that holdout and test rows are transformed identically. The commented lines show typical patterns.

In [ ]:
def custom_features(raw, feats):
    # feats['energy_x_dance'] = raw['energy'] * raw['danceability']
    # feats['duration_min'] = raw['duration_ms'] / 60000
    # feats['n_artists'] = raw['artists'].astype(str).str.count(';') + 1
    return feats

## Feature Builder

The builder follows the scikit-learn `fit` / `transform` contract. Categorical values are cast to strings with an explicit `(missing)` level so that every model sees the same categories.

In [ ]:
class FeatureBuilder:
    def fit(self, frame):
        self.num_cols = list(NUM_COLS)
        self.cat_cols = list(CAT_COLS)
        # near-unique columns give a constant frequency in training, which carries no signal
        self.freq = {c: frame[c].astype(str).value_counts(normalize=True) for c in self.cat_cols
                     if frame[c].nunique() <= 0.5 * len(frame)} if CFG.ADD_FREQ_ENCODING else {}
        self.miss_cols = [c for c in self.num_cols if frame[c].isna().any()] if CFG.ADD_MISSING_FLAGS else []
        self.text = {}
        for c in TEXT_COLS if CFG.TEXT_SVD_DIM else []:
            docs = frame[c].fillna('').astype(str)
            docs = docs.sample(min(len(docs), 30_000), random_state=CFG.SEED)
            vec = TfidfVectorizer(max_features=20_000, min_df=3, sublinear_tf=True, stop_words='english',
                                  token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z']+\b")
            mat = vec.fit_transform(docs)
            k = min(CFG.TEXT_SVD_DIM, mat.shape[1] - 1)
            if k >= 1:
                self.text[c] = (vec, TruncatedSVD(k, random_state=CFG.SEED).fit(mat))
        return self

    def transform(self, frame):
        out = pd.DataFrame(index=frame.index)
        for c in self.num_cols:
            out[c] = pd.to_numeric(frame[c], errors='coerce') if c in frame.columns else np.nan
        for c in self.cat_cols:
            s = frame[c] if c in frame.columns else pd.Series(np.nan, index=frame.index)
            out[c] = s.astype(object).where(s.notna(), '(missing)').astype(str).astype(object)
        for c in DT_COLS:
            d = pd.to_datetime(frame[c], errors='coerce')
            out[f'{c}_year'], out[f'{c}_month'] = d.dt.year, d.dt.month
            out[f'{c}_dow'], out[f'{c}_doy'] = d.dt.dayofweek, d.dt.dayofyear
            out[f'{c}_days'] = (d - pd.Timestamp('1970-01-01')).dt.days
        for c in TEXT_COLS:
            s = frame[c].fillna('').astype(str) if c in frame.columns else pd.Series('', index=frame.index)
            words = s.str.split()
            out[f'{c}_n_words'] = words.str.len()
            out[f'{c}_n_chars'] = s.str.len()
            out[f'{c}_n_lines'] = s.str.count('\n') + 1
            out[f'{c}_unique_ratio'] = [len(set(w)) / max(len(w), 1) for w in words]
            if c in self.text:
                vec, svd = self.text[c]
                Z = svd.transform(vec.transform(s)) if len(s) else np.zeros((0, svd.n_components))
                for j in range(Z.shape[1]):
                    out[f'{c}_svd{j}'] = Z[:, j]
        for c, f in self.freq.items():
            src = frame[c].astype(str) if c in frame.columns else pd.Series('nan', index=frame.index)
            out[f'{c}_freq'] = src.map(f).fillna(0).astype(float).values
        for c in self.miss_cols:
            out[f'{c}_isna'] = out[c].isna().astype(int)
        if self.miss_cols:
            out['n_missing'] = out[self.miss_cols].isna().sum(axis=1)
        return custom_features(frame, out)

FB = FeatureBuilder().fit(dev_raw)
X_dev = FB.transform(dev_raw).reset_index(drop=True)
X_hold = FB.transform(hold_raw).reset_index(drop=True)
X_test = FB.transform(test_df).reset_index(drop=True) if test_df is not None else None

F_CAT = [c for c in FB.cat_cols if c in X_dev.columns]
F_NUM = [c for c in X_dev.columns if c not in F_CAT]
const = [c for c in F_NUM if X_dev[c].nunique(dropna=False) <= 1] + [c for c in F_CAT if X_dev[c].nunique() <= 1]
if const:
    print(f'Dropping constant features: {const}')
    F_NUM = [c for c in F_NUM if c not in const]
    F_CAT = [c for c in F_CAT if c not in const]
weak = decide('DROP_WEAK', 'auto' if CFG.DROP_WEAK in ('auto', True) else [], 'weak_features', lambda: [],
              'no mutual information and no significant association with the target')
WEAK = [c for c in (weak if isinstance(weak, list) else []) if c in F_NUM + F_CAT]
if WEAK:
    print(f'Dropping weak features flagged by the EDA: {WEAK}')
    F_NUM = [c for c in F_NUM if c not in WEAK]
    F_CAT = [c for c in F_CAT if c not in WEAK]
FEATURES = F_NUM + F_CAT
for frame in (X_dev, X_hold, X_test):
    if frame is not None:
        frame[F_NUM] = frame[F_NUM].astype(float)
print(f'Features: {len(FEATURES)} ({len(F_NUM)} numeric, {len(F_CAT)} categorical)')

The inventory groups the final features by origin, which is the table to quote when the report describes the feature set.

In [ ]:
def origin(c):
    if c in NUM_COLS:
        return 'raw numeric'
    if c in CAT_COLS:
        return 'raw categorical'
    if c.endswith('_freq'):
        return 'frequency encoding'
    if c.endswith('_isna') or c == 'n_missing':
        return 'missing flags'
    if any(c.startswith(f'{t}_svd') for t in TEXT_COLS):
        return 'text TF-IDF SVD'
    if any(c.startswith(f'{t}_') for t in TEXT_COLS):
        return 'text statistics'
    if any(c.startswith(f'{t}_') for t in DT_COLS):
        return 'date parts'
    return 'custom'

inventory = pd.DataFrame({'feature': FEATURES, 'origin': [origin(c) for c in FEATURES],
                          'type': ['categorical' if c in F_CAT else 'numeric' for c in FEATURES]})
inventory.to_csv(CFG.FINAL_DIR / 'feature_inventory.csv', index=False)
display(inventory.groupby(['origin', 'type']).size().rename('n_features').to_frame())
display(X_dev.head(3))

#### Insights

> *Fill in after running.* Which feature families were created, and which domain features were added in `custom_features`? Justify each custom feature with an EDA finding.

---

# Validation Strategy <a name="20"></a>

---

All models share one fixed list of folds. Group folds keep every row of an entity in the same fold, which prevents near-duplicate leakage. Time folds always validate on rows later than the training rows. Stratified folds preserve class proportions. The diagnostics confirm that folds are balanced in size and target distribution.

In [ ]:
def make_folds(n_splits=None):
    n_splits = n_splits or CFG.N_FOLDS
    idx = np.arange(len(X_dev))
    if SCHEME == 'time':
        return list(TimeSeriesSplit(n_splits=n_splits).split(idx))
    if SCHEME == 'group':
        if IS_CLF or IS_ORD:
            return list(StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=CFG.SEED).split(idx, y_dev.astype(int), groups_dev))
        return list(GroupKFold(n_splits=n_splits).split(idx, y_dev, groups_dev))
    if SCHEME == 'stratified':
        return list(StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=CFG.SEED).split(idx, y_dev.astype(int)))
    return list(KFold(n_splits=n_splits, shuffle=True, random_state=CFG.SEED).split(idx))

FOLDS = make_folds()
rows = []
for k, (tr, va) in enumerate(FOLDS):
    r = {'fold': k + 1, 'n_train': len(tr), 'n_valid': len(va)}
    if IS_REG:
        r.update({'valid_mean': y_dev[va].mean(), 'valid_std': y_dev[va].std()})
    else:
        r.update({'valid_classes': len(np.unique(y_dev[va])), 'valid_major_share': np.bincount(y_dev[va]).max() / len(va)})
    if groups_dev is not None:
        r['group_overlap'] = len(set(groups_dev[tr]) & set(groups_dev[va]))
    rows.append(r)
fold_table = pd.DataFrame(rows).set_index('fold')
display(fold_table)

fig, axes = plt.subplots(1, 2, figsize=(15, 4.2))
axes[0].bar(fold_table.index, fold_table['n_train'], color=SOFT, label='train')
axes[0].bar(fold_table.index, fold_table['n_valid'], bottom=fold_table['n_train'], color=ACCENT, label='valid')
axes[0].set_xlabel('fold')
axes[0].legend()
axes[0].set_title('Rows per fold', loc='left')
if IS_REG:
    data = [y_dev[va] for _, va in FOLDS]
    bp = axes[1].boxplot(data, patch_artist=True, widths=0.5, showfliers=False)
    for patch, col in zip(bp['boxes'], PALETTE):
        patch.set_facecolor(col)
        patch.set_alpha(0.7)
    axes[1].set_title(f'{T} distribution in each validation fold', loc='left')
else:
    top = np.argsort(-np.bincount(y_dev))[:8]
    shares = np.array([[np.mean(y_dev[va] == c) for c in top] for _, va in FOLDS])
    bottom = np.zeros(len(FOLDS))
    for j, c in enumerate(top):
        axes[1].bar(range(1, len(FOLDS) + 1), shares[:, j], bottom=bottom, color=PALETTE[j % 10], label=short(CLASS_NAMES[c], 14))
        bottom += shares[:, j]
    axes[1].legend(fontsize=8, ncol=2)
    axes[1].set_title('Class share per validation fold (top 8 classes)', loc='left')
axes[1].set_xlabel('fold')
suptitle(fig, f'Validation design: {SCHEME} {CFG.N_FOLDS}-fold', f'{len(X_dev):,} dev rows, holdout {len(X_hold):,} rows')
save_fig(fig, 'validation_folds', 'final')

#### Insights

> *Fill in after running.* Are the folds balanced, and is the group overlap zero? State why this scheme mirrors how the model will be used (for example, scoring new songs or new users that were never seen in training).

---

# Baseline: Linear / Logistic Regression <a name="21"></a>

---

The baseline is a deliberately simple, fully interpretable linear model that depends only on scikit-learn, so it always runs. Every boosted model must beat it to justify its complexity. It also reveals whether the signal is mostly linear, which is valuable for the report narrative.

## Shared Model Interface

Every model in this notebook is a small wrapper with `fit(X_train, y_train, X_valid, y_valid)` and `predict(X)`. Regression wrappers apply the target transform internally and return values on the original scale. Classification wrappers return a probability matrix over all classes, even when a training fold misses a class. The `cross_validate` function runs any wrapper over the shared folds and averages fold models to predict the holdout and test sets.

In [ ]:
def sample_weights(y):
    if not USE_CW:
        return None
    counts = np.bincount(y, minlength=N_CLASSES).astype(float)
    w = len(y) / (np.count_nonzero(counts) * np.where(counts == 0, 1, counts))
    return w[y]

def expand_proba(p, classes):
    full = np.zeros((len(p), N_CLASSES))
    full[:, classes] = p
    return full

def cross_validate(name, make, folds=None, verbose=True, eval_sets=True):
    folds = folds or FOLDS
    shape = (len(y_dev),) if IS_REG else (len(y_dev), N_CLASSES)
    oof = np.full(shape, np.nan)
    hold = np.zeros((len(X_hold),) + shape[1:])
    test = np.zeros((len(X_test),) + shape[1:]) if X_test is not None else None
    models, fold_scores, t0 = [], [], time.time()
    for k, (tr, va) in enumerate(folds):
        m = make().fit(X_dev.iloc[tr], y_dev[tr], X_dev.iloc[va], y_dev[va])
        oof[va] = m.predict(X_dev.iloc[va])
        fold_scores.append(primary(y_dev[va], oof[va]))
        if verbose:
            print(f'  [{name}] fold {k + 1}/{len(folds)}  {METRIC} = {fold_scores[-1]:.4f}  ({time.time() - t0:.0f}s)')
        if eval_sets and HAS_HOLD:
            hold += m.predict(X_hold) / len(folds)
        if eval_sets and test is not None:
            test += m.predict(X_test) / len(folds)
        models.append(m)
    covered = ~np.isnan(oof if IS_REG else oof[:, 0])
    cv = score_all(y_dev[covered], oof[covered])
    res = {'name': name, 'oof': oof, 'covered': covered, 'hold': hold, 'test': test, 'models': models,
           'fold_scores': fold_scores, 'cv': cv, 'cv_primary': cv[METRIC], 'cv_std': float(np.std(fold_scores)),
           'time': time.time() - t0}
    res['holdout'] = score_all(y_hold, hold) if HAS_HOLD and eval_sets else {}
    if verbose:
        print(f'  [{name}] CV {METRIC} = {res["cv_primary"]:.4f} +/- {res["cv_std"]:.4f}  in {res["time"]:.0f}s')
    return res

## Linear Preprocessing and Model

Numeric features are median-imputed, standardised, and passed through a Yeo-Johnson power transform, which tames skew and outliers for a linear model. Transformed values are clipped to +/-5 so that a value never seen in training cannot push a prediction to 0 or 1. Low-cardinality categoricals (at most `MAX_OHE_LEVELS` levels) are one-hot encoded with rare levels grouped. High-cardinality categoricals use scikit-learn's cross-fitted `TargetEncoder`, which avoids target leakage, and fall back to their frequency encoding on older scikit-learn versions. The estimator is `LinearRegression` or `LogisticRegression` (multinomial, balanced class weights when imbalance is high). Setting `BASELINE = 'ridge'` swaps in Ridge regression for more stable coefficients.

In [ ]:
class LinearModel:
    def __init__(self, params=None):
        self.params = params or {}

    def _preprocessor(self, X):
        low = [c for c in F_CAT if X[c].nunique() <= CFG.MAX_OHE_LEVELS]
        high = [c for c in F_CAT if c not in low]
        num_pipe = Pipeline([('impute', SimpleImputer(strategy='median')), ('scale', StandardScaler()),
                             ('power', PowerTransformer(method='yeo-johnson')),
                             ('clip', FunctionTransformer(np.clip, kw_args={'a_min': -5, 'a_max': 5}, feature_names_out='one-to-one'))])
        steps = [('num', num_pipe, F_NUM)]
        if low:
            steps.append(('onehot', OneHotEncoder(handle_unknown='infrequent_if_exist', min_frequency=10,
                                                  drop='first', sparse_output=False), low))
        if high and AVAILABLE['target_encoder']:
            te = TargetEncoder(target_type='continuous' if IS_REG else 'auto', random_state=CFG.SEED)
            steps.append(('target_enc', Pipeline([('te', te), ('scale', StandardScaler())]), high))
        return ColumnTransformer(steps, remainder='drop', verbose_feature_names_out=False)

    def fit(self, X, y, X_valid=None, y_valid=None):
        pre = self._preprocessor(X)
        if IS_REG:
            est = Ridge(alpha=self.params.get('alpha', 1.0)) if CFG.BASELINE == 'ridge' else LinearRegression()
            self.pipe = Pipeline([('pre', pre), ('est', est)]).fit(X, fwd(y))
        else:
            self.classes, yl = np.unique(y, return_inverse=True)
            est = LogisticRegression(max_iter=3000, C=self.params.get('C', 1.0),
                                     class_weight='balanced' if USE_CW else None)
            self.pipe = Pipeline([('pre', pre), ('est', est)]).fit(X, yl)
        return self

    def predict(self, X):
        if IS_REG:
            return inv(self.pipe.predict(X))
        return expand_proba(self.pipe.predict_proba(X), self.classes)

    def coefficients(self):
        names = self.pipe.named_steps['pre'].get_feature_names_out()
        coef = self.pipe.named_steps['est'].coef_
        if coef.ndim == 2 and coef.shape[0] > 1:
            return pd.Series(np.abs(coef).mean(axis=0), index=names), True
        return pd.Series(np.ravel(coef), index=names), False

## Baseline Cross-Validation

The baseline runs on the shared folds. A naive predictor (training mean for regression, class priors for classification) is scored alongside it, so the improvement the linear model brings over guessing is explicit.

In [ ]:
def naive_prediction(n):
    if IS_REG:
        return np.full(n, y_dev.mean())
    prior = np.bincount(y_dev, minlength=N_CLASSES) / len(y_dev)
    return np.tile(prior, (n, 1))

BASE = cross_validate('baseline', lambda: LinearModel())
RESULTS = {'baseline': BASE}

NAIVE = {'cv': score_all(y_dev, naive_prediction(len(y_dev)))}
NAIVE['holdout'] = score_all(y_hold, naive_prediction(len(y_hold))) if HAS_HOLD else {}

base_table = pd.DataFrame({'naive (CV)': NAIVE['cv'], 'baseline (CV)': BASE['cv']})
if HAS_HOLD:
    base_table['naive (holdout)'] = NAIVE['holdout']
    base_table['baseline (holdout)'] = BASE['holdout']
display(base_table.round(4))
pd.DataFrame({'fold': range(1, len(FOLDS) + 1), METRIC: BASE['fold_scores']}).to_csv(CFG.BASELINE_DIR / 'cv_scores.csv', index=False)
base_table.to_csv(CFG.BASELINE_DIR / 'metrics.csv')

## Baseline Diagnostics

The diagnostics function below is shared with the final evaluation. For regression it shows predicted against actual values, residuals against predictions, and the residual distribution. For classification it shows the confusion matrix together with ROC and precision-recall curves (binary) or per-class F1 (multiclass).

In [ ]:
def plot_diagnostics(y_true, pred, title, stage, subtitle=''):
    fig, axes = plt.subplots(1, 3, figsize=(19, 5.4))
    if IS_REG:
        res = y_true - pred
        ax = axes[0]
        ax.hexbin(y_true, pred, gridsize=40, cmap=CMAP_SEQ, mincnt=1, bins='log', linewidths=0)
        lims = [min(y_true.min(), pred.min()), max(y_true.max(), pred.max())]
        ax.plot(lims, lims, color=ACCENT, ls='--', lw=1.5)
        ax.set_xlabel(f'actual {T}')
        ax.set_ylabel('predicted')
        ax.set_title('Predicted vs actual', loc='left')
        ax = axes[1]
        ax.hexbin(pred, res, gridsize=40, cmap=CMAP_SEQ, mincnt=1, bins='log', linewidths=0)
        d = pd.DataFrame({'p': pred, 'r': res})
        b = d.groupby(pd.qcut(d['p'].rank(method='first'), 20), observed=True).mean()
        ax.plot(b['p'], b['r'], color=ACCENT, marker='o', ms=3, lw=2)
        ax.axhline(0, color=DARK, lw=1)
        ax.set_xlabel('predicted')
        ax.set_ylabel('residual (actual - predicted)')
        ax.set_title('Residuals vs predicted (orange = binned mean)', loc='left')
        ax = axes[2]
        sns.histplot(res, bins=60, color=PRIMARY, edgecolor='white', ax=ax, kde=True)
        ax.axvline(0, color=ACCENT, ls='--')
        ax.set_title(f'Residual distribution (MAE {np.abs(res).mean():.3f})', loc='left')
    else:
        lab = to_label(pred)
        keep = np.argsort(-np.bincount(y_true, minlength=N_CLASSES))[:15]
        keep = np.sort(keep)
        cm = confusion_matrix(y_true, lab, labels=list(range(N_CLASSES)), normalize='true')[np.ix_(keep, keep)]
        names = [short(CLASS_NAMES[i], 12) for i in keep]
        sns.heatmap(cm, cmap=CMAP_SEQ, vmin=0, vmax=1, annot=len(keep) <= 12, fmt='.2f', ax=axes[0],
                    xticklabels=names, yticklabels=names, cbar_kws={'shrink': 0.7})
        axes[0].set_xlabel('predicted')
        axes[0].set_ylabel('actual')
        extra = f', top {len(keep)} of {N_CLASSES} classes' if N_CLASSES > len(keep) else ''
        axes[0].set_title(f'Row-normalised confusion matrix{extra}', loc='left')
        if IS_BINARY:
            fpr, tpr, _ = roc_curve(y_true, pred[:, 1])
            axes[1].plot(fpr, tpr, color=PRIMARY, lw=2.2)
            axes[1].fill_between(fpr, tpr, color=SOFT, alpha=0.3)
            axes[1].plot([0, 1], [0, 1], color=MUTED, ls='--')
            axes[1].set_xlabel('false positive rate')
            axes[1].set_ylabel('true positive rate')
            axes[1].set_title(f'ROC curve (AUC {roc_auc_score(y_true, pred[:, 1]):.3f})', loc='left')
            prec, rec, _ = precision_recall_curve(y_true, pred[:, 1])
            axes[2].plot(rec, prec, color=ACCENT, lw=2.2)
            axes[2].axhline(y_true.mean(), color=MUTED, ls='--', label='prevalence')
            axes[2].set_xlabel('recall')
            axes[2].set_ylabel('precision')
            axes[2].legend()
            axes[2].set_title(f'Precision-recall (AP {average_precision_score(y_true, pred[:, 1]):.3f})', loc='left')
        else:
            p, r, f, s = precision_recall_fscore_support(y_true, lab, labels=list(range(N_CLASSES)), zero_division=0)
            per = pd.DataFrame({'f1': f, 'support': s}, index=CLASS_NAMES).sort_values('f1')
            show = per if len(per) <= 25 else pd.concat([per.head(12), per.tail(12)])
            axes[1].barh([short(i, 18) for i in show.index], show['f1'], color=[ACCENT if v < per['f1'].median() else PRIMARY for v in show['f1']])
            axes[1].tick_params(axis='y', labelsize=7)
            axes[1].set_title('Per-class F1 (orange = below median)', loc='left')
            axes[2].scatter(per['support'], per['f1'], color=PRIMARY, s=30, alpha=0.7)
            for name, row in per.iterrows():
                axes[2].annotate(short(name, 12), (row['support'], row['f1']), fontsize=7, color=MUTED)
            axes[2].set_xscale('log')
            axes[2].set_xlabel('support (log)')
            axes[2].set_ylabel('F1')
            axes[2].set_title('F1 vs class size', loc='left')
    suptitle(fig, title, subtitle)
    save_fig(fig, title, stage)

plot_diagnostics(y_dev[BASE['covered']], BASE['oof'][BASE['covered']], 'Baseline out-of-fold diagnostics', 'baseline',
                 f'CV {METRIC} = {BASE["cv_primary"]:.4f}')
if HAS_HOLD:
    plot_diagnostics(y_hold, BASE['hold'], 'Baseline holdout diagnostics', 'baseline',
                     f'holdout {METRIC} = {BASE["holdout"][METRIC]:.4f}')

## Baseline Coefficients

A single baseline is refitted on the full development set to read its coefficients. Because numeric inputs are standardised, coefficient magnitudes are comparable across features. For multiclass problems the mean absolute coefficient across classes is shown.

In [ ]:
BASE_FULL = LinearModel().fit(X_dev, y_dev)
coef, is_abs = BASE_FULL.coefficients()
coef_table = coef.rename('coefficient').to_frame().assign(abs=coef.abs()).sort_values('abs', ascending=False)
coef_table.to_csv(CFG.BASELINE_DIR / 'coefficients.csv')

top = coef_table.head(25).iloc[::-1]
fig, ax = plt.subplots(figsize=(11, 0.32 * len(top) + 1.8))
colors = [PRIMARY if is_abs or v > 0 else ACCENT for v in top['coefficient']]
ax.barh([short(i, 34) for i in top.index], top['coefficient'], color=colors)
ax.axvline(0, color=DARK, lw=1)
scale = f'log1p({T})' if TRANSFORM else T
sub = 'mean |coefficient| across classes' if is_abs else f'effect of +1 SD on {scale if IS_REG else "log-odds"}, blue = positive, orange = negative'
suptitle(fig, 'Baseline: top 25 coefficients', sub)
save_fig(fig, 'baseline_coefficients', 'baseline')

## Baseline Export

All baseline artefacts are written to `pipeline-output/baseline/`: fold scores, metrics, out-of-fold and holdout predictions, coefficients, the fold models, and a submission file when a test set is configured.

In [ ]:
def prediction_frame(pred, raw=None, include_truth=None):
    out = pd.DataFrame(index=range(len(pred)))
    id_col = CFG.SUBMISSION_ID or next((c for c in ID_LIST if raw is not None and c in raw.columns), None)
    if id_col and raw is not None and id_col in raw.columns:
        out[id_col] = raw[id_col].values
    if include_truth is not None:
        out['actual'] = (np.array(CLASS_NAMES, dtype=object)[np.asarray(include_truth).astype(int)] if IS_ORD
                         else include_truth if IS_REG else LE.inverse_transform(include_truth))
    if IS_ORD:
        out[T] = np.array(CLASS_NAMES, dtype=object)[to_ordinal(pred)]
        out['score'] = pred
    elif IS_REG:
        out[T] = pred
    else:
        out[T] = LE.inverse_transform(to_label(pred))
        for j, c in enumerate(CLASS_NAMES):
            out[f'proba_{c}'] = pred[:, j]
    return out

def follow_sample(pred, frame):
    """Shape test predictions like sample_submission: same id order, column names, and label or probability format."""
    sid, tcols = SAMPLE.columns[0], list(SAMPLE.columns[1:])
    if sid not in frame.columns or not tcols or len(frame) != len(pred):
        print(f'[warn] sample submission id column {sid!r} is not in the test file; using the default submission layout')
        return None
    names = [str(c) for c in CLASS_NAMES] if not IS_REG else []
    if not IS_REG and len(tcols) > 1 and set(map(str, tcols)) <= set(names):
        vals = pd.DataFrame(pred[:, [names.index(str(c)) for c in tcols]], columns=tcols)
    else:
        ex = SAMPLE[tcols[0]].dropna()
        looks_proba = pd.api.types.is_float_dtype(ex) and not ex.isin([0, 1]).all()
        fmt = CFG.SUBMISSION_FORMAT if CFG.SUBMISSION_FORMAT != 'auto' else 'proba' if looks_proba else 'label'
        if IS_ORD:
            v = np.array(CLASS_NAMES, dtype=object)[to_ordinal(pred)]
        elif IS_REG:
            v = pred
        elif fmt == 'proba' and N_CLASSES == 2:
            v = pred[:, 1]
        else:
            v = LE.inverse_transform(to_label(pred))
        vals = pd.DataFrame({tcols[0]: v})
    vals.insert(0, sid, frame[sid].values)
    out = SAMPLE[[sid]].merge(vals, on=sid, how='left')
    n_missing = int(out[vals.columns[1:]].isna().any(axis=1).sum())
    if n_missing:
        print(f'[warn] {n_missing:,} sample-submission ids have no prediction')
    return out

def export_result(res, folder):
    cov = res['covered']
    prediction_frame(res['oof'][cov], dev_raw[cov], y_dev[cov]).to_csv(folder / 'oof_predictions.csv', index=False)
    if HAS_HOLD:
        prediction_frame(res['hold'], hold_raw, y_hold).to_csv(folder / 'holdout_predictions.csv', index=False)
    if res['test'] is not None:
        sub = prediction_frame(res['test'], test_df)
        id_cols = [c for c in sub.columns if c in (ID_LIST + [CFG.SUBMISSION_ID])]
        sub_out = follow_sample(res['test'], test_df) if SAMPLE is not None else None
        if sub_out is None:
            fmt = CFG.SUBMISSION_FORMAT
            if fmt == 'auto':
                fmt = 'proba' if not IS_REG and METRIC in ('roc_auc', 'pr_auc', 'logloss') else 'label'
            if fmt == 'proba' and not IS_REG and N_CLASSES == 2:
                sub_out = sub[id_cols].assign(**{T: res['test'][:, 1]})
            elif fmt == 'proba' and not IS_REG:
                sub_out = sub[id_cols + [f'proba_{c}' for c in CLASS_NAMES]].rename(columns=lambda c: c.removeprefix('proba_'))
            else:
                sub_out = sub[id_cols + [T]]
        sub_out.to_csv(folder / 'submission.csv', index=False)
        sub.to_csv(folder / 'test_predictions_full.csv', index=False)

export_result(BASE, CFG.BASELINE_DIR)
joblib.dump(BASE['models'], CFG.BASELINE_DIR / 'baseline_fold_models.joblib')
to_json({'task': TASK, 'metric': METRIC, 'scheme': SCHEME, 'target_transform': TRANSFORM,
         'model': 'Ridge' if (IS_REG and CFG.BASELINE == 'ridge') else 'LinearRegression' if IS_REG else 'LogisticRegression',
         'cv': BASE['cv'], 'cv_fold_scores': BASE['fold_scores'], 'holdout': BASE['holdout'], 'naive': NAIVE},
        CFG.BASELINE_DIR / 'metrics.json')
print('Baseline files:', sorted(p.name for p in CFG.BASELINE_DIR.iterdir()))

#### Insights

> *Fill in after running.* How much does the linear baseline improve on the naive predictor? Which coefficients dominate, and do their signs match the EDA? A small gain over naive means the signal is weak or non-linear, which motivates the boosted models below.

---

# Gradient Boosting Models <a name="22"></a>

---

Gradient boosted trees are the strongest general-purpose models for tabular data. They capture non-linearities and interactions without manual feature crosses and are insensitive to monotone transforms, so they use the raw engineered features without scaling. Each library handles categoricals natively. LightGBM and XGBoost receive pandas categoricals capped at `GBM_MAX_CAT_LEVELS` levels. XGBoost uses one-vs-rest category splits, because its partition splits overfit badly on high-cardinality columns in small data. CatBoost receives raw strings and applies ordered target statistics. Early stopping on the validation fold sets the number of trees. CatBoost is the slowest of the three on many-class problems, so for more than 10 classes it switches to cheaper categorical combinations automatically, and `USE_GPU = True` speeds it up considerably on Kaggle.

## Categorical Handling

Category levels are fixed from the development set, so every fold, the holdout, and the test set share an identical category mapping. Levels outside the most frequent `GBM_MAX_CAT_LEVELS - 1` are pooled into `(other)`, which also absorbs levels unseen at training time.

In [ ]:
CAT_LEVELS = {}
for c in F_CAT:
    keep = X_dev[c].value_counts().index[:CFG.GBM_MAX_CAT_LEVELS - 1].tolist()
    CAT_LEVELS[c] = keep + ([] if '(other)' in keep else ['(other)'])

def tree_frame(X, kind):
    Z = X[FEATURES].copy()
    for c in F_CAT:
        if kind == 'catboost':
            Z[c] = Z[c].astype(str).astype(object)
        else:
            s = Z[c].where(Z[c].isin(CAT_LEVELS[c]), '(other)')
            Z[c] = pd.Categorical(s, categories=CAT_LEVELS[c])
    return Z

print({c: len(v) for c, v in CAT_LEVELS.items()})

## Model Wrappers

The base class handles everything shared: the target transform for regression, mapping each fold's classes to a contiguous range (so a fold that misses a rare class does not crash), class weights, and expanding probabilities back to all classes. Each subclass only defines its default hyperparameters, the library-specific fit call, gain importance, and exact tree SHAP contributions through the library's own API (no `shap` package required). Defaults can be overridden per model through `Settings.MODEL_PARAMS`.

In [ ]:
class GBMModel:
    kind = None

    def __init__(self, params=None):
        self.params = {**self.defaults(), **CFG.MODEL_PARAMS.get(self.kind, {}), **(params or {})}

    def fit(self, X, y, X_valid=None, y_valid=None):
        Xt = tree_frame(X, self.kind)
        Xv = tree_frame(X_valid, self.kind) if X_valid is not None else None
        w = None
        if IS_REG:
            yt = fwd(y)
            yv = fwd(y_valid) if y_valid is not None else None
        else:
            self.classes = np.unique(y)
            yt = np.searchsorted(self.classes, y)
            w = sample_weights(y)
            yv = None
            if y_valid is not None:
                mask = np.isin(y_valid, self.classes)
                Xv, yv = Xv[mask], np.searchsorted(self.classes, y_valid[mask])
        self._fit(Xt, yt, Xv, yv, w)
        return self

    def predict(self, X):
        Z = tree_frame(X, self.kind)
        if IS_REG:
            return inv(np.asarray(self.model.predict(Z), dtype=float))
        return expand_proba(self.model.predict_proba(Z), self.classes)

    def n_local_classes(self):
        return 1 if IS_REG else len(self.classes)


class LGBModel(GBMModel):
    kind = 'lightgbm'

    def defaults(self):
        p = dict(n_estimators=CFG.N_ESTIMATORS, learning_rate=CFG.LEARNING_RATE, num_leaves=63,
                 min_child_samples=20, subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
                 reg_lambda=1.0, cat_smooth=10, random_state=CFG.SEED, n_jobs=-1, verbose=-1)
        if CFG.USE_GPU:
            p['device'] = 'gpu'
        return p

    def _fit(self, X, y, Xv, yv, w):
        cls = lgb.LGBMRegressor if IS_REG else lgb.LGBMClassifier
        self.model = cls(**self.params)
        kw = {}
        if Xv is not None:
            kw = dict(eval_set=[(Xv, yv)], callbacks=[lgb.early_stopping(CFG.EARLY_STOPPING, verbose=False)])
        self.model.fit(X, y, sample_weight=w, **kw)

    def best_iteration(self):
        return self.model.best_iteration_ or self.params['n_estimators']

    def importance(self):
        return pd.Series(self.model.booster_.feature_importance('gain'), index=FEATURES)

    def contributions(self, X):
        c = np.asarray(self.model.predict(tree_frame(X, self.kind), pred_contrib=True))
        k = self.n_local_classes()
        return c.reshape(len(X), k, -1)[:, :, :-1] if k > 2 else c[:, :-1]


class XGBModel(GBMModel):
    kind = 'xgboost'

    def defaults(self):
        p = dict(n_estimators=CFG.N_ESTIMATORS, learning_rate=CFG.LEARNING_RATE, max_depth=6,
                 min_child_weight=1, subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0,
                 tree_method='hist', enable_categorical=True, max_cat_to_onehot=CFG.GBM_MAX_CAT_LEVELS,
                 early_stopping_rounds=CFG.EARLY_STOPPING, random_state=CFG.SEED, n_jobs=-1)
        if CFG.USE_GPU:
            p['device'] = 'cuda'
        return p

    def _fit(self, X, y, Xv, yv, w):
        params = dict(self.params)
        if Xv is None:
            params.pop('early_stopping_rounds', None)
        cls = xgb.XGBRegressor if IS_REG else xgb.XGBClassifier
        self.model = cls(**params)
        self.model.fit(X, y, sample_weight=w, eval_set=[(Xv, yv)] if Xv is not None else None, verbose=False)

    def best_iteration(self):
        return getattr(self.model, 'best_iteration', None) or self.params['n_estimators']

    def importance(self):
        score = self.model.get_booster().get_score(importance_type='total_gain')
        return pd.Series(score).reindex(FEATURES).fillna(0.0)

    def contributions(self, X):
        dm = xgb.DMatrix(tree_frame(X, self.kind), enable_categorical=True)
        c = self.model.get_booster().predict(dm, pred_contribs=True, iteration_range=(0, self.best_iteration() + 1))
        return c[..., :-1]


class CatModel(GBMModel):
    kind = 'catboost'

    def defaults(self):
        p = dict(iterations=CFG.N_ESTIMATORS, learning_rate=CFG.LEARNING_RATE, depth=6, l2_leaf_reg=3.0,
                 random_seed=CFG.SEED, verbose=0, thread_count=-1, allow_writing_files=False,
                 early_stopping_rounds=CFG.EARLY_STOPPING)
        if IS_CLF and N_CLASSES > 10:
            p.update(max_ctr_complexity=1, boosting_type='Plain')
        if CFG.USE_GPU:
            p['task_type'] = 'GPU'
        return p

    def _fit(self, X, y, Xv, yv, w):
        params = dict(self.params)
        if IS_REG:
            params.setdefault('loss_function', 'RMSE')
            cls = CatBoostRegressor
        else:
            params.setdefault('loss_function', 'Logloss' if len(self.classes) == 2 else 'MultiClass')
            cls = CatBoostClassifier
        if Xv is None:
            params.pop('early_stopping_rounds', None)
        self.model = cls(**params)
        train_pool = Pool(X, y, cat_features=F_CAT, weight=w)
        eval_pool = Pool(Xv, yv, cat_features=F_CAT) if Xv is not None else None
        self.model.fit(train_pool, eval_set=eval_pool, use_best_model=eval_pool is not None)

    def best_iteration(self):
        return self.model.get_best_iteration() or self.params['iterations']

    def importance(self):
        return pd.Series(self.model.get_feature_importance(), index=FEATURES)

    def contributions(self, X):
        pool = Pool(tree_frame(X, self.kind), cat_features=F_CAT)
        return self.model.get_feature_importance(pool, type='ShapValues')[..., :-1]


MODEL_ZOO = {'lightgbm': LGBModel, 'xgboost': XGBModel, 'catboost': CatModel}
ACTIVE = [m for m in CFG.MODELS if m in MODEL_ZOO and AVAILABLE.get(m)]
SKIPPED = [m for m in CFG.MODELS if m not in ACTIVE]
print(f'Models to train: {ACTIVE}' + (f' | skipped (not installed or unknown): {SKIPPED}' if SKIPPED else ''))

## Cross-Validated Training

Each active model is trained on the same folds as the baseline. Fold scores, the mean best iteration (a useful sanity check: values pinned at `N_ESTIMATORS` mean the learning rate is too low, values near zero mean overfitting starts immediately), and runtime are collected.

In [ ]:
for name in ACTIVE:
    RESULTS[name] = cross_validate(name, lambda n=name: MODEL_ZOO[n]())
    RESULTS[name]['best_iter'] = float(np.mean([m.best_iteration() for m in RESULTS[name]['models']]))

if not ACTIVE:
    note('No boosting model is active, so the pipeline continues with the baseline only.')
else:
    rows = []
    for name, r in RESULTS.items():
        rows.append({'model': name, f'cv_{METRIC}': r['cv_primary'], 'cv_std': r['cv_std'],
                     f'holdout_{METRIC}': r['holdout'].get(METRIC, np.nan), 'mean_best_iter': r.get('best_iter', np.nan),
                     'seconds': r['time']})
    display(pd.DataFrame(rows).set_index('model').round(4))

    fig, ax = plt.subplots(figsize=(11, 4.6))
    for i, (name, r) in enumerate(RESULTS.items()):
        ax.plot(range(1, len(r['fold_scores']) + 1), r['fold_scores'], marker='o', lw=2, ms=6,
                color=MUTED if name == 'baseline' else PALETTE[i % 10], label=f'{name} ({r["cv_primary"]:.4f})')
    ax.set_xticks(range(1, len(FOLDS) + 1))
    ax.set_xlabel('fold')
    ax.set_ylabel(METRIC)
    ax.legend(loc='best')
    suptitle(fig, 'Per-fold scores', f'{METRIC}, {"higher" if GREATER_IS_BETTER[METRIC] else "lower"} is better; consistent gaps across folds mean a real improvement')
    save_fig(fig, 'fold_scores', 'final')

for name in ACTIVE:
    cov = RESULTS[name]['covered']
    prediction_frame(RESULTS[name]['oof'][cov], dev_raw[cov], y_dev[cov]).to_csv(CFG.FINAL_DIR / 'oof' / f'{name}.csv', index=False)

#### Insights

> *Fill in after running.* How much do the boosted models improve on the linear baseline, and is the gain consistent in every fold? Do the three libraries agree? Large disagreement hints at instability or an unhandled data issue.

---

# Hyperparameter Tuning <a name="23"></a>

---

When `TUNE = True`, Optuna's TPE sampler searches the hyperparameters of `TUNE_MODEL` on a lighter `TUNE_FOLDS`-fold version of the same validation scheme. The best configuration is then retrained on the full shared folds as `<model>_tuned`, so its score is directly comparable to every other model. The holdout set is never used during tuning.

In [ ]:
def suggest(trial, kind):
    lr = trial.suggest_float('learning_rate', 0.01, 0.15, log=True)
    if kind == 'lightgbm':
        return dict(learning_rate=lr, num_leaves=trial.suggest_int('num_leaves', 15, 255, log=True),
                    min_child_samples=trial.suggest_int('min_child_samples', 5, 200, log=True),
                    subsample=trial.suggest_float('subsample', 0.5, 1.0),
                    colsample_bytree=trial.suggest_float('colsample_bytree', 0.3, 1.0),
                    reg_lambda=trial.suggest_float('reg_lambda', 1e-3, 30, log=True),
                    reg_alpha=trial.suggest_float('reg_alpha', 1e-3, 10, log=True))
    if kind == 'xgboost':
        return dict(learning_rate=lr, max_depth=trial.suggest_int('max_depth', 3, 10),
                    min_child_weight=trial.suggest_float('min_child_weight', 0.5, 50, log=True),
                    subsample=trial.suggest_float('subsample', 0.5, 1.0),
                    colsample_bytree=trial.suggest_float('colsample_bytree', 0.3, 1.0),
                    gamma=trial.suggest_float('gamma', 1e-3, 5, log=True),
                    reg_lambda=trial.suggest_float('reg_lambda', 1e-3, 30, log=True))
    return dict(learning_rate=lr, depth=trial.suggest_int('depth', 4, 10),
                l2_leaf_reg=trial.suggest_float('l2_leaf_reg', 1, 30, log=True),
                random_strength=trial.suggest_float('random_strength', 1e-3, 5, log=True),
                bagging_temperature=trial.suggest_float('bagging_temperature', 0, 1))

BEST_PARAMS = None
if not CFG.TUNE:
    note('TUNE is False, so tuning is skipped. Set TUNE = True and choose TUNE_MODEL to enable it.')
elif not AVAILABLE['optuna']:
    note('optuna is not installed, so tuning is skipped.')
elif CFG.TUNE_MODEL not in ACTIVE:
    note(f'TUNE_MODEL={CFG.TUNE_MODEL!r} is not an active model ({ACTIVE}), so tuning is skipped.')
else:
    kind = CFG.TUNE_MODEL
    tune_folds = make_folds(CFG.TUNE_FOLDS)

    def objective(trial):
        params = suggest(trial, kind)
        res = cross_validate('trial', lambda: MODEL_ZOO[kind](params), folds=tune_folds, verbose=False, eval_sets=False)
        return res['cv_primary']

    study = optuna.create_study(direction='maximize' if GREATER_IS_BETTER[METRIC] else 'minimize',
                                sampler=optuna.samplers.TPESampler(seed=CFG.SEED))
    study.optimize(objective, n_trials=CFG.N_TRIALS, timeout=CFG.TUNE_TIMEOUT, show_progress_bar=False)
    BEST_PARAMS = study.best_params
    print(f'{len(study.trials)} trials | best {METRIC} = {study.best_value:.4f}')
    print('Best params:', BEST_PARAMS)
    trials = study.trials_dataframe()
    trials.to_csv(CFG.FINAL_DIR / f'optuna_trials_{kind}.csv', index=False)
    to_json(BEST_PARAMS, CFG.FINAL_DIR / f'best_params_{kind}.json')

    fig, axes = plt.subplots(1, 2, figsize=(16, 4.6))
    vals = trials['value'].values
    run_best = np.maximum.accumulate(vals) if GREATER_IS_BETTER[METRIC] else np.minimum.accumulate(vals)
    axes[0].scatter(trials['number'], vals, color=SOFT, s=25, label='trial')
    axes[0].plot(trials['number'], run_best, color=ACCENT, lw=2, label='best so far')
    axes[0].set_xlabel('trial')
    axes[0].set_ylabel(METRIC)
    axes[0].legend()
    axes[0].set_title('Optimisation history', loc='left')
    try:
        imp = pd.Series(optuna.importance.get_param_importances(study)).sort_values()
        axes[1].barh(imp.index, imp.values, color=PRIMARY)
        axes[1].set_title('Hyperparameter importance (fANOVA)', loc='left')
    except Exception:
        axes[1].set_visible(False)
    suptitle(fig, f'Optuna tuning: {kind}', f'{len(study.trials)} trials on {CFG.TUNE_FOLDS} folds')
    save_fig(fig, f'tuning_{kind}', 'final')

    RESULTS[f'{kind}_tuned'] = cross_validate(f'{kind}_tuned', lambda: MODEL_ZOO[kind](BEST_PARAMS))
    RESULTS[f'{kind}_tuned']['best_iter'] = float(np.mean([m.best_iteration() for m in RESULTS[f'{kind}_tuned']['models']]))

#### Insights

> *Fill in after running.* Did tuning improve the cross-validated score beyond fold noise (compare the gain with `cv_std`)? Which hyperparameters mattered most?

---

# Ensemble and Model Selection <a name="24"></a>

---

Different model families make different errors, so a weighted average of their predictions is usually better than any single one. Non-negative weights summing to one are fitted on the out-of-fold predictions by minimising RMSE (regression) or log loss (classification), which are smooth proxies for any selection metric. The final model is the candidate with the best cross-validated score. The holdout score is reported alongside but is not used for selection, which keeps it an honest estimate.

In [ ]:
def blend(arrs, w):
    return sum(wi * a for wi, a in zip(w, arrs))

WEIGHTS = None
cands = [k for k in RESULTS]
if CFG.ENSEMBLE and len(cands) >= 2:
    cov = np.logical_and.reduce([RESULTS[k]['covered'] for k in cands])
    oofs = [RESULTS[k]['oof'][cov] for k in cands]
    yc = y_dev[cov]

    def blend_loss(w):
        p = blend(oofs, w)
        if IS_REG:
            return np.sqrt(np.mean((yc - p) ** 2))
        return log_loss(yc, np.clip(p, 1e-7, 1), labels=list(range(N_CLASSES)))

    n = len(cands)
    opt = minimize(blend_loss, np.full(n, 1 / n), method='SLSQP', bounds=[(0, 1)] * n,
                   constraints={'type': 'eq', 'fun': lambda w: w.sum() - 1})
    WEIGHTS = {k: round(float(w), 4) for k, w in zip(cands, opt.x)}
    print('Ensemble weights:', WEIGHTS)

    oof = np.full_like(RESULTS[cands[0]]['oof'], np.nan)
    oof[cov] = blend(oofs, opt.x)
    ens = {'name': 'ensemble', 'oof': oof, 'covered': cov, 'models': None, 'time': 0.0,
           'hold': blend([RESULTS[k]['hold'] for k in cands], opt.x),
           'test': blend([RESULTS[k]['test'] for k in cands], opt.x) if X_test is not None else None}
    ens['fold_scores'] = [primary(y_dev[va], oof[va]) for _, va in FOLDS if cov[va].all()]
    ens['cv'] = score_all(y_dev[cov], oof[cov])
    ens['cv_primary'], ens['cv_std'] = ens['cv'][METRIC], float(np.std(ens['fold_scores']))
    ens['holdout'] = score_all(y_hold, ens['hold']) if HAS_HOLD else {}
    RESULTS['ensemble'] = ens
    to_json(WEIGHTS, CFG.FINAL_DIR / 'ensemble_weights.json')
else:
    note('Ensembling needs ENSEMBLE = True and at least two models.')

## Leaderboard

The leaderboard lists every candidate with its cross-validated metrics and holdout score. The chart shows the CV score with its fold standard deviation and the holdout score as a marker, so overfitting to the folds (CV much better than holdout) is visible at a glance.

In [ ]:
rows = []
for name, r in RESULTS.items():
    row = {'model': name, f'cv_{METRIC}': r['cv_primary'], 'cv_std': r['cv_std'],
           f'holdout_{METRIC}': r['holdout'].get(METRIC, np.nan), 'seconds': r['time']}
    row.update({f'cv_{k}': v for k, v in r['cv'].items() if k != METRIC})
    rows.append(row)
LEADERBOARD = pd.DataFrame(rows).set_index('model').sort_values(f'cv_{METRIC}', ascending=not GREATER_IS_BETTER[METRIC])
BEST_NAME = LEADERBOARD.index[0]
BEST = RESULTS[BEST_NAME]
LEADERBOARD.to_csv(CFG.FINAL_DIR / 'leaderboard.csv')
display(LEADERBOARD.round(4))
print(f'Selected model: {BEST_NAME}  (CV {METRIC} = {BEST["cv_primary"]:.4f})')

lb = LEADERBOARD.iloc[::-1]
fig, axes = plt.subplots(1, 2 if WEIGHTS else 1, figsize=(17 if WEIGHTS else 11, 0.55 * len(lb) + 2.2), squeeze=False)
ax = axes[0, 0]
colors = [ACCENT if m == BEST_NAME else MUTED if m == 'baseline' else PRIMARY for m in lb.index]
ax.barh(lb.index, lb[f'cv_{METRIC}'], xerr=lb['cv_std'], color=colors, error_kw=dict(ecolor=DARK, lw=1, capsize=3))
if HAS_HOLD:
    ax.scatter(lb[f'holdout_{METRIC}'], lb.index, marker='D', color=DARK, zorder=3, s=40, label='holdout')
    ax.legend(loc='lower right')
vals = pd.concat([lb[f'cv_{METRIC}'] - lb['cv_std'], lb[f'cv_{METRIC}'] + lb['cv_std']] + ([lb[f'holdout_{METRIC}']] if HAS_HOLD else [])).dropna()
lo, hi = vals.min(), vals.max()
pad = max((hi - lo) * 0.25, abs(hi) * 0.02, 1e-6)
ax.set_xlim(lo - pad, hi + 2 * pad)
for i, (v, sd) in enumerate(zip(lb[f'cv_{METRIC}'], lb['cv_std'])):
    ax.text(v + sd, i, f'  {v:.4f}', fontsize=8, color='#4A5568', va='center')
ax.set_xlabel(f'{METRIC} ({"higher" if GREATER_IS_BETTER[METRIC] else "lower"} is better)')
ax.set_title('CV score +/- fold std (bars) and holdout (diamonds)', loc='left')
if WEIGHTS:
    w = pd.Series(WEIGHTS).sort_values()
    axes[0, 1].barh(w.index, w.values, color=PALETTE[2])
    bar_labels(axes[0, 1], '{:.2f}')
    axes[0, 1].set_title('Ensemble weights', loc='left')
suptitle(fig, 'Model leaderboard', f'selected: {BEST_NAME} (orange), baseline in grey')
save_fig(fig, 'leaderboard', 'final')

#### Insights

> *Fill in after running.* Which model was selected and by how much does it beat the baseline in CV and holdout? Do CV and holdout agree? A holdout much worse than CV signals overfitting or a distribution difference. Which models carry most of the ensemble weight?

---

# Final Evaluation on Holdout <a name="25"></a>

---

The selected model is now scored once on the holdout set, which played no part in training, tuning, weighting, or selection. When the metric depends on hard labels (F1, accuracy, QWK), the label rule is first tuned on out-of-fold predictions only: the threshold for binary tasks, one scale per class for multiclass (argmax of scaled probabilities), and the cut-points between classes for ordinal targets. The baseline is scored with the same rule. A bootstrap over holdout rows gives a 95% confidence interval for the score and for the improvement over the baseline, which is the evidence the report needs to claim the improvement is real.

In [ ]:
LABEL_METRICS = {'f1': lambda y, l: f1_score(y, l), 'f1_macro': lambda y, l: f1_score(y, l, average='macro'),
                 'f1_weighted': lambda y, l: f1_score(y, l, average='weighted'), 'accuracy': accuracy_score,
                 'balanced_accuracy': balanced_accuracy_score,
                 'qwk': lambda y, l: cohen_kappa_score(y, l, weights='quadratic'),
                 'mae_class': lambda y, l: -np.mean(np.abs(y - l))}

def tune_decision(y, p, verbose=True):
    """Tune the label rule on out-of-fold predictions: binary threshold, multiclass class scales, ordinal cut-points."""
    if not CFG.TUNE_DECISION or METRIC not in LABEL_METRICS or IS_REG and not IS_ORD:
        return None
    y = np.asarray(y).astype(int)
    fn = LABEL_METRICS[METRIC]
    label = to_ordinal if IS_ORD else to_label
    current = lambda: fn(y, label(p))
    start = current()
    if IS_ORD:
        cuts = np.arange(N_CLASSES - 1) + 0.5
        CUTS[0] = cuts.copy()
        for _ in range(3):
            for i in range(len(cuts)):
                lo = cuts[i - 1] + 1e-3 if i else min(p.min(), cuts[0] - 1)
                hi = cuts[i + 1] - 1e-3 if i < len(cuts) - 1 else max(p.max(), cuts[-1] + 1)
                best_c, best_s = cuts[i], current()
                for c in np.linspace(lo, hi, 41):
                    cuts[i] = c
                    CUTS[0] = cuts
                    sc = current()
                    if sc > best_s:
                        best_c, best_s = c, sc
                cuts[i] = best_c
                CUTS[0] = cuts.copy()
        rule = 'cut-points ' + ', '.join(f'{c:.2f}' for c in CUTS[0])
    elif IS_BINARY:
        grid = np.linspace(0.05, 0.95, 91)
        scores = []
        for t in grid:
            THRESHOLD[0] = t
            scores.append(current())
        THRESHOLD[0] = float(grid[int(np.argmax(scores))])
        rule = f'threshold {THRESHOLD[0]:.2f}'
    else:
        scale = np.ones(N_CLASSES)
        grid = np.exp(np.linspace(-1.5, 1.5, 25))
        for _ in range(2):
            for k in range(N_CLASSES):
                best_v, best_s = scale[k], current()
                for v in grid:
                    scale[k] = v
                    CLASS_SCALE[0] = scale
                    sc = current()
                    if sc > best_s:
                        best_v, best_s = v, sc
                scale[k] = best_v
                CLASS_SCALE[0] = scale.copy()
        rule = 'class scales ' + ', '.join(f'{v:.2f}' for v in CLASS_SCALE[0][:8]) + (' ...' if N_CLASSES > 8 else '')
    end = current()
    if METRIC == 'mae_class':
        start, end = -start, -end
    if verbose:
        print(f'Tuned decision rule on OOF ({rule}): {METRIC} {start:.4f} -> {end:.4f}')
    return rule

cov = BEST['covered']
DECISION_RULE = tune_decision(y_dev[cov], BEST['oof'][cov])

if HAS_HOLD:
    EVAL_NAME, EVAL_Y = 'holdout', y_hold
    EVAL_P = {k: RESULTS[k]['hold'] for k in RESULTS}
    EVAL_RAW = hold_raw
else:
    note('HOLDOUT_SIZE is 0, so the final evaluation uses out-of-fold predictions instead.')
    cov = np.logical_and.reduce([r['covered'] for r in RESULTS.values()])
    EVAL_NAME, EVAL_Y = 'out-of-fold', y_dev[cov]
    EVAL_P = {k: RESULTS[k]['oof'][cov] for k in RESULTS}
    EVAL_RAW = dev_raw[cov].reset_index(drop=True)

naive_eval = naive_prediction(len(EVAL_Y))
final_table = pd.DataFrame({'naive': score_all(EVAL_Y, naive_eval),
                            'baseline': score_all(EVAL_Y, EVAL_P['baseline']),
                            BEST_NAME: score_all(EVAL_Y, EVAL_P[BEST_NAME])})
sign = 1 if GREATER_IS_BETTER[METRIC] else -1

rng = np.random.RandomState(CFG.SEED)
boot_best, boot_diff = [], []
for _ in range(200):
    b = rng.randint(0, len(EVAL_Y), len(EVAL_Y))
    sb = primary(EVAL_Y[b], EVAL_P[BEST_NAME][b])
    boot_best.append(sb)
    boot_diff.append(sign * (sb - primary(EVAL_Y[b], EVAL_P['baseline'][b])))
ci = np.nanpercentile(boot_best, [2.5, 97.5])
ci_diff = np.nanpercentile(boot_diff, [2.5, 97.5])
FINAL = {'model': BEST_NAME, 'evaluated_on': EVAL_NAME, 'metric': METRIC, 'threshold': THRESHOLD[0], 'decision_rule': DECISION_RULE,
         'scores': final_table[BEST_NAME].to_dict(), 'baseline_scores': final_table['baseline'].to_dict(),
         'naive_scores': final_table['naive'].to_dict(), 'ci95': ci.tolist(), 'improvement_vs_baseline_ci95': ci_diff.tolist()}
display(final_table.round(4))
rel = sign * (final_table.loc[METRIC, BEST_NAME] - final_table.loc[METRIC, 'baseline']) / abs(final_table.loc[METRIC, 'baseline']) * 100
print(f'{BEST_NAME} {EVAL_NAME} {METRIC} = {final_table.loc[METRIC, BEST_NAME]:.4f} (95% CI {ci[0]:.4f} to {ci[1]:.4f})')
print(f'Improvement over baseline: {rel:+.1f}% (bootstrap 95% CI of the gain: {ci_diff[0]:+.4f} to {ci_diff[1]:+.4f}, '
      f'{"significant" if ci_diff[0] > 0 else "not significant"})')
final_table.to_csv(CFG.FINAL_DIR / 'final_metrics.csv')
to_json(FINAL, CFG.FINAL_DIR / 'final_metrics.json')

## Final Diagnostics

The diagnostic panels mirror the baseline ones, so the two can be placed side by side in the report. For binary classification a calibration curve is added, because well-calibrated probabilities are what make a score usable as a business risk estimate.

In [ ]:
plot_diagnostics(EVAL_Y, EVAL_P[BEST_NAME], f'Final model ({BEST_NAME}) {EVAL_NAME} diagnostics', 'final',
                 f'{METRIC} = {final_table.loc[METRIC, BEST_NAME]:.4f} (95% CI {ci[0]:.4f} to {ci[1]:.4f}) vs baseline {final_table.loc[METRIC, "baseline"]:.4f}')

if IS_BINARY:
    fig, ax = plt.subplots(figsize=(7.5, 6))
    for name, col in ((BEST_NAME, ACCENT), ('baseline', MUTED)):
        frac, mean_pred = calibration_curve(EVAL_Y, EVAL_P[name][:, 1], n_bins=10, strategy='quantile')
        ax.plot(mean_pred, frac, marker='o', color=col, lw=2, label=name)
    ax.plot([0, 1], [0, 1], color=DARK, ls='--', lw=1, label='perfect')
    ax.set_xlabel('mean predicted probability')
    ax.set_ylabel('observed frequency')
    ax.legend()
    suptitle(fig, 'Calibration curve', f'{EVAL_NAME} set, quantile bins')
    save_fig(fig, 'calibration', 'final')

#### Insights

> *Fill in after running.* Quote the final score with its confidence interval and the improvement over the baseline. Is the gain significant (the CI of the gain excludes zero)? Where do the diagnostics show systematic error, for example under-prediction of high values?

---

# Model Interpretation <a name="26"></a>

---

Interpretation uses the best tree model (the ensemble itself is a weighted average of models, so its strongest tree component is explained). Gain importance shows how much each feature reduces loss inside the trees. Exact TreeSHAP contributions, computed with each library's own API and averaged over the fold models, show both how much and in which direction each feature moves individual predictions.

In [ ]:
TREE_NAMES = [k for k in RESULTS if RESULTS[k].get('models') and isinstance(RESULTS[k]['models'][0], GBMModel)]
INTERP = min(TREE_NAMES, key=lambda k: -sign * RESULTS[k]['cv_primary']) if TREE_NAMES else None
IMPORTANCE = pd.DataFrame()
if INTERP is None:
    note('No tree model was trained, so interpretation relies on the baseline coefficients in section 7.')
else:
    models = RESULTS[INTERP]['models']
    gain = pd.concat([m.importance() / max(m.importance().sum(), 1e-12) for m in models], axis=1).mean(axis=1)
    X_int = (X_hold if HAS_HOLD else X_dev).sample(min(2000, len(X_hold) if HAS_HOLD else len(X_dev)), random_state=CFG.SEED)
    contribs = [m.contributions(X_int) for m in models]
    mean_abs = np.mean([np.abs(c).mean(axis=tuple(range(c.ndim - 1))) for c in contribs], axis=0)
    IMPORTANCE = pd.DataFrame({'gain_share': gain, 'mean_abs_shap': mean_abs}, index=FEATURES)
    IMPORTANCE = IMPORTANCE.sort_values('mean_abs_shap', ascending=False)
    IMPORTANCE.to_csv(CFG.FINAL_DIR / 'feature_importance.csv')
    display(IMPORTANCE.head(15).round(4))

    top = IMPORTANCE.head(20).iloc[::-1]
    fig, axes = plt.subplots(1, 2, figsize=(17, 0.34 * len(top) + 2), sharey=True)
    cols = [ACCENT if c in F_CAT else PRIMARY for c in top.index]
    axes[0].barh([short(c, 30) for c in top.index], top['mean_abs_shap'], color=cols)
    axes[0].set_title('Mean |SHAP| (impact on prediction)', loc='left')
    axes[1].barh([short(c, 30) for c in top.index], top['gain_share'], color=cols)
    axes[1].set_title('Gain share inside the trees', loc='left')
    suptitle(fig, f'Feature importance: {INTERP}', 'orange = categorical, blue = numeric; averaged over fold models')
    save_fig(fig, 'feature_importance', 'final')

## SHAP Summary and Dependence

For regression and binary classification, the beeswarm shows each sampled row as a dot: its horizontal position is the feature's contribution and its colour is the feature value (red is high, blue is low, grey is categorical). The dependence panels then plot contribution against value for the four strongest numeric features, which turns the model into business statements such as 'the target rises with feature X up to a threshold and then flattens'.

In [ ]:
if INTERP is not None and (IS_REG or IS_BINARY):
    S = np.mean(contribs, axis=0)
    feats = IMPORTANCE.index[:15].tolist()
    fig, ax = plt.subplots(figsize=(12, 0.45 * len(feats) + 2))
    cmap = LinearSegmentedColormap.from_list('bw', ['#3D5A80', '#D6E4F0', '#EE6C4D'])
    jitter = np.random.RandomState(CFG.SEED)
    for i, f in enumerate(feats[::-1]):
        j = FEATURES.index(f)
        sv = S[:, j]
        yj = i + jitter.uniform(-0.3, 0.3, len(sv))
        if f in F_CAT:
            ax.scatter(sv, yj, s=6, color=MUTED, alpha=0.5)
        else:
            v = X_int[f].rank(pct=True).values
            ax.scatter(sv, yj, s=6, c=v, cmap=cmap, alpha=0.7)
    ax.set_yticks(range(len(feats)))
    ax.set_yticklabels([short(f, 30) for f in feats[::-1]])
    ax.axvline(0, color=DARK, lw=1)
    scale = f'log1p({T})' if (IS_REG and TRANSFORM) else (T if IS_REG else 'log-odds')
    ax.set_xlabel(f'SHAP value (contribution to {scale})')
    sm = plt.cm.ScalarMappable(cmap=cmap)
    cb = fig.colorbar(sm, ax=ax, shrink=0.6, ticks=[0, 1])
    cb.ax.set_yticklabels(['low', 'high'])
    suptitle(fig, f'SHAP summary: {INTERP}', f'{len(X_int):,} sampled {EVAL_NAME} rows, grey = categorical feature')
    save_fig(fig, 'shap_beeswarm', 'final')

    dep = [f for f in IMPORTANCE.index if f in F_NUM][:4]
    if dep:
        fig, axes = plt.subplots(1, len(dep), figsize=(4.6 * len(dep), 4.2), squeeze=False)
        for ax, f in zip(axes[0], dep):
            x = X_int[f].values
            sv = S[:, FEATURES.index(f)]
            ok = ~np.isnan(x)
            lo, hi = np.nanpercentile(x, [1, 99])
            m = ok & (x >= lo) & (x <= hi)
            ax.scatter(x[m], sv[m], s=7, color=PRIMARY, alpha=0.35)
            d = pd.DataFrame({'x': x[m], 's': sv[m]})
            b = d.groupby(pd.qcut(d['x'].rank(method='first'), 15), observed=True).mean()
            ax.plot(b['x'], b['s'], color=ACCENT, lw=2.2)
            ax.axhline(0, color=DARK, lw=0.8)
            ax.set_title(short(f, 26), loc='left', fontsize=10)
            ax.set_ylabel('SHAP value')
        suptitle(fig, 'SHAP dependence of the top numeric features', 'orange = binned mean contribution')
        save_fig(fig, 'shap_dependence', 'final')
elif INTERP is not None:
    note('Multiclass SHAP has one contribution per class, so only the mean |SHAP| ranking above is shown.')

#### Insights

> *Fill in after running.* Which features drive predictions, and in which direction? Do the SHAP directions agree with the baseline coefficients and the EDA? Translate the top three dependence shapes into plain business statements for the presentation.

---

# Error Analysis <a name="27"></a>

---

An aggregate score hides where the model fails. This section breaks the evaluation-set error down by the most important categorical feature and by the size of the target (regression), or by class and confusion pair (classification), and lists the worst individual predictions. These are the cases to inspect manually and the limitations to state honestly in the report.

In [ ]:
pred_best = EVAL_P[BEST_NAME]
seg_cols = [c for c in (IMPORTANCE.index if not IMPORTANCE.empty else F_CAT) if c in F_CAT]
seg_col = next((c for c in seg_cols if EVAL_RAW[c].nunique() <= 60), seg_cols[0] if seg_cols else None)

if IS_REG:
    err = pd.DataFrame({'actual': EVAL_Y, 'pred': pred_best})
    err['residual'] = err['actual'] - err['pred']
    err['abs_error'] = err['residual'].abs()
    err['decile'] = pd.qcut(err['actual'].rank(method='first'), 10, labels=[f'D{i}' for i in range(1, 11)])
    by_dec = err.groupby('decile', observed=True).agg(mae=('abs_error', 'mean'), bias=('residual', 'mean'), actual_mean=('actual', 'mean'))
    ncols = 2 if seg_col else 1
    fig, axes = plt.subplots(1, ncols, figsize=(8.5 * ncols, 5.5), squeeze=False)
    ax = axes[0, 0]
    ax.bar(by_dec.index.astype(str), by_dec['mae'], color=SOFT, label='MAE')
    ax.plot(by_dec.index.astype(str), by_dec['bias'], color=ACCENT, marker='o', lw=2, label='bias (actual - pred)')
    ax.axhline(0, color=DARK, lw=0.8)
    ax.set_xlabel(f'decile of actual {T} (D1 = lowest)')
    ax.legend()
    ax.set_title('Error by target size', loc='left')
    if seg_col:
        err['segment'] = EVAL_RAW[seg_col].astype(str).values
        top_lv = err['segment'].value_counts().index[:15]
        by_seg = err[err['segment'].isin(top_lv)].groupby('segment').agg(mae=('abs_error', 'mean'), bias=('residual', 'mean'), n=('abs_error', 'size')).sort_values('mae')
        ax = axes[0, 1]
        ax.barh([short(i, 22) for i in by_seg.index], by_seg['mae'], color=[ACCENT if b > 0 else PRIMARY for b in by_seg['bias']])
        ax.axvline(err['abs_error'].mean(), color=DARK, ls='--', lw=1)
        ax.set_title(f'MAE by {seg_col} (orange = under-predicted on average)', loc='left')
        by_seg.to_csv(CFG.FINAL_DIR / f'error_by_{slug(seg_col)}.csv')
    suptitle(fig, 'Where does the final model fail?', f'{EVAL_NAME} set')
    save_fig(fig, 'error_analysis', 'final')
    by_dec.to_csv(CFG.FINAL_DIR / 'error_by_target_decile.csv')
    worst = EVAL_RAW.assign(actual=EVAL_Y, predicted=pred_best, abs_error=np.abs(EVAL_Y - pred_best)).nlargest(15, 'abs_error')
else:
    lab = to_label(pred_best)
    p, r, f, s = precision_recall_fscore_support(EVAL_Y, lab, labels=list(range(N_CLASSES)), zero_division=0)
    report = pd.DataFrame({'precision': p, 'recall': r, 'f1': f, 'support': s}, index=CLASS_NAMES).sort_values('f1')
    report.to_csv(CFG.FINAL_DIR / 'classification_report.csv')
    display(report.round(3).head(15))
    cm = pd.crosstab(pd.Series(EVAL_Y, name='actual'), pd.Series(lab, name='predicted'))
    pairs = cm.stack().rename('count').reset_index()
    pairs = pairs[pairs['actual'] != pairs['predicted']].nlargest(12, 'count')
    pairs['pair'] = [f'{short(CLASS_NAMES[a], 14)} -> {short(CLASS_NAMES[b], 14)}' for a, b in zip(pairs['actual'], pairs['predicted'])]
    ncols = 2 if seg_col else 1
    fig, axes = plt.subplots(1, ncols, figsize=(8.5 * ncols, 5.5), squeeze=False)
    axes[0, 0].barh(pairs['pair'][::-1], pairs['count'][::-1], color=ACCENT)
    axes[0, 0].set_title('Most frequent confusions (actual -> predicted)', loc='left')
    if seg_col:
        seg = pd.Series(EVAL_RAW[seg_col].astype(str).values)
        top_lv = seg.value_counts().index[:15]
        er = pd.Series(lab != EVAL_Y).groupby(seg).mean().reindex(top_lv).sort_values()
        axes[0, 1].barh([short(i, 22) for i in er.index], er.values, color=PRIMARY)
        axes[0, 1].axvline((lab != EVAL_Y).mean(), color=DARK, ls='--', lw=1)
        axes[0, 1].set_title(f'Error rate by {seg_col}', loc='left')
    suptitle(fig, 'Where does the final model fail?', f'{EVAL_NAME} set')
    save_fig(fig, 'error_analysis', 'final')
    conf = pred_best.max(axis=1)
    wrong = EVAL_RAW.assign(actual=LE.inverse_transform(EVAL_Y), predicted=LE.inverse_transform(lab), confidence=conf)[lab != EVAL_Y]
    worst = wrong.nlargest(15, 'confidence')

show_cols = [c for c in worst.columns if c not in TEXT_COLS][:12]
worst[show_cols].to_csv(CFG.FINAL_DIR / 'worst_predictions.csv', index=False)
display(worst[show_cols].head(10))

#### Insights

> *Fill in after running.* Which segments or classes are hardest? Does the model shrink extreme targets toward the mean (positive bias in the top deciles, negative in the bottom)? Inspect the worst rows: are they label noise, missing information, or a fixable feature gap?

---

# Predictions and Export <a name="28"></a>

---

The final artefacts are written to `pipeline-output/final/`: predictions for the out-of-fold, holdout, and test sets (with `submission.csv` when a test file is configured), the fold models together with the fitted preprocessing objects, the resolved configuration, and a markdown report that summarises the whole run.

## Inference Function

`predict_new` reproduces the full path from raw rows to predictions with the fitted feature builder and the selected model (or ensemble). It is verified against the stored holdout predictions, which guarantees that the exported submission uses exactly the evaluated model.

In [ ]:
def model_predict(name, X):
    if name == 'ensemble':
        parts = [model_predict(k, X) for k in WEIGHTS]
        return blend(parts, list(WEIGHTS.values()))
    return np.mean([m.predict(X) for m in RESULTS[name]['models']], axis=0)

def predict_new(raw):
    X = FB.transform(prepare(raw)).reset_index(drop=True)
    X[F_NUM] = X[F_NUM].astype(float)
    return model_predict(BEST_NAME, X)

if HAS_HOLD:
    check = predict_new(hold_raw.head(200))
    ok = np.allclose(check, RESULTS[BEST_NAME]['hold'][:200], atol=1e-4)
    print(f'predict_new reproduces stored holdout predictions: {ok}')
display(prediction_frame(predict_new(dev_raw.head(5)), dev_raw.head(5)))

## Export Artefacts

Saved models are pickles of the notebook's wrapper classes, so they are reloaded inside this notebook (or any script that defines the same classes) with `joblib.load`.

In [ ]:
export_result(BEST, CFG.FINAL_DIR)

names = list(WEIGHTS) if BEST_NAME == 'ensemble' else [BEST_NAME]
for k in names:
    joblib.dump(RESULTS[k]['models'], CFG.FINAL_DIR / 'models' / f'{k}_fold_models.joblib')
joblib.dump({'feature_builder': FB, 'cat_levels': CAT_LEVELS, 'label_encoder': LE if IS_CLF else None,
             'features': FEATURES, 'num_features': F_NUM, 'cat_features': F_CAT, 'transform': TRANSFORM,
             'threshold': THRESHOLD[0], 'class_scale': CLASS_SCALE[0], 'cuts': CUTS[0], 'class_names': CLASS_NAMES,
             'best': BEST_NAME, 'weights': WEIGHTS},
            CFG.FINAL_DIR / 'models' / 'preprocessing.joblib')

settings = {k: v for k, v in vars(Settings).items() if not k.startswith('__') and not callable(v)}
resolved = {'task': TASK, 'metric': METRIC, 'scheme': SCHEME, 'target_transform': TRANSFORM, 'class_weights': USE_CW,
            'n_classes': N_CLASSES, 'n_dev': len(X_dev), 'n_holdout': len(X_hold), 'n_test': 0 if X_test is None else len(X_test),
            'features': FEATURES, 'active_models': ACTIVE, 'skipped_models': SKIPPED, 'best_model': BEST_NAME,
            'ensemble_weights': WEIGHTS, 'tuned_params': BEST_PARAMS, 'threshold': THRESHOLD[0], 'libraries': AVAILABLE}
decision_table = pd.DataFrame(DECISIONS).set_index('setting')
decision_table.to_csv(CFG.FINAL_DIR / 'decisions.csv')
to_json({'settings': settings, 'resolved': resolved, 'eda_decisions': EDA}, CFG.FINAL_DIR / 'config.json')

## Run Report

The report collects the key numbers of the run into `pipeline-output/final/report.md`. It is a factual starting point for the methodology and results chapters of the written report.

In [ ]:
fmt = lambda v: f'{v:.4f}' if isinstance(v, (float, np.floating)) and not np.isnan(v) else '-'
lb_cols = [f'cv_{METRIC}', 'cv_std', f'holdout_{METRIC}']
lines = [f'# Modelling Report: {DATA_PATH.name}', '',
         '## Setup',
         f'- Task: **{TASK}**' + (f' ({N_CLASSES} classes)' if (IS_CLF or IS_ORD) else '') + f', target `{T}`, metric **{METRIC}**'
         + (f'; decision rule tuned on OOF: {DECISION_RULE}' if DECISION_RULE else ''),
         f'- Rows: {len(X_dev):,} development, {len(X_hold):,} holdout' + (f', {len(X_test):,} test' if X_test is not None else ''),
         f'- Validation: {SCHEME} {CFG.N_FOLDS}-fold' + (f' grouped by `{CV_GROUP}`' if SCHEME == 'group' else ''),
         f'- Target transform: {TRANSFORM}; class weights: {USE_CW}',
         f'- Features: {len(FEATURES)} ({len(F_NUM)} numeric, {len(F_CAT)} categorical)', '',
         '## Decisions', '', '| setting | value | source | why |', '|---|---|---|---|']
lines += [f'| {k} | `{r.value}` | {r.source} | {r.why} |' for k, r in decision_table.iterrows()]
lines += ['', '## Leaderboard', '', '| model | ' + ' | '.join(lb_cols) + ' |', '|---' * (len(lb_cols) + 1) + '|']
for m, r in LEADERBOARD.iterrows():
    lines.append(f'| {m}{" (selected)" if m == BEST_NAME else ""} | ' + ' | '.join(fmt(r[c]) for c in lb_cols) + ' |')
lines += ['', f'## Final result ({EVAL_NAME})',
          f'- {BEST_NAME}: {METRIC} = {FINAL["scores"][METRIC]:.4f} (95% CI {ci[0]:.4f} to {ci[1]:.4f})',
          f'- Baseline: {FINAL["baseline_scores"][METRIC]:.4f}; naive: {FINAL["naive_scores"][METRIC]:.4f}',
          f'- Improvement over baseline: {rel:+.1f}% (gain CI {ci_diff[0]:+.4f} to {ci_diff[1]:+.4f})',
          '- All metrics: ' + ', '.join(f'{k} {v:.4f}' for k, v in FINAL['scores'].items())]
if WEIGHTS:
    lines.append('- Ensemble weights: ' + ', '.join(f'{k} {v:.2f}' for k, v in WEIGHTS.items()))
if not IMPORTANCE.empty:
    lines += ['', f'## Top features ({INTERP}, mean |SHAP|)'] + [f'{i + 1}. `{f}` ({v:.4f})' for i, (f, v) in enumerate(IMPORTANCE['mean_abs_shap'].head(10).items())]
top_coef = coef_table.head(5)
lines += ['', '## Baseline top coefficients'] + [f'- `{f}`: {v:+.4f}' for f, v in top_coef['coefficient'].items()]
report_md = '\n'.join(lines)
(CFG.FINAL_DIR / 'report.md').write_text(report_md, encoding='utf-8')
display(Markdown(report_md))

## Additional Targets (Multi-Target and Multi-Label)

When `TARGET` is a list, the sample submission names several target columns, or `MULTILABEL_SEP` splits one column into binary labels, every further target gets the same treatment as the first one: the shared features and folds, the linear baseline, the active boosting models, a blend, and its own tuned decision rule. Each target keeps its own task and metric. The table is saved as `multi_target_metrics.csv`, multi-label runs add micro, macro, and per-sample F1, and `submission.csv` is rewritten with every target (labels joined back with the separator for multi-label).

In [ ]:
def test_column(pred):
    """Submission values for the current target: ordinal label, regression value, probability, or label."""
    fmt = CFG.SUBMISSION_FORMAT
    if fmt == 'auto':
        fmt = 'proba' if IS_CLF and METRIC in ('roc_auc', 'pr_auc', 'logloss') else 'label'
    if IS_ORD:
        return np.array(CLASS_NAMES, dtype=object)[to_ordinal(pred)]
    if IS_REG:
        return pred
    if fmt == 'proba' and IS_BINARY:
        return pred[:, 1]
    return LE.inverse_transform(to_label(pred))

def fit_current_target():
    """Baseline + active boosting models (+ blend) for the current target on the shared folds."""
    res = {'baseline': cross_validate(f'{T}:baseline', lambda: LinearModel(), verbose=False)}
    for name in ACTIVE:
        res[name] = cross_validate(f'{T}:{name}', lambda n=name: MODEL_ZOO[n](), verbose=False)
    if CFG.ENSEMBLE and len(res) >= 2:
        cands = list(res)
        cov = np.logical_and.reduce([res[k]['covered'] for k in cands])
        oofs, yc = [res[k]['oof'][cov] for k in cands], y_dev[cov]

        def loss(w):
            p = blend(oofs, w)
            return np.sqrt(np.mean((yc - p) ** 2)) if IS_REG else log_loss(yc, np.clip(p, 1e-7, 1), labels=list(range(N_CLASSES)))

        w = minimize(loss, np.full(len(cands), 1 / len(cands)), method='SLSQP', bounds=[(0, 1)] * len(cands),
                     constraints={'type': 'eq', 'fun': lambda w: w.sum() - 1}).x
        oof = np.full_like(res[cands[0]]['oof'], np.nan)
        oof[cov] = blend(oofs, w)
        ens = {'oof': oof, 'covered': cov, 'hold': blend([res[k]['hold'] for k in cands], w),
               'test': blend([res[k]['test'] for k in cands], w) if X_test is not None else None}
        ens['cv_primary'] = primary(y_dev[cov], oof[cov])
        res['ensemble'] = ens
    order = sorted(res, key=lambda k: res[k]['cv_primary'], reverse=GREATER_IS_BETTER[METRIC])
    return res, order[0]

MULTI = None
if len(TARGETS) > 1:
    KEEP = ('T', 'TASK', 'IS_REG', 'IS_CLF', 'IS_ORD', 'IS_BINARY', 'LE', 'CLASS_NAMES', 'N_CLASSES', 'USE_CW',
            'TRANSFORM', 'METRIC', 'y_dev', 'y_hold')
    saved = {k: globals().get(k) for k in KEEP}
    saved_rules = (THRESHOLD[0], CLASS_SCALE[0], CUTS[0])
    rows, preds, hold_labels = [], {}, {}
    rows.append({'target': T, 'task': TASK, 'metric': METRIC, 'model': BEST_NAME, 'cv': BEST['cv_primary'],
                 'holdout': FINAL['scores'].get(METRIC, np.nan), 'baseline_holdout': FINAL['baseline_scores'].get(METRIC, np.nan),
                 'decision_rule': DECISION_RULE})
    if X_test is not None:
        preds[T] = test_column(BEST['test'])
    if HAS_HOLD and IS_CLF:
        hold_labels[T] = (y_hold, to_label(BEST['hold']))
    for t in TARGETS[1:]:
        T = t
        s_dev, s_hold = dev_raw[t], hold_raw[t]
        numeric = pd.api.types.is_numeric_dtype(s_dev)
        auto_task = 'classification' if (ML_SOURCE or not numeric or s_dev.nunique() <= 20) else 'regression'
        TASK = CFG.TASK if CFG.TASK in ('regression', 'classification') else auto_task
        IS_ORD, IS_REG, IS_CLF = False, TASK == 'regression', TASK == 'classification'
        THRESHOLD[0], CLASS_SCALE[0], CUTS[0] = 0.5, None, None
        if IS_CLF:
            LE = LabelEncoder().fit(pd.concat([s_dev, s_hold]).astype(str))
            y_dev, y_hold = LE.transform(s_dev.astype(str)), LE.transform(s_hold.astype(str))
            CLASS_NAMES, N_CLASSES = list(LE.classes_), len(LE.classes_)
            IS_BINARY, TRANSFORM = N_CLASSES == 2, None
            counts = np.bincount(y_dev, minlength=N_CLASSES)
            USE_CW = bool(CFG.CLASS_WEIGHT is True or (CFG.CLASS_WEIGHT == 'auto' and counts.max() / max(counts.min(), 1) > 3))
        else:
            y_dev, y_hold = s_dev.astype(float).values, s_hold.astype(float).values
            LE, CLASS_NAMES, N_CLASSES, IS_BINARY, USE_CW = None, None, 1, False, False
            TRANSFORM = 'log1p' if (y_dev.min() >= 0 and pd.Series(y_dev).skew() > 1) else None
        METRIC = CFG.METRIC if CFG.METRIC != 'auto' else ('f1' if ML_SOURCE else 'rmse' if IS_REG else 'roc_auc' if IS_BINARY else 'f1_macro')
        res, best_name = fit_current_target()
        best = res[best_name]
        cov = best['covered']
        rule = tune_decision(y_dev[cov], best['oof'][cov], verbose=False)
        hold_score = primary(y_hold, best['hold']) if HAS_HOLD else np.nan
        base_hold = primary(y_hold, res['baseline']['hold']) if HAS_HOLD else np.nan
        rows.append({'target': t, 'task': TASK, 'metric': METRIC, 'model': best_name, 'cv': best['cv_primary'],
                     'holdout': hold_score, 'baseline_holdout': base_hold, 'decision_rule': rule})
        print(f'  [{t}] {TASK}, {METRIC}: {best_name} CV {best["cv_primary"]:.4f} | holdout {hold_score:.4f} (baseline {base_hold:.4f})')
        if X_test is not None:
            preds[t] = test_column(best['test'])
        if HAS_HOLD and IS_CLF:
            hold_labels[t] = (y_hold, to_label(best['hold']))
    globals().update(saved)
    THRESHOLD[0], CLASS_SCALE[0], CUTS[0] = saved_rules

    MULTI = pd.DataFrame(rows).set_index('target')
    MULTI.to_csv(CFG.FINAL_DIR / 'multi_target_metrics.csv')
    display(MULTI.round(4))
    if ML_SOURCE and len(hold_labels) == len(TARGETS):
        Y_true = np.column_stack([hold_labels[t][0] for t in TARGETS])
        Y_pred = np.column_stack([hold_labels[t][1] for t in TARGETS])
        ML_SCORES = {avg: float(f1_score(Y_true, Y_pred, average=avg, zero_division=1 if avg == 'samples' else 0)) for avg in ('micro', 'macro', 'samples')}
        print('Multi-label holdout F1: ' + ', '.join(f'{k} {v:.4f}' for k, v in ML_SCORES.items()))
        to_json(ML_SCORES, CFG.FINAL_DIR / 'multilabel_scores.json')

    fig, ax = plt.subplots(figsize=(11, 0.5 * len(MULTI) + 1.8))
    sign = np.where([GREATER_IS_BETTER[m] for m in MULTI['metric']], 1, -1)
    gain = sign * (MULTI['holdout'] - MULTI['baseline_holdout']) / MULTI['baseline_holdout'].abs().replace(0, np.nan) * 100
    ax.barh([short(i) for i in MULTI.index[::-1]], gain[::-1], color=[PRIMARY if g >= 0 else ACCENT for g in gain[::-1]])
    ax.axvline(0, color=DARK, lw=1)
    ax.set_xlabel('holdout improvement over the baseline (%)')
    suptitle(fig, 'Per-target results', f'{len(MULTI)} targets, each with its own metric, model, and decision rule')
    save_fig(fig, 'multi_target', 'final')

    if X_test is not None and preds:
        sample_id = SAMPLE.columns[0] if SAMPLE is not None and SAMPLE.columns[0] in test_df.columns else None
        id_col = CFG.SUBMISSION_ID or sample_id or next((c for c in ID_LIST if c in test_df.columns), None)
        sub = pd.DataFrame({id_col: test_df[id_col].values} if id_col else {}, index=range(len(test_df)))
        if ML_SOURCE:
            on = pd.DataFrame({t: np.asarray(preds[t]).astype(str) == '1' for t in TARGETS})
            sub[ML_SOURCE] = [CFG.MULTILABEL_SEP.join(lab for lab, hit in zip(ML_LABELS, row) if hit) for row in on.values]
        else:
            for t in TARGETS:
                sub[t] = preds[t]
        if SAMPLE is not None and id_col and SAMPLE.columns[0] == id_col:
            sub = SAMPLE[[id_col]].merge(sub, on=id_col, how='left')[[c for c in SAMPLE.columns if c in sub.columns]]
        sub.to_csv(CFG.FINAL_DIR / 'submission.csv', index=False)
        print(f'Multi-target submission: {sub.shape[0]:,} rows, columns {list(sub.columns)}')
else:
    print('Single target: nothing more to train.')

The last cell prints the output tree, so every artefact can be located quickly when writing the report or uploading the submission.

In [ ]:
for stage in (CFG.BASELINE_DIR, CFG.FINAL_DIR):
    print(f'{stage}/')
    for p in sorted(stage.rglob('*')):
        if p.is_file():
            print(f'   {str(p.relative_to(stage)):<55} {p.stat().st_size / 1e3:>9,.1f} KB')

#### Insights

> *Fill in after running.* Summarise the final model, its honest score with confidence interval, the gain over the baseline, the top drivers, and the main limitation found in the error analysis. These five points form the backbone of the conclusion chapter.